<a href="https://colab.research.google.com/github/Ximena5745/Modelo_Prospectivo/blob/main/Copia_de_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🔹 1. Librerías
!pip install prophet openpyxl tbats

In [ ]:
# Librerías base
!pip install --upgrade pip
!pip install --upgrade numpy==1.26.4
!pip install --upgrade scipy==1.12.0
!pip install --upgrade pandas
!pip install openpyxl

# Modelos estadísticos
!pip install --upgrade statsmodels==0.14.1
!pip install --upgrade pmdarima==2.0.4
!pip install prophet
# Machine Learning
!pip install --upgrade scikit-learn
!pip install --upgrade xgboost
!pip install --upgrade lightgbm

# Series temporales adicionales
!pip install tbats
!pip install arch  # Para modelos GARCH si necesitas volatilidad

# Visualización (opcional)
!pip install plotly
!pip install matplotlib
!pip install seaborn

  Using cached scipy-1.12.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Using cached scipy-1.12.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.8 MB)
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.2
    Uninstalling scipy-1.16.2:
      Successfully uninstalled scipy-1.16.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires scipy>=1.13, but you have scipy 1.12.0 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires scipy>=1.13, but you have scipy 1.12.0 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.1

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from pmdarima import auto_arima
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import itertools
import warnings
warnings.filterwarnings("ignore")


In [ ]:
# ===========================
# Si tu archivo está en Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ===========================
# 🔹 3. Horizonte de proyección
# ===========================
future_dates = pd.date_range(start="2026-06-30", end="2030-12-31", freq="6M")

In [ ]:
# ===========================
# 🔹 4. Funciones auxiliares
# ===========================

# Calcular tasa de crecimiento histórico promedio
def calcular_tasa_crecimiento(ts):
    ts = ts.dropna()
    if len(ts) < 2:
        return 0
    tasas = ts.pct_change().dropna()
    return tasas.mean() if not tasas.empty else 0

# Ajustar serie futura según la tasa de crecimiento histórica
def ajustar_por_tasa(forecast, tasa):
    forecast_ajustado = []
    valor = forecast[0]
    for i in range(len(forecast)):
        valor = valor * (1 + tasa)
        forecast_ajustado.append(valor)
    return np.array(forecast_ajustado)

# Generar escenarios según sentido
def generar_escenarios(serie_pred, sentido):
    if sentido.lower() == "positivo":
        return {
            "Base": serie_pred,
            "Optimista": serie_pred * 1.05,
            "Pesimista": serie_pred * 0.95
        }
    elif sentido.lower() == "negativo":
        return {
            "Base": serie_pred,
            "Optimista": serie_pred * 0.9,
            "Pesimista": serie_pred * 1.1
        }
    else:
        return {"Base": serie_pred}


In [ ]:
# ===========================
# 🔹 5. Función de proyección
# ===========================
def forecast_modelos(indicador, df_ind):
    resultados = []

    # Serie de tiempo
    ts = df_ind[["Fecha","Ejecución"]].rename(columns={"Fecha":"ds","Ejecución":"y"}).dropna()
    sentido = df_ind["Sentido"].iloc[0] if "Sentido" in df_ind.columns else "Positivo"

    # Calcular tasa de crecimiento histórico
    tasa = calcular_tasa_crecimiento(ts["y"])

    # ---------------------------
    # Modelo 1: Prophet
    # ---------------------------
    try:
        model_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
        model_prophet.fit(ts)
        future = pd.DataFrame({"ds": future_dates})
        forecast = model_prophet.predict(future)[["ds","yhat"]]
        forecast_adj = ajustar_por_tasa(forecast["yhat"].values, tasa)
        escenarios = generar_escenarios(forecast_adj, sentido)
        for esc, valores in escenarios.items():
            for fecha, val in zip(forecast["ds"], valores):
                resultados.append([indicador, fecha, val, "Prophet", esc])
    except Exception as e:
        print(f"[Prophet] Error en {indicador}: {e}")

    # ---------------------------
    # Modelo 2: Holt-Winters
    # ---------------------------
    try:
        hw_model = ExponentialSmoothing(ts["y"], trend="add", seasonal=None)
        hw_fit = hw_model.fit()
        forecast = hw_fit.forecast(len(future_dates))
        forecast_adj = ajustar_por_tasa(forecast, tasa)
        escenarios = generar_escenarios(forecast_adj, sentido)
        for esc, valores in escenarios.items():
            for fecha, val in zip(future_dates, valores):
                resultados.append([indicador, fecha, val, "Holt-Winters", esc])
    except Exception as e:
        print(f"[HW] Error en {indicador}: {e}")

    # ---------------------------
    # Modelo 3: Regresión Lineal
    # ---------------------------
    try:
        X = np.arange(len(ts)).reshape(-1,1)
        y = ts["y"].values
        reg = LinearRegression().fit(X, y)
        X_future = np.arange(len(ts), len(ts)+len(future_dates)).reshape(-1,1)
        forecast = reg.predict(X_future)
        forecast_adj = ajustar_por_tasa(forecast, tasa)
        escenarios = generar_escenarios(forecast_adj, sentido)
        for esc, valores in escenarios.items():
            for fecha, val in zip(future_dates, valores):
                resultados.append([indicador, fecha, val, "Regresión Lineal", esc])
    except Exception as e:
        print(f"[Regresión Lineal] Error en {indicador}: {e}")

    # ---------------------------
    # Modelo 4: XGBoost
    # ---------------------------
    try:
        ts["year"] = ts["ds"].dt.year
        ts["month"] = ts["ds"].dt.month
        ts["semester"] = np.where(ts["month"] <= 6, 1, 2)
        ts["t"] = np.arange(len(ts))

        X = ts[["year", "month", "semester", "t"]]
        y = ts["y"].values

        model_xgb = XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
        model_xgb.fit(X, y)

        future_df = pd.DataFrame({"ds": future_dates})
        future_df["year"] = future_df["ds"].dt.year
        future_df["month"] = future_df["ds"].dt.month
        future_df["semester"] = np.where(future_df["month"] <= 6, 1, 2)
        future_df["t"] = np.arange(len(ts), len(ts) + len(future_df))

        forecast = model_xgb.predict(future_df[["year", "month", "semester", "t"]])
        forecast_adj = ajustar_por_tasa(forecast, tasa)
        escenarios = generar_escenarios(forecast_adj, sentido)
        for esc, valores in escenarios.items():
            for fecha, val in zip(future_df["ds"], valores):
                resultados.append([indicador, fecha, val, "XGBoost", esc])
    except Exception as e:
        print(f"[XGBoost] Error en {indicador}: {e}")

    return resultados


#Multimodelo v1


In [ ]:
# ============================
# IMPORTS
# ============================
import pandas as pd
import numpy as np
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# ============================
# RUTAS EN DRIVE
# ============================
INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Dataset_Unificado.xlsx"
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Proyecciones_Miltimodelo v1.xlsx"

# ============================
# LECTURA DE BASE
# ============================
df = pd.read_excel(INPUT_FILE, sheet_name="Unificado")

# Nos quedamos con los campos relevantes
df = df[["Id","Indicador","Periodicidad","Fecha","Ejecución"]].copy()
df["Fecha"] = pd.to_datetime(df["Fecha"])
df = df.sort_values(["Id","Fecha"])

# ============================
# FUNCIÓN MEJORADA DE PROYECCIÓN
# ============================
def proyectar_indicador_mejorado(subdf, horizonte=10):
    """
    Genera proyecciones más realistas basadas en variaciones históricas
    """
    resultados = []
    id_ind = subdf["Id"].iloc[0]
    indicador = subdf["Indicador"].iloc[0]
    periodicidad = subdf["Periodicidad"].iloc[0]

    # Validar que tenemos suficientes datos (mínimo 3 puntos)
    if len(subdf) < 3:
        print(f"⚠️  Indicador {indicador}: Insuficientes datos ({len(subdf)} puntos)")
        return resultados

    # Serie temporal limpia
    ts = subdf.set_index("Fecha")["Ejecución"].astype(float)
    ts = ts.replace(0, np.nan).dropna()

    if len(ts) < 3:
        print(f"⚠️  Indicador {indicador}: Insuficientes datos válidos después de limpieza")
        return resultados

    # Calcular estadísticas históricas más robustas
    crecimiento_pct = ts.pct_change().dropna()
    crecimiento_promedio = crecimiento_pct.mean()
    volatilidad = crecimiento_pct.std()

    # Detectar tendencia general
    tendencia_lineal = np.polyfit(range(len(ts)), ts.values, 1)[0]

    ultimo_valor = ts.iloc[-1]
    penultimo_valor = ts.iloc[-2] if len(ts) > 1 else ultimo_valor

    # Definir fechas futuras según periodicidad
    if periodicidad == "Semestral":
        freq = "6M"
        days_offset = 180
    else:  # Anual
        freq = "A"
        days_offset = 365

    fechas_futuras = pd.date_range(
        start=ts.index[-1] + timedelta(days=days_offset),
        periods=horizonte,
        freq=freq
    )

    # ======================
    # 1. CRECIMIENTO HISTÓRICO MEJORADO
    # ======================
    for i, fecha in enumerate(fechas_futuras):
        # Proyección base usando tendencia suavizada
        periodos_adelante = i + 1

        # Usar promedio ponderado entre crecimiento % y tendencia lineal
        if abs(crecimiento_promedio) > 0.5:  # Si el crecimiento es muy alto, suavizar
            crecimiento_ajustado = crecimiento_promedio * 0.5
        else:
            crecimiento_ajustado = crecimiento_promedio

        # Proyección base
        base = ultimo_valor * ((1 + crecimiento_ajustado) ** periodos_adelante)

        # Ajustar por tendencia lineal si es significativa
        if abs(tendencia_lineal) > ultimo_valor * 0.01:  # Si la tendencia es > 1% del valor actual
            ajuste_tendencia = tendencia_lineal * periodos_adelante * (days_offset / 365)
            base += ajuste_tendencia

        # Escenarios basados en volatilidad histórica (más conservadores)
        factor_volatilidad = min(volatilidad, 0.2)  # Limitar volatilidad máxima al 20%

        optimista = base * (1 + factor_volatilidad)
        pesimista = base * (1 - factor_volatilidad)

        # Asegurar valores positivos
        base = max(0, base)
        optimista = max(0, optimista)
        pesimista = max(0, pesimista)

        resultados.append([
            id_ind, indicador, periodicidad, fecha, "Crecimiento_Historico",
            round(base, 2), round(pesimista, 2), round(optimista, 2)
        ])

    # ======================
    # 2. ARIMA MEJORADO
    # ======================
    try:
        # Usar auto_arima para encontrar mejores parámetros
        from pmdarima import auto_arima

        modelo_auto = auto_arima(
            ts,
            start_p=0, start_q=0, max_p=3, max_q=3,
            seasonal=False,
            stepwise=True,
            suppress_warnings=True,
            error_action='ignore'
        )

        pred = modelo_auto.predict(n_periods=horizonte, return_conf_int=True)
        forecast = pred[0]
        conf_int = pred[1]

        for i, fecha in enumerate(fechas_futuras):
            base = max(0, forecast[i])
            pesimista = max(0, conf_int[i][0])
            optimista = max(0, conf_int[i][1])

            resultados.append([
                id_ind, indicador, periodicidad, fecha, "ARIMA",
                round(base, 2), round(pesimista, 2), round(optimista, 2)
            ])

    except Exception as e:
        print(f"⚠️  Error en ARIMA para {indicador}: {str(e)}")

    # ======================
    # 3. PROPHET MEJORADO
    # ======================
    try:
        prophet_df = subdf.rename(columns={"Fecha":"ds","Ejecución":"y"})[["ds","y"]]
        prophet_df = prophet_df.dropna()

        if len(prophet_df) >= 3:
            m = Prophet(
                growth='linear',
                seasonality_mode='multiplicative',
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False,
                interval_width=0.8  # Intervalos de confianza más conservadores
            )

            m.fit(prophet_df)

            # Crear fechas futuras
            future = pd.DataFrame({'ds': fechas_futuras})
            forecast = m.predict(future)

            for i, row in forecast.iterrows():
                base = max(0, row["yhat"])
                optimista = max(0, row["yhat_upper"])
                pesimista = max(0, row["yhat_lower"])

                resultados.append([
                    id_ind, indicador, periodicidad, fechas_futuras[i], "Prophet",
                    round(base, 2), round(pesimista, 2), round(optimista, 2)
                ])

    except Exception as e:
        print(f"⚠️  Error en Prophet para {indicador}: {str(e)}")

    return resultados

# ============================
# APLICAR A TODOS LOS INDICADORES
# ============================
print("🚀 Iniciando proyecciones mejoradas...")

all_results = []
total_indicadores = df["Id"].nunique()
contador = 0

for id_ind, subdf in df.groupby("Id"):
    contador += 1
    indicador_nombre = subdf["Indicador"].iloc[0]
    print(f"📊 Procesando ({contador}/{total_indicadores}): {indicador_nombre}")

    res = proyectar_indicador_mejorado(subdf, horizonte=10)
    all_results.extend(res)

# ============================
# CREAR DATAFRAME FINAL Y VALIDACIONES
# ============================
if all_results:
    df_proy = pd.DataFrame(all_results,
                           columns=["Id","Indicador","Periodicidad","Fecha_Proyeccion",
                                    "Modelo","Escenario_Base","Escenario_Pesimista","Escenario_Optimista"])

    # Validaciones de calidad
    print("\n📋 RESUMEN DE PROYECCIONES:")
    print(f"✅ Total de proyecciones generadas: {len(df_proy)}")
    print(f"📈 Indicadores procesados: {df_proy['Indicador'].nunique()}")
    print(f"🔧 Modelos utilizados: {df_proy['Modelo'].unique()}")

    # Verificar valores extremos
    valores_extremos = df_proy[
        (df_proy['Escenario_Optimista'] > df_proy['Escenario_Base'] * 3) |
        (df_proy['Escenario_Pesimista'] < df_proy['Escenario_Base'] * 0.3)
    ]

    if len(valores_extremos) > 0:
        print(f"⚠️  Se detectaron {len(valores_extremos)} proyecciones con valores extremos")
        print("Indicadores afectados:", valores_extremos['Indicador'].unique())

    # Guardar resultado
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        df_proy.to_excel(writer, sheet_name='Proyecciones', index=False)

        # Guardar también estadísticas por indicador
        stats_df = df.groupby(['Id', 'Indicador']).agg({
            'Ejecución': ['count', 'mean', 'std', 'min', 'max']
        }).round(2)
        stats_df.columns = ['Observaciones', 'Promedio', 'Desv_Std', 'Mínimo', 'Máximo']
        stats_df.reset_index().to_excel(writer, sheet_name='Estadisticas_Historicas', index=False)

    print(f"✅ Proyecciones guardadas exitosamente en: {OUTPUT_FILE}")
    print("📊 Se incluyó una hoja adicional con estadísticas históricas para validación")

else:
    print("❌ No se generaron proyecciones. Verificar datos de entrada.")

print("\n🎯 RECOMENDACIONES:")
print("1. Revisa los indicadores con valores extremos")
print("2. Valida que los escenarios sean coherentes con la tendencia histórica")
print("3. Considera ajustar el horizonte de proyección según la estabilidad del indicador")

🚀 Iniciando proyecciones mejoradas...
📊 Procesando (1/47): Total Población
⚠️  Error en ARIMA para Total Población: 0
⚠️  Error en Prophet para Total Población: 'Prophet' object has no attribute 'stan_backend'
📊 Procesando (2/47): Estudiantes Presencial
⚠️  Error en ARIMA para Estudiantes Presencial: 0
⚠️  Error en Prophet para Estudiantes Presencial: 'Prophet' object has no attribute 'stan_backend'
📊 Procesando (3/47): Estudiantes Virtual
⚠️  Error en ARIMA para Estudiantes Virtual: 0
⚠️  Error en Prophet para Estudiantes Virtual: 'Prophet' object has no attribute 'stan_backend'
📊 Procesando (4/47): Estudiantes Pregrado
⚠️  Error en ARIMA para Estudiantes Pregrado: 0
⚠️  Error en Prophet para Estudiantes Pregrado: 'Prophet' object has no attribute 'stan_backend'
📊 Procesando (5/47): Estudiantes Posgrado
⚠️  Error en ARIMA para Estudiantes Posgrado: 0
⚠️  Error en Prophet para Estudiantes Posgrado: 'Prophet' object has no attribute 'stan_backend'
📊 Procesando (6/47): Disponibilidad de 

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================
# IMPORTS EXPANDIDOS
# ============================
import pandas as pd
import numpy as np
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
from datetime import timedelta


In [ ]:
 #============================
# RUTAS EN DRIVE
# ============================
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Proyecciones_Multimodelo v2.xlsx"


#Multimodelo V2

In [ ]:
# ============================
# LECTURA DE BASE
# ============================
df = pd.read_excel(INPUT_FILE, sheet_name="Unificado")
df = df[["Id","Indicador","Periodicidad","Fecha","Ejecución"]].copy()
df["Fecha"] = pd.to_datetime(df["Fecha"])
df = df.sort_values(["Id","Fecha"])

# ============================
# FUNCIONES AUXILIARES
# ============================
def crear_features_temporales(ts):
    """Crear features para modelos ML"""
    features = []
    target = []

    # Crear lags y features de ventana móvil
    for i in range(3, len(ts)):
        feature_row = [
            ts.iloc[i-1],  # lag 1
            ts.iloc[i-2],  # lag 2
            ts.iloc[i-3],  # lag 3
            ts.iloc[i-3:i].mean(),  # media móvil 3
            ts.iloc[max(0,i-6):i].mean(),  # media móvil 6
            ts.iloc[i-3:i].std() if i > 3 else 0,  # volatilidad
            (ts.iloc[i-1] - ts.iloc[i-2])/ts.iloc[i-2] if ts.iloc[i-2] != 0 else 0,  # cambio %
            i  # tendencia temporal
        ]
        features.append(feature_row)
        target.append(ts.iloc[i])

    return np.array(features), np.array(target)

def calcular_intervalos_confianza(predicciones, ts_historico, factor_confianza=0.2):
    """Calcular intervalos de confianza basados en error histórico"""
    error_historico = np.std(ts_historico.pct_change().dropna())
    error_ajustado = min(error_historico, factor_confianza)

    optimista = predicciones * (1 + error_ajustado)
    pesimista = predicciones * (1 - error_ajustado)

    return np.maximum(0, pesimista), np.maximum(0, optimista)

# ============================
# FUNCIÓN PRINCIPAL DE PROYECCIÓN MULTIMODELO
# ============================
def proyectar_multimodelo(subdf, horizonte=10):
    """
    Genera proyecciones con múltiples modelos de ML y estadísticos
    """
    resultados = []
    id_ind = subdf["Id"].iloc[0]
    indicador = subdf["Indicador"].iloc[0]
    periodicidad = subdf["Periodicidad"].iloc[0]

    # Validar datos suficientes
    if len(subdf) < 6:
        print(f"⚠️  {indicador}: Insuficientes datos ({len(subdf)} puntos)")
        return resultados

    # Serie temporal limpia
    ts = subdf.set_index("Fecha")["Ejecución"].astype(float)
    ts = ts.replace(0, np.nan).dropna()

    if len(ts) < 6:
        return resultados

    # Estadísticas base
    volatilidad = ts.pct_change().std()
    ultimo_valor = ts.iloc[-1]

    # Fechas futuras
    if periodicidad == "Semestral":
        freq = "6M"
        days_offset = 180
    else:
        freq = "A"
        days_offset = 365

    fechas_futuras = pd.date_range(
        start=ts.index[-1] + timedelta(days=days_offset),
        periods=horizonte,
        freq=freq
    )

    # ======================
    # 1. ARIMA
    # ======================
    try:
        modelo_arima = ARIMA(ts, order=(1,1,1))
        arima_fit = modelo_arima.fit()
        pred_arima = arima_fit.get_forecast(steps=horizonte)
        forecast_arima = pred_arima.predicted_mean
        conf_int_arima = pred_arima.conf_int()

        for i, fecha in enumerate(fechas_futuras):
            base = max(0, forecast_arima.iloc[i])
            pesimista = max(0, conf_int_arima.iloc[i, 0])
            optimista = max(0, conf_int_arima.iloc[i, 1])

            resultados.append([id_ind, indicador, periodicidad, fecha, "ARIMA",
                             round(base, 2), round(pesimista, 2), round(optimista, 2)])
    except:
        print(f"⚠️  ARIMA falló para {indicador}")

    # ======================
    # 2. EXPONENTIAL SMOOTHING (HOLT-WINTERS)
    # ======================
    try:
        if len(ts) >= 8:  # Necesita más datos para estacionalidad
            modelo_hw = ExponentialSmoothing(
                ts,
                trend='add',
                seasonal='add' if len(ts) >= 12 else None,
                seasonal_periods=2 if periodicidad == "Semestral" else 4
            )
            hw_fit = modelo_hw.fit()
            pred_hw = hw_fit.forecast(horizonte)

            pesimista, optimista = calcular_intervalos_confianza(pred_hw, ts)

            for i, fecha in enumerate(fechas_futuras):
                base = max(0, pred_hw.iloc[i])
                resultados.append([id_ind, indicador, periodicidad, fecha, "Holt_Winters",
                                 round(base, 2), round(pesimista[i], 2), round(optimista[i], 2)])
    except:
        print(f"⚠️  Holt-Winters falló para {indicador}")

    # ======================
    # 3. ETS (Error, Trend, Seasonality)
    # ======================
    try:
        modelo_ets = ETSModel(ts, error='add', trend='add', seasonal=None)
        ets_fit = modelo_ets.fit()
        pred_ets = ets_fit.get_forecast(horizonte)
        forecast_ets = pred_ets.predicted_mean

        pesimista, optimista = calcular_intervalos_confianza(forecast_ets, ts)

        for i, fecha in enumerate(fechas_futuras):
            base = max(0, forecast_ets.iloc[i])
            resultados.append([id_ind, indicador, periodicidad, fecha, "ETS",
                             round(base, 2), round(pesimista[i], 2), round(optimista[i], 2)])
    except:
        print(f"⚠️  ETS falló para {indicador}")

    # ======================
    # 4. RANDOM FOREST
    # ======================
    try:
        X, y = crear_features_temporales(ts)
        if len(X) > 0:
            modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42)
            modelo_rf.fit(X, y)

            # Predicciones futuras
            predicciones_rf = []
            ts_extended = ts.copy()

            for _ in range(horizonte):
                if len(ts_extended) >= 8:
                    # Crear features para el siguiente punto
                    ultimo_features = [
                        ts_extended.iloc[-1],
                        ts_extended.iloc[-2],
                        ts_extended.iloc[-3],
                        ts_extended.iloc[-3:].mean(),
                        ts_extended.iloc[-6:].mean(),
                        ts_extended.iloc[-3:].std(),
                        (ts_extended.iloc[-1] - ts_extended.iloc[-2])/ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0,
                        len(ts_extended)
                    ]

                    pred = modelo_rf.predict([ultimo_features])[0]
                    predicciones_rf.append(max(0, pred))

                    # Agregar predicción para siguiente iteración
                    ts_extended = pd.concat([ts_extended, pd.Series([pred])])
                else:
                    break

            if predicciones_rf:
                pesimista, optimista = calcular_intervalos_confianza(np.array(predicciones_rf), ts)

                for i, fecha in enumerate(fechas_futuras[:len(predicciones_rf)]):
                    base = predicciones_rf[i]
                    resultados.append([id_ind, indicador, periodicidad, fecha, "Random_Forest",
                                     round(base, 2), round(pesimista[i], 2), round(optimista[i], 2)])
    except Exception as e:
        print(f"⚠️  Random Forest falló para {indicador}: {e}")

    # ======================
    # 5. SUPPORT VECTOR REGRESSION (SVR)
    # ======================
    try:
        X, y = crear_features_temporales(ts)
        if len(X) > 0:
            scaler_X = StandardScaler()
            scaler_y = StandardScaler()

            X_scaled = scaler_X.fit_transform(X)
            y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

            modelo_svr = SVR(kernel='rbf', C=1.0, gamma='scale')
            modelo_svr.fit(X_scaled, y_scaled)

            # Predicciones futuras
            predicciones_svr = []
            ts_extended = ts.copy()

            for _ in range(horizonte):
                if len(ts_extended) >= 8:
                    ultimo_features = np.array([[
                        ts_extended.iloc[-1],
                        ts_extended.iloc[-2],
                        ts_extended.iloc[-3],
                        ts_extended.iloc[-3:].mean(),
                        ts_extended.iloc[-6:].mean(),
                        ts_extended.iloc[-3:].std(),
                        (ts_extended.iloc[-1] - ts_extended.iloc[-2])/ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0,
                        len(ts_extended)
                    ]])

                    features_scaled = scaler_X.transform(ultimo_features)
                    pred_scaled = modelo_svr.predict(features_scaled)
                    pred = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1))[0, 0]

                    predicciones_svr.append(max(0, pred))
                    ts_extended = pd.concat([ts_extended, pd.Series([pred])])
                else:
                    break

            if predicciones_svr:
                pesimista, optimista = calcular_intervalos_confianza(np.array(predicciones_svr), ts)

                for i, fecha in enumerate(fechas_futuras[:len(predicciones_svr)]):
                    base = predicciones_svr[i]
                    resultados.append([id_ind, indicador, periodicidad, fecha, "SVR",
                                     round(base, 2), round(pesimista[i], 2), round(optimista[i], 2)])
    except Exception as e:
        print(f"⚠️  SVR falló para {indicador}: {e}")

    # ======================
    # 6. REGRESIÓN LINEAL CON FEATURES TEMPORALES
    # ======================
    try:
        X, y = crear_features_temporales(ts)
        if len(X) > 0:
            modelo_lr = LinearRegression()
            modelo_lr.fit(X, y)

            # Predicciones futuras
            predicciones_lr = []
            ts_extended = ts.copy()

            for _ in range(horizonte):
                if len(ts_extended) >= 8:
                    ultimo_features = [[
                        ts_extended.iloc[-1],
                        ts_extended.iloc[-2],
                        ts_extended.iloc[-3],
                        ts_extended.iloc[-3:].mean(),
                        ts_extended.iloc[-6:].mean(),
                        ts_extended.iloc[-3:].std(),
                        (ts_extended.iloc[-1] - ts_extended.iloc[-2])/ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0,
                        len(ts_extended)
                    ]]

                    pred = modelo_lr.predict(ultimo_features)[0]
                    predicciones_lr.append(max(0, pred))
                    ts_extended = pd.concat([ts_extended, pd.Series([pred])])
                else:
                    break

            if predicciones_lr:
                pesimista, optimista = calcular_intervalos_confianza(np.array(predicciones_lr), ts)

                for i, fecha in enumerate(fechas_futuras[:len(predicciones_lr)]):
                    base = predicciones_lr[i]
                    resultados.append([id_ind, indicador, periodicidad, fecha, "Linear_Regression",
                                     round(base, 2), round(pesimista[i], 2), round(optimista[i], 2)])
    except Exception as e:
        print(f"⚠️  Linear Regression falló para {indicador}: {e}")

    # ======================
    # 7. PROPHET
    # ======================
    try:
        prophet_df = subdf.rename(columns={"Fecha":"ds","Ejecución":"y"})[["ds","y"]]
        prophet_df = prophet_df.dropna()

        if len(prophet_df) >= 6:
            m = Prophet(
                growth='linear',
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False,
                interval_width=0.8
            )
            m.fit(prophet_df)

            future = pd.DataFrame({'ds': fechas_futuras})
            forecast = m.predict(future)

            for i, row in forecast.iterrows():
                base = max(0, row["yhat"])
                optimista = max(0, row["yhat_upper"])
                pesimista = max(0, row["yhat_lower"])

                resultados.append([id_ind, indicador, periodicidad, fechas_futuras[i], "Prophet",
                                 round(base, 2), round(pesimista, 2), round(optimista, 2)])
    except Exception as e:
        print(f"⚠️  Prophet falló para {indicador}: {e}")

    # ======================
    # 8. CRECIMIENTO HISTÓRICO (BASELINE)
    # ======================
    try:
        crecimiento_pct = ts.pct_change().dropna()
        crecimiento_promedio = crecimiento_pct.mean()
        volatilidad = min(crecimiento_pct.std(), 0.2)

        for i, fecha in enumerate(fechas_futuras):
            periodos = i + 1
            base = ultimo_valor * ((1 + crecimiento_promedio) ** periodos)
            base = max(0, base)

            optimista = base * (1 + volatilidad)
            pesimista = base * (1 - volatilidad)

            resultados.append([id_ind, indicador, periodicidad, fecha, "Crecimiento_Historico",
                             round(base, 2), round(pesimista, 2), round(optimista, 2)])
    except Exception as e:
        print(f"⚠️  Crecimiento Histórico falló para {indicador}: {e}")

    return resultados

# ============================
# EJECUTAR PROYECCIONES
# ============================
print("🚀 Iniciando proyecciones con múltiples modelos...")
print("📊 Modelos disponibles: ARIMA, Holt-Winters, ETS, Random Forest, SVR, Linear Regression, Prophet, Crecimiento Histórico")

all_results = []
total_indicadores = df["Id"].nunique()
contador = 0

for id_ind, subdf in df.groupby("Id"):
    contador += 1
    indicador_nombre = subdf["Indicador"].iloc[0]
    print(f"\n📈 Procesando ({contador}/{total_indicadores}): {indicador_nombre}")

    res = proyectar_multimodelo(subdf, horizonte=10)
    all_results.extend(res)

    if res:
        modelos_exitosos = list(set([r[4] for r in res]))
        print(f"   ✅ Modelos exitosos: {', '.join(modelos_exitosos)}")

# ============================
# CREAR REPORTE FINAL
# ============================
if all_results:
    df_proy = pd.DataFrame(all_results,
                           columns=["Id","Indicador","Periodicidad","Fecha_Proyeccion",
                                    "Modelo","Escenario_Base","Escenario_Pesimista","Escenario_Optimista"])

    print(f"\n✅ RESULTADOS FINALES:")
    print(f"📊 Total proyecciones: {len(df_proy)}")
    print(f"🏢 Indicadores procesados: {df_proy['Indicador'].nunique()}")
    print(f"🤖 Modelos utilizados: {df_proy['Modelo'].nunique()}")

    # Estadísticas por modelo
    print(f"\n📈 PROYECCIONES POR MODELO:")
    modelo_stats = df_proy['Modelo'].value_counts()
    for modelo, count in modelo_stats.items():
        print(f"   {modelo}: {count} proyecciones")

    # Guardar resultados
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        df_proy.to_excel(writer, sheet_name='Proyecciones_Multimodelo', index=False)

        # Resumen por modelo e indicador
        resumen = df_proy.groupby(['Indicador', 'Modelo']).agg({
            'Escenario_Base': ['mean', 'std'],
            'Escenario_Pesimista': 'mean',
            'Escenario_Optimista': 'mean'
        }).round(2)
        resumen.to_excel(writer, sheet_name='Resumen_Por_Modelo')

        # Comparación de modelos
        comparacion = df_proy.pivot_table(
            index=['Indicador', 'Fecha_Proyeccion'],
            columns='Modelo',
            values='Escenario_Base'
        ).round(2)
        comparacion.to_excel(writer, sheet_name='Comparacion_Modelos')

    print(f"✅ Archivo guardado: {OUTPUT_FILE}")
    print("📋 Hojas incluidas: Proyecciones_Multimodelo, Resumen_Por_Modelo, Comparacion_Modelos")

else:
    print("❌ No se generaron proyecciones")

print(f"\n🎯 MODELOS IMPLEMENTADOS:")
print("   1. ARIMA - Modelo autoregresivo integrado de medias móviles")
print("   2. Holt-Winters - Suavizado exponencial con tendencia y estacionalidad")
print("   3. ETS - Error, Trend, Seasonality")
print("   4. Random Forest - Ensemble de árboles de decisión")
print("   5. SVR - Support Vector Regression")
print("   6. Linear Regression - Regresión lineal con features temporales")
print("   7. Prophet - Modelo de Facebook para series temporales")
print("   8. Crecimiento Histórico - Baseline basado en tendencia histórica")

🚀 Iniciando proyecciones con múltiples modelos...
📊 Modelos disponibles: ARIMA, Holt-Winters, ETS, Random Forest, SVR, Linear Regression, Prophet, Crecimiento Histórico

📈 Procesando (1/47): Total Población
⚠️  Holt-Winters falló para Total Población
⚠️  ETS falló para Total Población
⚠️  Prophet falló para Total Población: 'Prophet' object has no attribute 'stan_backend'
   ✅ Modelos exitosos: SVR, Random_Forest, Linear_Regression, Crecimiento_Historico, ARIMA

📈 Procesando (2/47): Estudiantes Presencial
⚠️  Holt-Winters falló para Estudiantes Presencial
⚠️  ETS falló para Estudiantes Presencial
⚠️  Prophet falló para Estudiantes Presencial: 'Prophet' object has no attribute 'stan_backend'
   ✅ Modelos exitosos: SVR, Random_Forest, Linear_Regression, Crecimiento_Historico, ARIMA

📈 Procesando (3/47): Estudiantes Virtual
⚠️  Holt-Winters falló para Estudiantes Virtual
⚠️  ETS falló para Estudiantes Virtual
⚠️  Prophet falló para Estudiantes Virtual: 'Prophet' object has no attribute 's

# MultiModelo V3

In [ ]:
# IMPORTS
# ============================
import pandas as pd
import numpy as np
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# CONFIGURACIÓN OPTIMIZADA
# ============================
MIN_DATA_POINTS = 6
OUTLIER_THRESHOLD = 3  # Desviaciones estándar para detectar outliers
RECENT_WEIGHT = 0.7    # Peso para datos recientes vs históricos
MIN_CONFIDENCE = 0.15  # Intervalo mínimo de confianza (15%)
MAX_CONFIDENCE = 0.30  # Intervalo máximo de confianza (30%)

In [ ]:
# ===========================
# Si tu archivo está en Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================
# RUTAS
# ============================
INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Dataset_Unificado.xlsx"
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Proyecciones_Multimodelo v3.xlsx"

In [ ]:
# ============================
# LECTURA Y PREPARACIÓN
# ============================
df = pd.read_excel(INPUT_FILE, sheet_name="Unificado")
df = df[["Id", "Indicador", "Periodicidad", "Fecha", "Ejecución"]].copy()
df["Fecha"] = pd.to_datetime(df["Fecha"])
df = df.sort_values(["Id", "Fecha"])

In [ ]:
# ============================
# FUNCIONES AUXILIARES OPTIMIZADAS
# ============================

def detectar_outliers(ts, threshold=OUTLIER_THRESHOLD):
    """Detecta y marca outliers usando Z-score"""
    if len(ts) < 4:
        return ts, []

    z_scores = np.abs(scipy_stats.zscore(ts))
    outliers = np.where(z_scores > threshold)[0]

    return ts, outliers.tolist()

def calcular_estadisticas_robustas(ts, outliers=[]):
    """
    Calcula estadísticas robustas sin outliers y con pesos para datos recientes
    """
    # Crear serie sin outliers
    ts_clean = ts.copy()
    if len(outliers) > 0:
        ts_clean = ts_clean.drop(ts_clean.index[outliers])

    if len(ts_clean) < 3:
        ts_clean = ts

    # Crecimientos (usando mediana para robustez)
    crecimientos = ts_clean.pct_change().dropna()

    if len(crecimientos) == 0:
        return {
            'crecimiento_mediano': 0,
            'crecimiento_promedio_ponderado': 0,
            'volatilidad': 0.10,
            'tendencia': 'estable',
            'limite_superior': 0.08,
            'limite_inferior': -0.08,
            'ultimo_valor': ts.iloc[-1],
            'n_datos': len(ts)
        }

    # Mediana (más robusta que promedio)
    crecimiento_mediano = crecimientos.median()

    # Promedio ponderado (más peso a datos recientes)
    n = len(crecimientos)
    pesos = np.array([RECENT_WEIGHT ** (n - i - 1) for i in range(n)])
    pesos = pesos / pesos.sum()
    crecimiento_ponderado = np.average(crecimientos, weights=pesos)

    # Volatilidad (usando MAD - Median Absolute Deviation)
    mad = np.median(np.abs(crecimientos - crecimiento_mediano))
    volatilidad = mad * 1.4826  # Factor para aproximar desviación estándar
    volatilidad = np.clip(volatilidad, MIN_CONFIDENCE, MAX_CONFIDENCE)

    # Detectar tendencia
    if crecimiento_ponderado > 0.03:  # >3%
        tendencia = 'creciente'
        limite_superior = min(crecimiento_ponderado * 1.5, 0.15)
        limite_inferior = max(crecimiento_ponderado * 0.3, -0.05)
    elif crecimiento_ponderado < -0.03:  # <-3%
        tendencia = 'decreciente'
        limite_superior = max(crecimiento_ponderado * 0.3, 0.05)
        limite_inferior = max(crecimiento_ponderado * 1.5, -0.15)
    else:
        tendencia = 'estable'
        limite_superior = 0.08
        limite_inferior = -0.08

    return {
        'crecimiento_mediano': crecimiento_mediano,
        'crecimiento_promedio_ponderado': crecimiento_ponderado,
        'volatilidad': volatilidad,
        'tendencia': tendencia,
        'limite_superior': limite_superior,
        'limite_inferior': limite_inferior,
        'ultimo_valor': ts.iloc[-1],
        'n_datos': len(ts_clean)
    }

def validar_proyeccion_realista(valor_proyectado, stats, periodo):
    """
    Valida que la proyección sea realista usando límites adaptativos
    """
    ultimo_valor = stats['ultimo_valor']

    if ultimo_valor <= 0 or valor_proyectado <= 0:
        return max(0, valor_proyectado)

    # Calcular crecimiento implícito
    try:
        crecimiento_total = (valor_proyectado / ultimo_valor) - 1
        crecimiento_anual = crecimiento_total / max(periodo, 1)
    except:
        return ultimo_valor

    # Aplicar límites adaptativos (más estrictos a mayor horizonte)
    factor_horizonte = 1 + (periodo * 0.05)  # Más conservador a futuro
    limite_sup_ajustado = stats['limite_superior'] / factor_horizonte
    limite_inf_ajustado = stats['limite_inferior'] / factor_horizonte

    if crecimiento_anual > limite_sup_ajustado:
        valor_corregido = ultimo_valor * (1 + limite_sup_ajustado) ** periodo
        return valor_corregido
    elif crecimiento_anual < limite_inf_ajustado:
        valor_corregido = ultimo_valor * (1 + limite_inf_ajustado) ** periodo
        return max(0, valor_corregido)

    return valor_proyectado

def calcular_intervalos_adaptativos(base, stats, periodo):
    """
    Calcula intervalos de confianza que se amplían con el horizonte
    """
    # Ampliar intervalo según horizonte (mayor incertidumbre a futuro)
    volatilidad_ajustada = stats['volatilidad'] * (1 + periodo * 0.08)
    volatilidad_ajustada = min(volatilidad_ajustada, MAX_CONFIDENCE)

    optimista = base * (1 + volatilidad_ajustada)
    pesimista = base * (1 - volatilidad_ajustada)

    return max(0, pesimista), max(0, optimista)

def calcular_pesos_ensemble(predicciones, stats):
    """
    Calcula pesos para el ensemble basados en cercanía a la tendencia histórica
    """
    if len(predicciones) == 0:
        return {}

    ultimo_valor = stats['ultimo_valor']
    crec_esperado = stats['crecimiento_promedio_ponderado']

    pesos = {}
    for modelo, valor in predicciones.items():
        if ultimo_valor > 0:
            crec_implicito = (valor / ultimo_valor) - 1
            diferencia = abs(crec_implicito - crec_esperado)
            # Menor diferencia = mayor peso
            pesos[modelo] = 1 / (1 + diferencia * 10)
        else:
            pesos[modelo] = 1

    # Normalizar
    suma_pesos = sum(pesos.values())
    if suma_pesos > 0:
        pesos = {k: v/suma_pesos for k, v in pesos.items()}

    return pesos

def crear_features_ml_optimizado(ts, n_lags=3):
    """
    Crear features con mejor ingeniería para ML
    """
    if len(ts) < n_lags + 3:
        return None, None

    features = []
    target = []

    for i in range(n_lags, len(ts)):
        try:
            # Lags
            lags = [ts.iloc[i-j] for j in range(1, n_lags+1)]

            # Medias móviles (corta y larga)
            ma_3 = ts.iloc[max(0, i-3):i].mean()
            ma_6 = ts.iloc[max(0, i-6):i].mean() if i >= 6 else ma_3

            # Volatilidad reciente
            vol = ts.iloc[max(0, i-3):i].std() if i >= 3 else 0

            # Momentum
            pct_change = (ts.iloc[i-1] - ts.iloc[i-2]) / ts.iloc[i-2] if ts.iloc[i-2] != 0 else 0

            # Aceleración
            if i >= 3:
                accel = ((ts.iloc[i-1] - ts.iloc[i-2]) - (ts.iloc[i-2] - ts.iloc[i-3])) / ts.iloc[i-2] if ts.iloc[i-2] != 0 else 0
            else:
                accel = 0

            # Tendencia lineal de últimos 3 puntos
            if i >= 3:
                x_trend = np.arange(3)
                y_trend = ts.iloc[i-3:i].values
                if len(y_trend) == 3:
                    slope = np.polyfit(x_trend, y_trend, 1)[0]
                else:
                    slope = 0
            else:
                slope = 0

            feature_row = lags + [ma_3, ma_6, vol, pct_change, accel, slope]
            features.append(feature_row)
            target.append(ts.iloc[i])
        except:
            continue

    if len(features) == 0:
        return None, None

    return np.array(features), np.array(target)

# ============================
# FUNCIÓN PRINCIPAL OPTIMIZADA
# ============================
def proyectar_optimizado(subdf, horizonte=10):
    """
    Sistema de proyecciones optimizado con validaciones robustas
    """
    resultados = []
    id_ind = subdf["Id"].iloc[0]
    indicador = subdf["Indicador"].iloc[0]
    periodicidad = subdf["Periodicidad"].iloc[0]

    # Validar datos mínimos
    if len(subdf) < MIN_DATA_POINTS:
        print(f"⚠️  {indicador}: Datos insuficientes ({len(subdf)} < {MIN_DATA_POINTS})")
        return resultados

    # Preparar serie temporal
    ts = subdf.set_index("Fecha")["Ejecución"].astype(float)
    ts = ts.replace(0, np.nan).dropna()

    if len(ts) < MIN_DATA_POINTS:
        print(f"⚠️  {indicador}: Datos válidos insuficientes")
        return resultados

    # Detectar outliers
    ts_values, outliers = detectar_outliers(ts.values)

    # Calcular estadísticas robustas
    stats = calcular_estadisticas_robustas(ts, outliers)

    # Generar fechas futuras
    if periodicidad == "Semestral":
        freq = "6MS"
    else:
        freq = "AS"

    try:
        fechas_futuras = pd.date_range(
            start=ts.index[-1] + pd.DateOffset(months=6 if periodicidad=="Semestral" else 12),
            periods=horizonte,
            freq=freq
        )
    except:
        print(f"⚠️  {indicador}: Error generando fechas")
        return resultados

    print(f"\n📊 {indicador}")
    print(f"   Último: {stats['ultimo_valor']:.2f} | Tendencia: {stats['tendencia']}")
    print(f"   Crec.Ponderado: {stats['crecimiento_promedio_ponderado']*100:.2f}% | Vol: {stats['volatilidad']*100:.1f}%")
    print(f"   Límites: [{stats['limite_inferior']*100:.1f}%, {stats['limite_superior']*100:.1f}%]")
    if outliers:
        print(f"   ⚠️  Outliers detectados: {len(outliers)}")

    predicciones_por_periodo = {i: {} for i in range(horizonte)}

    # ======================
    # MODELOS DE PROYECCIÓN
    # ======================

    # 1. ARIMA (robusto)
    try:
        modelo = ARIMA(ts, order=(1,1,1))
        fit = modelo.fit()
        pred = fit.get_forecast(steps=horizonte)
        forecast = pred.predicted_mean

        for i, fecha in enumerate(fechas_futuras):
            valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
            predicciones_por_periodo[i]['ARIMA'] = valor

        print(f"   ✅ ARIMA")
    except Exception as e:
        print(f"   ❌ ARIMA: {str(e)[:30]}")

    # 2. ETS (suavizado exponencial)
    try:
        modelo = ETSModel(ts, error='add', trend='add', seasonal=None, damped_trend=True)
        fit = modelo.fit(maxiter=1000, disp=False)
        forecast = fit.get_forecast(horizonte).predicted_mean

        for i, fecha in enumerate(fechas_futuras):
            valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
            predicciones_por_periodo[i]['ETS'] = valor

        print(f"   ✅ ETS")
    except Exception as e:
        print(f"   ❌ ETS: {str(e)[:30]}")

    # 3. Holt-Winters (solo si hay suficientes datos)
    try:
        if len(ts) >= 8:
            modelo = ExponentialSmoothing(ts, trend='add', seasonal=None, damped_trend=True)
            fit = modelo.fit()
            forecast = fit.forecast(horizonte)

            for i, fecha in enumerate(fechas_futuras):
                valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
                predicciones_por_periodo[i]['Holt_Winters'] = valor

            print(f"   ✅ Holt-Winters")
    except Exception as e:
        print(f"   ❌ Holt-Winters: {str(e)[:30]}")

    # 4. Random Forest (conservador)
    try:
        X, y = crear_features_ml_optimizado(ts, n_lags=3)

        if X is not None and len(X) >= 5:
            modelo = RandomForestRegressor(
                n_estimators=100,
                max_depth=5,
                min_samples_split=3,
                min_samples_leaf=2,
                random_state=42
            )
            modelo.fit(X, y)

            ts_extended = ts.copy()

            for periodo in range(horizonte):
                if len(ts_extended) >= 6:
                    lags = [ts_extended.iloc[-1], ts_extended.iloc[-2], ts_extended.iloc[-3]]
                    ma_3 = ts_extended.iloc[-3:].mean()
                    ma_6 = ts_extended.iloc[-6:].mean() if len(ts_extended) >= 6 else ma_3
                    vol = ts_extended.iloc[-3:].std()
                    pct = (ts_extended.iloc[-1] - ts_extended.iloc[-2])/ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0

                    if len(ts_extended) >= 3:
                        accel = ((ts_extended.iloc[-1] - ts_extended.iloc[-2]) - (ts_extended.iloc[-2] - ts_extended.iloc[-3])) / ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0
                        x_trend = np.arange(3)
                        y_trend = ts_extended.iloc[-3:].values
                        slope = np.polyfit(x_trend, y_trend, 1)[0]
                    else:
                        accel = 0
                        slope = 0

                    features = [lags + [ma_3, ma_6, vol, pct, accel, slope]]
                    pred = modelo.predict(features)[0]
                    pred = validar_proyeccion_realista(pred, stats, periodo+1)

                    predicciones_por_periodo[periodo]['Random_Forest'] = pred
                    ts_extended = pd.concat([ts_extended, pd.Series([pred])])

            print(f"   ✅ Random Forest")
    except Exception as e:
        print(f"   ❌ Random Forest: {str(e)[:30]}")

    # 5. Prophet (si hay suficientes datos)
    try:
        prophet_df = subdf.rename(columns={"Fecha":"ds","Ejecución":"y"})[["ds","y"]].dropna()

        if len(prophet_df) >= MIN_DATA_POINTS:
            m = Prophet(
                growth='linear',
                changepoint_prior_scale=0.01,  # Más conservador
                yearly_seasonality=False,
                weekly_seasonality=False,
                daily_seasonality=False,
                interval_width=0.8
            )
            m.fit(prophet_df)

            future = pd.DataFrame({'ds': fechas_futuras})
            forecast = m.predict(future)

            for i, row in forecast.iterrows():
                valor = validar_proyeccion_realista(row["yhat"], stats, i+1)
                predicciones_por_periodo[i]['Prophet'] = valor

            print(f"   ✅ Prophet")
    except Exception as e:
        print(f"   ❌ Prophet: {str(e)[:30]}")

    # 6. Tendencia Histórica (baseline)
    try:
        for i in range(horizonte):
            valor = stats['ultimo_valor'] * ((1 + stats['crecimiento_promedio_ponderado']) ** (i+1))
            valor = max(0, valor)
            predicciones_por_periodo[i]['Tendencia_Historica'] = valor

        print(f"   ✅ Tendencia Histórica")
    except Exception as e:
        print(f"   ❌ Tendencia Histórica: {str(e)[:30]}")

    # ======================
    # ENSEMBLE PONDERADO
    # ======================
    for i, fecha in enumerate(fechas_futuras):
        preds = predicciones_por_periodo[i]

        if len(preds) == 0:
            continue

        # Calcular pesos basados en coherencia con histórico
        pesos = calcular_pesos_ensemble(preds, stats)

        # Ensemble ponderado
        if len(pesos) > 0:
            base_ensemble = sum(preds[m] * pesos[m] for m in preds.keys())
            base_ensemble = validar_proyeccion_realista(base_ensemble, stats, i+1)
        else:
            base_ensemble = np.mean(list(preds.values()))

        pesimista, optimista = calcular_intervalos_adaptativos(base_ensemble, stats, i+1)

        resultados.append([
            id_ind, indicador, periodicidad, fecha, "Ensemble_Ponderado",
            round(base_ensemble, 2), round(pesimista, 2), round(optimista, 2)
        ])

        # Agregar predicciones individuales
        for modelo, valor in preds.items():
            pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
            resultados.append([
                id_ind, indicador, periodicidad, fecha, modelo,
                round(valor, 2), round(pes, 2), round(opt, 2)
            ])

    print(f"   📈 Proyección final (año 1): {base_ensemble:.2f}")

    return resultados

# ============================
# EJECUCIÓN
# ============================
print("="*70)
print("🚀 SISTEMA DE PROYECCIONES OPTIMIZADO")
print("="*70)
print(f"📊 Configuración:")
print(f"   - Validaciones robustas contra extrapolaciones")
print(f"   - Detección de outliers activa")
print(f"   - Pesos adaptativos para datos recientes")
print(f"   - Intervalos conservadores que se amplían con horizonte")
print(f"   - Ensemble ponderado por coherencia histórica")
print("="*70)

all_results = []
stats_summary = []
total = df["Id"].nunique()

for idx, (id_ind, subdf) in enumerate(df.groupby("Id"), 1):
    print(f"\n[{idx}/{total}]")

    res = proyectar_optimizado(subdf, horizonte=10)
    all_results.extend(res)

    if len(res) > 0:
        ts = subdf.set_index("Fecha")["Ejecución"].astype(float).replace(0, np.nan).dropna()
        if len(ts) >= MIN_DATA_POINTS:
            _, outliers = detectar_outliers(ts.values)
            stats = calcular_estadisticas_robustas(ts, outliers)

            stats_summary.append({
                'Id': id_ind,
                'Indicador': subdf["Indicador"].iloc[0],
                'Tendencia': stats['tendencia'],
                'Ultimo_Valor': stats['ultimo_valor'],
                'Crecimiento_Mediano_%': stats['crecimiento_mediano'] * 100,
                'Crecimiento_Ponderado_%': stats['crecimiento_promedio_ponderado'] * 100,
                'Volatilidad_%': stats['volatilidad'] * 100,
                'Outliers_Detectados': len(outliers),
                'Puntos_Datos': len(ts)
            })

# ============================
# GUARDAR RESULTADOS
# ============================
print("\n" + "="*70)
print("💾 GUARDANDO RESULTADOS")
print("="*70)

if all_results:
    df_proy = pd.DataFrame(
        all_results,
        columns=["Id", "Indicador", "Periodicidad", "Fecha_Proyeccion",
                 "Modelo", "Escenario_Base", "Escenario_Pesimista", "Escenario_Optimista"]
    )

    df_stats = pd.DataFrame(stats_summary)

    print(f"\n✅ RESUMEN:")
    print(f"   📈 Proyecciones totales: {len(df_proy):,}")
    print(f"   🏢 Indicadores: {df_proy['Indicador'].nunique()}")
    print(f"   🤖 Modelos: {df_proy['Modelo'].nunique()}")

    print(f"\n📊 DISTRIBUCIÓN POR MODELO:")
    for modelo, count in df_proy['Modelo'].value_counts().items():
        print(f"   {modelo:.<35} {count:>5,}")

    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        df_proy.to_excel(writer, sheet_name='Proyecciones', index=False)
        df_stats.to_excel(writer, sheet_name='Estadisticas_Robustas', index=False)

        # Solo proyecciones ensemble
        df_ensemble = df_proy[df_proy['Modelo'] == 'Ensemble_Ponderado'].copy()
        df_ensemble.to_excel(writer, sheet_name='Ensemble_Recomendado', index=False)

        # Comparación modelos
        pivot = df_proy.pivot_table(
            index=['Indicador', 'Fecha_Proyeccion'],
            columns='Modelo',
            values='Escenario_Base'
        ).round(2)
        pivot.to_excel(writer, sheet_name='Comparacion_Modelos')

    print(f"\n✅ Archivo guardado: {OUTPUT_FILE}")
    print(f"   📋 Hojas: Proyecciones | Estadisticas_Robustas | Ensemble_Recomendado | Comparacion_Modelos")

else:
    print("\n❌ No se generaron proyecciones")

print("\n" + "="*70)
print("✅ PROCESO COMPLETADO")
print("="*70)

🚀 SISTEMA DE PROYECCIONES OPTIMIZADO
📊 Configuración:
   - Validaciones robustas contra extrapolaciones
   - Detección de outliers activa
   - Pesos adaptativos para datos recientes
   - Intervalos conservadores que se amplían con horizonte
   - Ensemble ponderado por coherencia histórica

[1/47]

📊 Total Población
   Último: 56935.00 | Tendencia: estable
   Crec.Ponderado: 1.50% | Vol: 15.0%
   Límites: [-8.0%, 8.0%]
   ✅ ARIMA
   ❌ ETS: 'ETSResults' object has no att
   ✅ Holt-Winters
   ✅ Random Forest
   ❌ Prophet: 'Prophet' object has no attrib
   ✅ Tendencia Histórica
   📈 Proyección final (año 1): 58281.18

[2/47]

📊 Estudiantes Presencial
   Último: 17485.00 | Tendencia: creciente
   Crec.Ponderado: 35.16% | Vol: 30.0%
   Límites: [10.5%, 15.0%]
   ✅ ARIMA
   ❌ ETS: 'ETSResults' object has no att
   ✅ Holt-Winters
   ✅ Random Forest
   ❌ Prophet: 'Prophet' object has no attrib
   ✅ Tendencia Histórica
   📈 Proyección final (año 1): 45351.59

[3/47]

📊 Estudiantes Virtual
   Últ

# MultiModelo V4

In [ ]:
# ============================
# IMPORTS
# ============================
import pandas as pd
import numpy as np
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================
# CONFIGURACIÓN OPTIMIZADA
# ============================
MIN_DATA_POINTS = 6
OUTLIER_THRESHOLD = 3  # Desviaciones estándar para detectar outliers
RECENT_WEIGHT = 0.7    # Peso para datos recientes vs históricos
MIN_CONFIDENCE = 0.03  # Intervalo mínimo de confianza (3%)
MAX_CONFIDENCE = 0.08  # Intervalo máximo de confianza (8%)


In [ ]:
# ===========================
# Si tu archivo está en Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================
# RUTAS
# ============================
INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Dataset_Unificado.xlsx"
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Proyecciones_Multimodelo v4.xlsx"


In [ ]:
# ============================
# LECTURA Y PREPARACIÓN
# ============================
df = pd.read_excel(INPUT_FILE, sheet_name="Unificado")
df = df[["Id", "Indicador", "Periodicidad", "Fecha", "Ejecución"]].copy()
df["Fecha"] = pd.to_datetime(df["Fecha"])
df = df.sort_values(["Id", "Fecha"])

# ============================
# FUNCIONES AUXILIARES OPTIMIZADAS
# ============================

def detectar_outliers(ts, threshold=OUTLIER_THRESHOLD):
    """Detecta y marca outliers usando Z-score"""
    if len(ts) < 4:
        return ts, []

    z_scores = np.abs(scipy_stats.zscore(ts))
    outliers = np.where(z_scores > threshold)[0]

    return ts, outliers.tolist()

def calcular_estadisticas_robustas(ts, outliers=[]):
    """
    Calcula estadísticas robustas sin outliers y con pesos para datos recientes
    """
    # Crear serie sin outliers
    ts_clean = ts.copy()
    if len(outliers) > 0:
        ts_clean = ts_clean.drop(ts_clean.index[outliers])

    if len(ts_clean) < 3:
        ts_clean = ts

    # Crecimientos (usando mediana para robustez)
    crecimientos = ts_clean.pct_change().dropna()

    if len(crecimientos) == 0:
        return {
            'crecimiento_mediano': 0,
            'crecimiento_promedio_ponderado': 0,
            'volatilidad': 0.10,
            'tendencia': 'estable',
            'limite_superior': 0.08,
            'limite_inferior': -0.08,
            'ultimo_valor': ts.iloc[-1],
            'n_datos': len(ts)
        }

    # Mediana (más robusta que promedio)
    crecimiento_mediano = crecimientos.median()

    # Promedio ponderado (más peso a datos recientes)
    n = len(crecimientos)
    pesos = np.array([RECENT_WEIGHT ** (n - i - 1) for i in range(n)])
    pesos = pesos / pesos.sum()
    crecimiento_ponderado = np.average(crecimientos, weights=pesos)

    # Volatilidad (usando MAD - Median Absolute Deviation)
    mad = np.median(np.abs(crecimientos - crecimiento_mediano))
    volatilidad = mad * 1.4826  # Factor para aproximar desviación estándar

    # IMPORTANTE: Limitar volatilidad para intervalos realistas
    # Máximo 5% para indicadores estables, 8% para volátiles
    volatilidad = np.clip(volatilidad, 0.02, 0.05)  # Entre 2% y 5%

    # Detectar tendencia
    if crecimiento_ponderado > 0.03:  # >3%
        tendencia = 'creciente'
        limite_superior = min(crecimiento_ponderado * 1.5, 0.15)
        limite_inferior = max(crecimiento_ponderado * 0.3, -0.05)
    elif crecimiento_ponderado < -0.03:  # <-3%
        tendencia = 'decreciente'
        limite_superior = max(crecimiento_ponderado * 0.3, 0.05)
        limite_inferior = max(crecimiento_ponderado * 1.5, -0.15)
    else:
        tendencia = 'estable'
        limite_superior = 0.08
        limite_inferior = -0.08

    return {
        'crecimiento_mediano': crecimiento_mediano,
        'crecimiento_promedio_ponderado': crecimiento_ponderado,
        'volatilidad': volatilidad,
        'tendencia': tendencia,
        'limite_superior': limite_superior,
        'limite_inferior': limite_inferior,
        'ultimo_valor': ts.iloc[-1],
        'n_datos': len(ts_clean)
    }

def validar_proyeccion_realista(valor_proyectado, stats, periodo):
    """
    Valida que la proyección sea realista usando límites adaptativos
    """
    ultimo_valor = stats['ultimo_valor']

    if ultimo_valor <= 0 or valor_proyectado <= 0:
        return max(0, valor_proyectado)

    # Calcular crecimiento implícito
    try:
        crecimiento_total = (valor_proyectado / ultimo_valor) - 1
        crecimiento_anual = crecimiento_total / max(periodo, 1)
    except:
        return ultimo_valor

    # Aplicar límites adaptativos (más estrictos a mayor horizonte)
    factor_horizonte = 1 + (periodo * 0.05)  # Más conservador a futuro
    limite_sup_ajustado = stats['limite_superior'] / factor_horizonte
    limite_inf_ajustado = stats['limite_inferior'] / factor_horizonte

    if crecimiento_anual > limite_sup_ajustado:
        valor_corregido = ultimo_valor * (1 + limite_sup_ajustado) ** periodo
        return valor_corregido
    elif crecimiento_anual < limite_inf_ajustado:
        valor_corregido = ultimo_valor * (1 + limite_inf_ajustado) ** periodo
        return max(0, valor_corregido)

    return valor_proyectado

def calcular_intervalos_adaptativos(base, stats, periodo):
    """
    Calcula intervalos de confianza REALISTAS y cercanos a la base
    Los intervalos son estrechos y crecen moderadamente con el horizonte
    """
    # Usar volatilidad histórica como base, pero limitada
    vol_base = min(stats['volatilidad'], 0.05)  # Máximo 5% de base

    # Ampliar MUY LIGERAMENTE con el horizonte
    # Año 1: +2-3%, Año 5: +4-5%, Año 10: +6-8%
    factor_horizonte = 1 + (periodo * 0.005)  # 0.5% adicional por período
    volatilidad_ajustada = vol_base * factor_horizonte

    # Límite absoluto: entre 3% y 8%
    volatilidad_ajustada = np.clip(volatilidad_ajustada, MIN_CONFIDENCE, MAX_CONFIDENCE)

    # Calcular intervalos simétricos
    optimista = base * (1 + volatilidad_ajustada)
    pesimista = base * (1 - volatilidad_ajustada)

    # Asegurar que pesimista no sea negativo
    return max(0, pesimista), max(0, optimista)

def calcular_pesos_ensemble(predicciones, stats):
    """
    Calcula pesos para el ensemble basados en cercanía a la tendencia histórica
    """
    if len(predicciones) == 0:
        return {}

    ultimo_valor = stats['ultimo_valor']
    crec_esperado = stats['crecimiento_promedio_ponderado']

    pesos = {}
    for modelo, valor in predicciones.items():
        if ultimo_valor > 0:
            crec_implicito = (valor / ultimo_valor) - 1
            diferencia = abs(crec_implicito - crec_esperado)
            # Menor diferencia = mayor peso
            pesos[modelo] = 1 / (1 + diferencia * 10)
        else:
            pesos[modelo] = 1

    # Normalizar
    suma_pesos = sum(pesos.values())
    if suma_pesos > 0:
        pesos = {k: v/suma_pesos for k, v in pesos.items()}

    return pesos

def crear_features_ml_optimizado(ts, n_lags=3):
    """
    Crear features con mejor ingeniería para ML
    """
    if len(ts) < n_lags + 3:
        return None, None

    features = []
    target = []

    for i in range(n_lags, len(ts)):
        try:
            # Lags
            lags = [ts.iloc[i-j] for j in range(1, n_lags+1)]

            # Medias móviles (corta y larga)
            ma_3 = ts.iloc[max(0, i-3):i].mean()
            ma_6 = ts.iloc[max(0, i-6):i].mean() if i >= 6 else ma_3

            # Volatilidad reciente
            vol = ts.iloc[max(0, i-3):i].std() if i >= 3 else 0

            # Momentum
            pct_change = (ts.iloc[i-1] - ts.iloc[i-2]) / ts.iloc[i-2] if ts.iloc[i-2] != 0 else 0

            # Aceleración
            if i >= 3:
                accel = ((ts.iloc[i-1] - ts.iloc[i-2]) - (ts.iloc[i-2] - ts.iloc[i-3])) / ts.iloc[i-2] if ts.iloc[i-2] != 0 else 0
            else:
                accel = 0

            # Tendencia lineal de últimos 3 puntos
            if i >= 3:
                x_trend = np.arange(3)
                y_trend = ts.iloc[i-3:i].values
                if len(y_trend) == 3:
                    slope = np.polyfit(x_trend, y_trend, 1)[0]
                else:
                    slope = 0
            else:
                slope = 0

            feature_row = lags + [ma_3, ma_6, vol, pct_change, accel, slope]
            features.append(feature_row)
            target.append(ts.iloc[i])
        except:
            continue

    if len(features) == 0:
        return None, None

    return np.array(features), np.array(target)

# ============================
# FUNCIÓN PRINCIPAL OPTIMIZADA
# ============================
def proyectar_optimizado(subdf, horizonte=10):
    """
    Sistema de proyecciones optimizado con validaciones robustas
    """
    resultados = []
    id_ind = subdf["Id"].iloc[0]
    indicador = subdf["Indicador"].iloc[0]
    periodicidad = subdf["Periodicidad"].iloc[0]

    # Validar datos mínimos
    if len(subdf) < MIN_DATA_POINTS:
        print(f"⚠️  {indicador}: Datos insuficientes ({len(subdf)} < {MIN_DATA_POINTS})")
        return resultados

    # Preparar serie temporal
    ts = subdf.set_index("Fecha")["Ejecución"].astype(float)
    ts = ts.replace(0, np.nan).dropna()

    if len(ts) < MIN_DATA_POINTS:
        print(f"⚠️  {indicador}: Datos válidos insuficientes")
        return resultados

    # Detectar outliers
    ts_values, outliers = detectar_outliers(ts.values)

    # Calcular estadísticas robustas
    stats = calcular_estadisticas_robustas(ts, outliers)

    # Generar fechas futuras
    if periodicidad == "Semestral":
        freq = "6MS"
    else:
        freq = "AS"

    try:
        fechas_futuras = pd.date_range(
            start=ts.index[-1] + pd.DateOffset(months=6 if periodicidad=="Semestral" else 12),
            periods=horizonte,
            freq=freq
        )
    except:
        print(f"⚠️  {indicador}: Error generando fechas")
        return resultados

    print(f"\n📊 {indicador}")
    print(f"   Último: {stats['ultimo_valor']:.2f} | Tendencia: {stats['tendencia']}")
    print(f"   Crec.Ponderado: {stats['crecimiento_promedio_ponderado']*100:.2f}% | Vol: {stats['volatilidad']*100:.1f}%")
    print(f"   Límites: [{stats['limite_inferior']*100:.1f}%, {stats['limite_superior']*100:.1f}%]")
    if outliers:
        print(f"   ⚠️  Outliers detectados: {len(outliers)}")

    predicciones_por_periodo = {i: {} for i in range(horizonte)}

    # ======================
    # MODELOS DE PROYECCIÓN
    # ======================

    # 1. ARIMA (robusto)
    try:
        modelo = ARIMA(ts, order=(1,1,1))
        fit = modelo.fit()
        pred = fit.get_forecast(steps=horizonte)
        forecast = pred.predicted_mean
        conf_int = pred.conf_int(alpha=0.32)  # Intervalos más estrechos (68% confianza)

        for i, fecha in enumerate(fechas_futuras):
            valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
            predicciones_por_periodo[i]['ARIMA'] = valor

            # Usar intervalos de ARIMA pero ajustados a rangos realistas
            pes_arima = validar_proyeccion_realista(conf_int.iloc[i, 0], stats, i+1)
            opt_arima = validar_proyeccion_realista(conf_int.iloc[i, 1], stats, i+1)

            # Limitar a máximo 8% de diferencia con la base
            pes_arima = max(pes_arima, valor * 0.92)
            opt_arima = min(opt_arima, valor * 1.08)

            resultados.append([
                id_ind, indicador, periodicidad, fecha, "ARIMA",
                round(valor, 2), round(pes_arima, 2), round(opt_arima, 2)
            ])

        print(f"   ✅ ARIMA")
    except Exception as e:
        print(f"   ❌ ARIMA: {str(e)[:30]}")

    # 2. ETS (suavizado exponencial)
    try:
        modelo = ETSModel(ts, error='add', trend='add', seasonal=None, damped_trend=True)
        fit = modelo.fit(maxiter=1000, disp=False)
        pred = fit.get_forecast(horizonte)
        forecast = pred.predicted_mean

        for i, fecha in enumerate(fechas_futuras):
            valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
            predicciones_por_periodo[i]['ETS'] = valor

            # Intervalos conservadores para ETS
            pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
            resultados.append([
                id_ind, indicador, periodicidad, fecha, "ETS",
                round(valor, 2), round(pes, 2), round(opt, 2)
            ])

        print(f"   ✅ ETS")
    except Exception as e:
        print(f"   ❌ ETS: {str(e)[:30]}")

    # 3. Holt-Winters (solo si hay suficientes datos)
    try:
        if len(ts) >= 8:
            modelo = ExponentialSmoothing(ts, trend='add', seasonal=None, damped_trend=True)
            fit = modelo.fit()
            forecast = fit.forecast(horizonte)

            for i, fecha in enumerate(fechas_futuras):
                valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
                predicciones_por_periodo[i]['Holt_Winters'] = valor

                pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
                resultados.append([
                    id_ind, indicador, periodicidad, fecha, "Holt_Winters",
                    round(valor, 2), round(pes, 2), round(opt, 2)
                ])

            print(f"   ✅ Holt-Winters")
    except Exception as e:
        print(f"   ❌ Holt-Winters: {str(e)[:30]}")

    # 4. Random Forest (conservador)
    try:
        X, y = crear_features_ml_optimizado(ts, n_lags=3)

        if X is not None and len(X) >= 5:
            modelo = RandomForestRegressor(
                n_estimators=100,
                max_depth=5,
                min_samples_split=3,
                min_samples_leaf=2,
                random_state=42
            )
            modelo.fit(X, y)

            ts_extended = ts.copy()

            for periodo in range(horizonte):
                if len(ts_extended) >= 6:
                    lags = [ts_extended.iloc[-1], ts_extended.iloc[-2], ts_extended.iloc[-3]]
                    ma_3 = ts_extended.iloc[-3:].mean()
                    ma_6 = ts_extended.iloc[-6:].mean() if len(ts_extended) >= 6 else ma_3
                    vol = ts_extended.iloc[-3:].std()
                    pct = (ts_extended.iloc[-1] - ts_extended.iloc[-2])/ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0

                    if len(ts_extended) >= 3:
                        accel = ((ts_extended.iloc[-1] - ts_extended.iloc[-2]) - (ts_extended.iloc[-2] - ts_extended.iloc[-3])) / ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0
                        x_trend = np.arange(3)
                        y_trend = ts_extended.iloc[-3:].values
                        slope = np.polyfit(x_trend, y_trend, 1)[0]
                    else:
                        accel = 0
                        slope = 0

                    features = [lags + [ma_3, ma_6, vol, pct, accel, slope]]
                    pred = modelo.predict(features)[0]
                    pred = validar_proyeccion_realista(pred, stats, periodo+1)

                    predicciones_por_periodo[periodo]['Random_Forest'] = pred
                    ts_extended = pd.concat([ts_extended, pd.Series([pred])])

            # Guardar con intervalos
            for i, fecha in enumerate(fechas_futuras):
                if i in predicciones_por_periodo and 'Random_Forest' in predicciones_por_periodo[i]:
                    valor = predicciones_por_periodo[i]['Random_Forest']
                    pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
                    resultados.append([
                        id_ind, indicador, periodicidad, fecha, "Random_Forest",
                        round(valor, 2), round(pes, 2), round(opt, 2)
                    ])

            print(f"   ✅ Random Forest")
    except Exception as e:
        print(f"   ❌ Random Forest: {str(e)[:30]}")

    # 5. Prophet (si hay suficientes datos)
    try:
        prophet_df = subdf.rename(columns={"Fecha":"ds","Ejecución":"y"})[["ds","y"]].dropna()

        if len(prophet_df) >= MIN_DATA_POINTS:
            m = Prophet(
                growth='linear',
                changepoint_prior_scale=0.01,  # Más conservador
                yearly_seasonality=False,
                weekly_seasonality=False,
                daily_seasonality=False,
                interval_width=0.8
            )
            m.fit(prophet_df)

            future = pd.DataFrame({'ds': fechas_futuras})
            forecast = m.predict(future)

            for i, row in forecast.iterrows():
                valor = validar_proyeccion_realista(row["yhat"], stats, i+1)
                predicciones_por_periodo[i]['Prophet'] = valor

                # Prophet tiene intervalos propios, pero los ajustamos
                opt_prophet = validar_proyeccion_realista(row["yhat_upper"], stats, i+1)
                pes_prophet = validar_proyeccion_realista(row["yhat_lower"], stats, i+1)

                # Limitar a rangos realistas (máximo ±8%)
                opt_prophet = min(opt_prophet, valor * 1.08)
                pes_prophet = max(pes_prophet, valor * 0.92)

                resultados.append([
                    id_ind, indicador, periodicidad, fechas_futuras[i], "Prophet",
                    round(valor, 2), round(pes_prophet, 2), round(opt_prophet, 2)
                ])

            print(f"   ✅ Prophet")
    except Exception as e:
        print(f"   ❌ Prophet: {str(e)[:30]}")

    # 6. Tendencia Histórica (baseline)
    try:
        for i in range(horizonte):
            valor = stats['ultimo_valor'] * ((1 + stats['crecimiento_promedio_ponderado']) ** (i+1))
            valor = max(0, valor)
            predicciones_por_periodo[i]['Tendencia_Historica'] = valor

            pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
            resultados.append([
                id_ind, indicador, periodicidad, fechas_futuras[i], "Tendencia_Historica",
                round(valor, 2), round(pes, 2), round(opt, 2)
            ])

        print(f"   ✅ Tendencia Histórica")
    except Exception as e:
        print(f"   ❌ Tendencia Histórica: {str(e)[:30]}")

    # ======================
    # ENSEMBLE PONDERADO
    # ======================
    for i, fecha in enumerate(fechas_futuras):
        preds = predicciones_por_periodo[i]

        if len(preds) == 0:
            continue

        # Calcular pesos basados en coherencia con histórico
        pesos = calcular_pesos_ensemble(preds, stats)

        # Ensemble ponderado
        if len(pesos) > 0:
            base_ensemble = sum(preds[m] * pesos[m] for m in preds.keys())
            base_ensemble = validar_proyeccion_realista(base_ensemble, stats, i+1)
        else:
            base_ensemble = np.mean(list(preds.values()))

        pesimista, optimista = calcular_intervalos_adaptativos(base_ensemble, stats, i+1)

        resultados.append([
            id_ind, indicador, periodicidad, fecha, "Ensemble_Ponderado",
            round(base_ensemble, 2), round(pesimista, 2), round(optimista, 2)
        ])

    print(f"   📈 Proyección final (año 1): {base_ensemble:.2f}")

    return resultados

# ============================
# EJECUCIÓN
# ============================
print("="*70)
print("🚀 SISTEMA DE PROYECCIONES OPTIMIZADO V2")
print("="*70)
print(f"📊 Configuración:")
print(f"   - Intervalos CONSERVADORES: 3% - 8% máximo")
print(f"   - Escenarios realistas cercanos a línea base")
print(f"   - Validaciones robustas contra extrapolaciones")
print(f"   - Detección de outliers activa")
print(f"   - Crecimiento suave y progresivo por horizonte")
print("="*70)

all_results = []
stats_summary = []
total = df["Id"].nunique()

for idx, (id_ind, subdf) in enumerate(df.groupby("Id"), 1):
    print(f"\n[{idx}/{total}]")

    res = proyectar_optimizado(subdf, horizonte=10)
    all_results.extend(res)

    if len(res) > 0:
        ts = subdf.set_index("Fecha")["Ejecución"].astype(float).replace(0, np.nan).dropna()
        if len(ts) >= MIN_DATA_POINTS:
            _, outliers = detectar_outliers(ts.values)
            stats = calcular_estadisticas_robustas(ts, outliers)

            stats_summary.append({
                'Id': id_ind,
                'Indicador': subdf["Indicador"].iloc[0],
                'Tendencia': stats['tendencia'],
                'Ultimo_Valor': stats['ultimo_valor'],
                'Crecimiento_Mediano_%': stats['crecimiento_mediano'] * 100,
                'Crecimiento_Ponderado_%': stats['crecimiento_promedio_ponderado'] * 100,
                'Volatilidad_%': stats['volatilidad'] * 100,
                'Outliers_Detectados': len(outliers),
                'Puntos_Datos': len(ts)
            })

# ============================
# GUARDAR RESULTADOS
# ============================
print("\n" + "="*70)
print("💾 GUARDANDO RESULTADOS")
print("="*70)

if all_results:
    df_proy = pd.DataFrame(
        all_results,
        columns=["Id", "Indicador", "Periodicidad", "Fecha_Proyeccion",
                 "Modelo", "Escenario_Base", "Escenario_Pesimista", "Escenario_Optimista"]
    )

    df_stats = pd.DataFrame(stats_summary)

    print(f"\n✅ RESUMEN:")
    print(f"   📈 Proyecciones totales: {len(df_proy):,}")
    print(f"   🏢 Indicadores: {df_proy['Indicador'].nunique()}")
    print(f"   🤖 Modelos: {df_proy['Modelo'].nunique()}")

    print(f"\n📊 DISTRIBUCIÓN POR MODELO:")
    for modelo, count in df_proy['Modelo'].value_counts().items():
        print(f"   {modelo:.<35} {count:>5,}")

    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        df_proy.to_excel(writer, sheet_name='Proyecciones', index=False)
        df_stats.to_excel(writer, sheet_name='Estadisticas_Robustas', index=False)

        # Solo proyecciones ensemble
        df_ensemble = df_proy[df_proy['Modelo'] == 'Ensemble_Ponderado'].copy()
        df_ensemble.to_excel(writer, sheet_name='Ensemble_Recomendado', index=False)

        # Comparación modelos
        pivot = df_proy.pivot_table(
            index=['Indicador', 'Fecha_Proyeccion'],
            columns='Modelo',
            values='Escenario_Base'
        ).round(2)
        pivot.to_excel(writer, sheet_name='Comparacion_Modelos')

    print(f"\n✅ Archivo guardado: {OUTPUT_FILE}")
    print(f"   📋 Hojas: Proyecciones | Estadisticas_Robustas | Ensemble_Recomendado | Comparacion_Modelos")

else:
    print("\n❌ No se generaron proyecciones")

print("\n" + "="*70)
print("✅ PROCESO COMPLETADO")
print("="*70)

🚀 SISTEMA DE PROYECCIONES OPTIMIZADO V2
📊 Configuración:
   - Intervalos CONSERVADORES: 3% - 8% máximo
   - Escenarios realistas cercanos a línea base
   - Validaciones robustas contra extrapolaciones
   - Detección de outliers activa
   - Crecimiento suave y progresivo por horizonte

[1/47]

📊 Total Población
   Último: 56935.00 | Tendencia: estable
   Crec.Ponderado: 1.50% | Vol: 2.0%
   Límites: [-8.0%, 8.0%]
   ✅ ARIMA
   ❌ ETS: 'ETSResults' object has no att
   ✅ Holt-Winters
   ✅ Random Forest
   ❌ Prophet: 'Prophet' object has no attrib
   ✅ Tendencia Histórica
   📈 Proyección final (año 1): 58281.18

[2/47]

📊 Estudiantes Presencial
   Último: 17485.00 | Tendencia: creciente
   Crec.Ponderado: 35.16% | Vol: 5.0%
   Límites: [10.5%, 15.0%]
   ✅ ARIMA
   ❌ ETS: 'ETSResults' object has no att
   ✅ Holt-Winters
   ✅ Random Forest
   ❌ Prophet: 'Prophet' object has no attrib
   ✅ Tendencia Histórica
   📈 Proyección final (año 1): 45351.59

[3/47]

📊 Estudiantes Virtual
   Último: 95

#Multimodelo V5

In [ ]:
# ============================
# IMPORTS
# ============================
import pandas as pd
import numpy as np
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')


# ============================
# CONFIGURACIÓN OPTIMIZADA
# ============================
MIN_DATA_POINTS = 6
OUTLIER_THRESHOLD = 3
RECENT_WEIGHT = 0.7

# LÍMITES DIFERENCIADOS POR TIPO DE INDICADOR
LIMITES_POR_TIPO = {
    '%': {  # Indicadores porcentuales
        'min_confidence': 0.01,      # 1%
        'max_confidence': 0.03,      # 3%
        'limite_superior': 0.05,     # 5% máximo crecimiento anual
        'limite_inferior': -0.05,    # -5% mínimo
        'volatilidad_max': 0.03,     # 3% volatilidad máxima
        'valor_min': 0,              # No puede ser negativo
        'valor_max': 100             # No puede superar 100%
    },
    'ENT': {  # Enteros (conteos, cantidades)
        'min_confidence': 0.03,
        'max_confidence': 0.08,
        'limite_superior': 0.15,
        'limite_inferior': -0.10,
        'volatilidad_max': 0.08,
        'valor_min': 0,
        'valor_max': None
    },
    'DEC': {  # Decimales
        'min_confidence': 0.03,
        'max_confidence': 0.08,
        'limite_superior': 0.12,
        'limite_inferior': -0.08,
        'volatilidad_max': 0.08,
        'valor_min': 0,
        'valor_max': None
    },
    '$': {  # Moneda
        'min_confidence': 0.03,
        'max_confidence': 0.08,
        'limite_superior': 0.15,
        'limite_inferior': -0.10,
        'volatilidad_max': 0.08,
        'valor_min': 0,
        'valor_max': None
    },
    'DEFAULT': {  # Por defecto
        'min_confidence': 0.03,
        'max_confidence': 0.08,
        'limite_superior': 0.12,
        'limite_inferior': -0.08,
        'volatilidad_max': 0.08,
        'valor_min': 0,
        'valor_max': None
    }
}


# ===========================
# Si tu archivo está en Drive
from google.colab import drive
drive.mount('/content/drive')

# ============================
# RUTAS
# ============================
INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Dataset_Unificado.xlsx"
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Proyecciones_Optimizadas_v5.xlsx"

# ============================
# LECTURA Y PREPARACIÓN
# ============================
df = pd.read_excel(INPUT_FILE, sheet_name="Unificado")
df = df[["Id", "Indicador", "Periodicidad", "Fecha", "Ejecución", "Meta"]].copy()
df["Fecha"] = pd.to_datetime(df["Fecha"])
df = df.sort_values(["Id", "Fecha"])

# ============================
# FUNCIÓN PARA DETECTAR TIPO DE INDICADOR
# ============================
def detectar_tipo_indicador(meta_value):
    """
    Detecta el tipo de indicador basado en la columna Meta
    """
    if pd.isna(meta_value):
        return 'DEFAULT'

    meta_str = str(meta_value).upper().strip()

    if '%' in meta_str:
        return '%'
    elif 'ENT' in meta_str:
        return 'ENT'
    elif 'DEC' in meta_str:
        return 'DEC'
    elif '$' in meta_str:
        return '$'
    else:
        return 'DEFAULT'

def obtener_limites(tipo_indicador):
    """
    Obtiene los límites específicos para el tipo de indicador
    """
    return LIMITES_POR_TIPO.get(tipo_indicador, LIMITES_POR_TIPO['DEFAULT'])

# ============================
# FUNCIONES AUXILIARES OPTIMIZADAS
# ============================

def detectar_outliers(ts, threshold=OUTLIER_THRESHOLD):
    """Detecta y marca outliers usando Z-score"""
    if len(ts) < 4:
        return ts, []

    z_scores = np.abs(scipy_stats.zscore(ts))
    outliers = np.where(z_scores > threshold)[0]

    return ts, outliers.tolist()

def calcular_estadisticas_robustas(ts, outliers=[], tipo_indicador='DEFAULT'):
    """
    Calcula estadísticas robustas considerando el tipo de indicador
    """
    limites = obtener_limites(tipo_indicador)

    # Crear serie sin outliers
    ts_clean = ts.copy()
    if len(outliers) > 0:
        ts_clean = ts_clean.drop(ts_clean.index[outliers])

    if len(ts_clean) < 3:
        ts_clean = ts

    # Crecimientos (usando mediana para robustez)
    crecimientos = ts_clean.pct_change().dropna()

    if len(crecimientos) == 0:
        return {
            'crecimiento_mediano': 0,
            'crecimiento_promedio_ponderado': 0,
            'volatilidad': limites['min_confidence'],
            'tendencia': 'estable',
            'limite_superior': limites['limite_superior'],
            'limite_inferior': limites['limite_inferior'],
            'ultimo_valor': ts.iloc[-1],
            'n_datos': len(ts),
            'tipo_indicador': tipo_indicador
        }

    # Mediana (más robusta que promedio)
    crecimiento_mediano = crecimientos.median()

    # Promedio ponderado (más peso a datos recientes)
    n = len(crecimientos)
    pesos = np.array([RECENT_WEIGHT ** (n - i - 1) for i in range(n)])
    pesos = pesos / pesos.sum()
    crecimiento_ponderado = np.average(crecimientos, weights=pesos)

    # Volatilidad (usando MAD - Median Absolute Deviation)
    mad = np.median(np.abs(crecimientos - crecimiento_mediano))
    volatilidad = mad * 1.4826

    # LIMITAR VOLATILIDAD SEGÚN TIPO
    volatilidad = np.clip(volatilidad, limites['min_confidence'], limites['volatilidad_max'])

    # Detectar tendencia CON LÍMITES ESPECÍFICOS
    if crecimiento_ponderado > 0.02:
        tendencia = 'creciente'
        limite_superior = min(crecimiento_ponderado * 1.5, limites['limite_superior'])
        limite_inferior = max(crecimiento_ponderado * 0.3, limites['limite_inferior'])
    elif crecimiento_ponderado < -0.02:
        tendencia = 'decreciente'
        limite_superior = max(crecimiento_ponderado * 0.3, limites['limite_superior'] * 0.5)
        limite_inferior = max(crecimiento_ponderado * 1.5, limites['limite_inferior'])
    else:
        tendencia = 'estable'
        limite_superior = limites['limite_superior']
        limite_inferior = limites['limite_inferior']

    return {
        'crecimiento_mediano': crecimiento_mediano,
        'crecimiento_promedio_ponderado': crecimiento_ponderado,
        'volatilidad': volatilidad,
        'tendencia': tendencia,
        'limite_superior': limite_superior,
        'limite_inferior': limite_inferior,
        'ultimo_valor': ts.iloc[-1],
        'n_datos': len(ts_clean),
        'tipo_indicador': tipo_indicador
    }

def validar_proyeccion_realista(valor_proyectado, stats, periodo):
    """
    Valida que la proyección sea realista según el tipo de indicador
    """
    ultimo_valor = stats['ultimo_valor']
    tipo_indicador = stats['tipo_indicador']
    limites = obtener_limites(tipo_indicador)

    # VALIDACIÓN CRÍTICA PARA PORCENTUALES
    if tipo_indicador == '%':
        # Forzar rango 0-100%
        valor_proyectado = np.clip(valor_proyectado, limites['valor_min'], limites['valor_max'])

        # Si ya está cerca del límite, limitar crecimiento
        if ultimo_valor >= 95:
            valor_proyectado = min(valor_proyectado, 100)
        elif ultimo_valor <= 5:
            valor_proyectado = max(valor_proyectado, 0)

    # VALIDACIÓN DE CRECIMIENTO
    if ultimo_valor <= 0 or valor_proyectado <= 0:
        return max(limites['valor_min'], valor_proyectado) if limites['valor_min'] is not None else max(0, valor_proyectado)

    try:
        crecimiento_total = (valor_proyectado / ultimo_valor) - 1
        crecimiento_anual = crecimiento_total / max(periodo, 1)
    except:
        return ultimo_valor

    # Aplicar límites adaptativos más estrictos para porcentuales
    factor_horizonte = 1 + (periodo * 0.03) if tipo_indicador == '%' else 1 + (periodo * 0.05)
    limite_sup_ajustado = stats['limite_superior'] / factor_horizonte
    limite_inf_ajustado = stats['limite_inferior'] / factor_horizonte

    if crecimiento_anual > limite_sup_ajustado:
        valor_corregido = ultimo_valor * (1 + limite_sup_ajustado) ** periodo
        # Para porcentuales, asegurar que no supere 100
        if tipo_indicador == '%':
            valor_corregido = min(valor_corregido, 100)
        return valor_corregido
    elif crecimiento_anual < limite_inf_ajustado:
        valor_corregido = ultimo_valor * (1 + limite_inf_ajustado) ** periodo
        return max(limites['valor_min'] if limites['valor_min'] is not None else 0, valor_corregido)

    # Validar límite máximo si existe
    if limites['valor_max'] is not None:
        valor_proyectado = min(valor_proyectado, limites['valor_max'])

    return valor_proyectado

def calcular_intervalos_adaptativos(base, stats, periodo):
    """
    Calcula intervalos de confianza adaptativos según tipo de indicador
    """
    tipo_indicador = stats['tipo_indicador']
    limites = obtener_limites(tipo_indicador)

    # Usar volatilidad histórica pero limitada según tipo
    vol_base = min(stats['volatilidad'], limites['volatilidad_max'])

    # Ampliar LIGERAMENTE con el horizonte (más conservador para porcentuales)
    factor_crecimiento = 0.003 if tipo_indicador == '%' else 0.005
    factor_horizonte = 1 + (periodo * factor_crecimiento)
    volatilidad_ajustada = vol_base * factor_horizonte

    # Límite según tipo
    volatilidad_ajustada = np.clip(volatilidad_ajustada,
                                   limites['min_confidence'],
                                   limites['max_confidence'])

    # Calcular intervalos
    optimista = base * (1 + volatilidad_ajustada)
    pesimista = base * (1 - volatilidad_ajustada)

    # VALIDACIONES ESPECÍFICAS POR TIPO
    if tipo_indicador == '%':
        # Para porcentuales, forzar rango 0-100
        optimista = min(optimista, 100)
        pesimista = max(pesimista, 0)

        # Si la base está cerca de los límites, ajustar intervalos asimétricamente
        if base >= 95:
            optimista = min(optimista, 100)
            pesimista = max(base * 0.97, 0)
        elif base <= 5:
            pesimista = max(pesimista, 0)
            optimista = min(base * 1.03, 100)

    # Asegurar que pesimista no sea negativo
    pesimista = max(limites['valor_min'] if limites['valor_min'] is not None else 0, pesimista)

    return pesimista, optimista

def calcular_pesos_ensemble(predicciones, stats):
    """
    Calcula pesos para el ensemble basados en cercanía a la tendencia histórica
    """
    if len(predicciones) == 0:
        return {}

    ultimo_valor = stats['ultimo_valor']
    crec_esperado = stats['crecimiento_promedio_ponderado']

    pesos = {}
    for modelo, valor in predicciones.items():
        if ultimo_valor > 0:
            crec_implicito = (valor / ultimo_valor) - 1
            diferencia = abs(crec_implicito - crec_esperado)
            # Menor diferencia = mayor peso
            pesos[modelo] = 1 / (1 + diferencia * 10)
        else:
            pesos[modelo] = 1

    # Normalizar
    suma_pesos = sum(pesos.values())
    if suma_pesos > 0:
        pesos = {k: v/suma_pesos for k, v in pesos.items()}

    return pesos

def crear_features_ml_optimizado(ts, n_lags=3):
    """
    Crear features con mejor ingeniería para ML
    """
    if len(ts) < n_lags + 3:
        return None, None

    features = []
    target = []

    for i in range(n_lags, len(ts)):
        try:
            # Lags
            lags = [ts.iloc[i-j] for j in range(1, n_lags+1)]

            # Medias móviles
            ma_3 = ts.iloc[max(0, i-3):i].mean()
            ma_6 = ts.iloc[max(0, i-6):i].mean() if i >= 6 else ma_3

            # Volatilidad reciente
            vol = ts.iloc[max(0, i-3):i].std() if i >= 3 else 0

            # Momentum
            pct_change = (ts.iloc[i-1] - ts.iloc[i-2]) / ts.iloc[i-2] if ts.iloc[i-2] != 0 else 0

            # Aceleración
            if i >= 3:
                accel = ((ts.iloc[i-1] - ts.iloc[i-2]) - (ts.iloc[i-2] - ts.iloc[i-3])) / ts.iloc[i-2] if ts.iloc[i-2] != 0 else 0
            else:
                accel = 0

            # Tendencia lineal
            if i >= 3:
                x_trend = np.arange(3)
                y_trend = ts.iloc[i-3:i].values
                if len(y_trend) == 3:
                    slope = np.polyfit(x_trend, y_trend, 1)[0]
                else:
                    slope = 0
            else:
                slope = 0

            feature_row = lags + [ma_3, ma_6, vol, pct_change, accel, slope]
            features.append(feature_row)
            target.append(ts.iloc[i])
        except:
            continue

    if len(features) == 0:
        return None, None

    return np.array(features), np.array(target)

# ============================
# FUNCIÓN PRINCIPAL OPTIMIZADA
# ============================
def proyectar_optimizado(subdf, horizonte=10):
    """
    Sistema de proyecciones optimizado con validaciones por tipo de indicador
    """
    resultados = []
    id_ind = subdf["Id"].iloc[0]
    indicador = subdf["Indicador"].iloc[0]
    periodicidad = subdf["Periodicidad"].iloc[0]
    meta = subdf["Meta"].iloc[0]

    # DETECTAR TIPO DE INDICADOR
    tipo_indicador = detectar_tipo_indicador(meta)
    limites = obtener_limites(tipo_indicador)

    # Validar datos mínimos
    if len(subdf) < MIN_DATA_POINTS:
        print(f"⚠️  {indicador}: Datos insuficientes ({len(subdf)} < {MIN_DATA_POINTS})")
        return resultados

    # Preparar serie temporal
    ts = subdf.set_index("Fecha")["Ejecución"].astype(float)
    ts = ts.replace(0, np.nan).dropna()

    if len(ts) < MIN_DATA_POINTS:
        print(f"⚠️  {indicador}: Datos válidos insuficientes")
        return resultados

    # Detectar outliers
    ts_values, outliers = detectar_outliers(ts.values)

    # Calcular estadísticas robustas CON TIPO DE INDICADOR
    stats = calcular_estadisticas_robustas(ts, outliers, tipo_indicador)

    # Generar fechas futuras
    if periodicidad == "Semestral":
        freq = "6MS"
    else:
        freq = "AS"

    try:
        fechas_futuras = pd.date_range(
            start=ts.index[-1] + pd.DateOffset(months=6 if periodicidad=="Semestral" else 12),
            periods=horizonte,
            freq=freq
        )
    except:
        print(f"⚠️  {indicador}: Error generando fechas")
        return resultados

    print(f"\n📊 {indicador}")
    print(f"   📌 Tipo: {tipo_indicador} | Meta: {meta}")
    print(f"   Último: {stats['ultimo_valor']:.2f} | Tendencia: {stats['tendencia']}")
    print(f"   Crec.Ponderado: {stats['crecimiento_promedio_ponderado']*100:.2f}% | Vol: {stats['volatilidad']*100:.1f}%")
    print(f"   Límites: [{stats['limite_inferior']*100:.1f}%, {stats['limite_superior']*100:.1f}%]")
    if tipo_indicador == '%':
        print(f"   ⚠️  INDICADOR PORCENTUAL - Validación 0-100% activa")
    if outliers:
        print(f"   ⚠️  Outliers detectados: {len(outliers)}")

    predicciones_por_periodo = {i: {} for i in range(horizonte)}

    # ======================
    # MODELOS DE PROYECCIÓN
    # ======================

    # 1. ARIMA
    try:
        modelo = ARIMA(ts, order=(1,1,1))
        fit = modelo.fit()
        pred = fit.get_forecast(steps=horizonte)
        forecast = pred.predicted_mean
        conf_int = pred.conf_int(alpha=0.32)

        for i, fecha in enumerate(fechas_futuras):
            valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
            predicciones_por_periodo[i]['ARIMA'] = valor

            pes_arima = validar_proyeccion_realista(conf_int.iloc[i, 0], stats, i+1)
            opt_arima = validar_proyeccion_realista(conf_int.iloc[i, 1], stats, i+1)

            # Ajustar intervalos
            rango_max = valor * (1 + limites['max_confidence'])
            rango_min = valor * (1 - limites['max_confidence'])
            pes_arima = max(pes_arima, rango_min)
            opt_arima = min(opt_arima, rango_max)

            resultados.append([
                id_ind, indicador, periodicidad, tipo_indicador, fecha, "ARIMA",
                round(valor, 2), round(pes_arima, 2), round(opt_arima, 2)
            ])

        print(f"   ✅ ARIMA")
    except Exception as e:
        print(f"   ❌ ARIMA: {str(e)[:30]}")

    # 2. ETS
    try:
        modelo = ETSModel(ts, error='add', trend='add', seasonal=None, damped_trend=True)
        fit = modelo.fit(maxiter=1000, disp=False)
        pred = fit.get_forecast(horizonte)
        forecast = pred.predicted_mean

        for i, fecha in enumerate(fechas_futuras):
            valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
            predicciones_por_periodo[i]['ETS'] = valor

            pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
            resultados.append([
                id_ind, indicador, periodicidad, tipo_indicador, fecha, "ETS",
                round(valor, 2), round(pes, 2), round(opt, 2)
            ])

        print(f"   ✅ ETS")
    except Exception as e:
        print(f"   ❌ ETS: {str(e)[:30]}")

    # 3. Holt-Winters
    try:
        if len(ts) >= 8:
            modelo = ExponentialSmoothing(ts, trend='add', seasonal=None, damped_trend=True)
            fit = modelo.fit()
            forecast = fit.forecast(horizonte)

            for i, fecha in enumerate(fechas_futuras):
                valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
                predicciones_por_periodo[i]['Holt_Winters'] = valor

                pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
                resultados.append([
                    id_ind, indicador, periodicidad, tipo_indicador, fecha, "Holt_Winters",
                    round(valor, 2), round(pes, 2), round(opt, 2)
                ])

            print(f"   ✅ Holt-Winters")
    except Exception as e:
        print(f"   ❌ Holt-Winters: {str(e)[:30]}")

    # 4. Random Forest
    try:
        X, y = crear_features_ml_optimizado(ts, n_lags=3)

        if X is not None and len(X) >= 5:
            modelo = RandomForestRegressor(
                n_estimators=100,
                max_depth=5,
                min_samples_split=3,
                min_samples_leaf=2,
                random_state=42
            )
            modelo.fit(X, y)

            ts_extended = ts.copy()

            for periodo in range(horizonte):
                if len(ts_extended) >= 6:
                    lags = [ts_extended.iloc[-1], ts_extended.iloc[-2], ts_extended.iloc[-3]]
                    ma_3 = ts_extended.iloc[-3:].mean()
                    ma_6 = ts_extended.iloc[-6:].mean() if len(ts_extended) >= 6 else ma_3
                    vol = ts_extended.iloc[-3:].std()
                    pct = (ts_extended.iloc[-1] - ts_extended.iloc[-2])/ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0

                    if len(ts_extended) >= 3:
                        accel = ((ts_extended.iloc[-1] - ts_extended.iloc[-2]) - (ts_extended.iloc[-2] - ts_extended.iloc[-3])) / ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0
                        x_trend = np.arange(3)
                        y_trend = ts_extended.iloc[-3:].values
                        slope = np.polyfit(x_trend, y_trend, 1)[0]
                    else:
                        accel = 0
                        slope = 0

                    features = [lags + [ma_3, ma_6, vol, pct, accel, slope]]
                    pred = modelo.predict(features)[0]
                    pred = validar_proyeccion_realista(pred, stats, periodo+1)

                    predicciones_por_periodo[periodo]['Random_Forest'] = pred
                    ts_extended = pd.concat([ts_extended, pd.Series([pred])])

            for i, fecha in enumerate(fechas_futuras):
                if i in predicciones_por_periodo and 'Random_Forest' in predicciones_por_periodo[i]:
                    valor = predicciones_por_periodo[i]['Random_Forest']
                    pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
                    resultados.append([
                        id_ind, indicador, periodicidad, tipo_indicador, fecha, "Random_Forest",
                        round(valor, 2), round(pes, 2), round(opt, 2)
                    ])

            print(f"   ✅ Random Forest")
    except Exception as e:
        print(f"   ❌ Random Forest: {str(e)[:30]}")

    # 5. Prophet
    try:
        prophet_df = subdf.rename(columns={"Fecha":"ds","Ejecución":"y"})[["ds","y"]].dropna()

        if len(prophet_df) >= MIN_DATA_POINTS:
            m = Prophet(
                growth='linear',
                changepoint_prior_scale=0.01,
                yearly_seasonality=False,
                weekly_seasonality=False,
                daily_seasonality=False,
                interval_width=0.8
            )
            m.fit(prophet_df)

            future = pd.DataFrame({'ds': fechas_futuras})
            forecast = m.predict(future)

            for i, row in forecast.iterrows():
                valor = validar_proyeccion_realista(row["yhat"], stats, i+1)
                predicciones_por_periodo[i]['Prophet'] = valor

                opt_prophet = validar_proyeccion_realista(row["yhat_upper"], stats, i+1)
                pes_prophet = validar_proyeccion_realista(row["yhat_lower"], stats, i+1)

                # Limitar a rangos según tipo
                rango_max = valor * (1 + limites['max_confidence'])
                rango_min = valor * (1 - limites['max_confidence'])
                opt_prophet = min(opt_prophet, rango_max)
                pes_prophet = max(pes_prophet, rango_min)

                resultados.append([
                    id_ind, indicador, periodicidad, tipo_indicador, fechas_futuras[i], "Prophet",
                    round(valor, 2), round(pes_prophet, 2), round(opt_prophet, 2)
                ])

            print(f"   ✅ Prophet")
    except Exception as e:
        print(f"   ❌ Prophet: {str(e)[:30]}")

    # 6. Tendencia Histórica
    try:
        for i in range(horizonte):
            valor = stats['ultimo_valor'] * ((1 + stats['crecimiento_promedio_ponderado']) ** (i+1))
            valor = validar_proyeccion_realista(valor, stats, i+1)
            predicciones_por_periodo[i]['Tendencia_Historica'] = valor

            pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
            resultados.append([
                id_ind, indicador, periodicidad, tipo_indicador, fechas_futuras[i], "Tendencia_Historica",
                round(valor, 2), round(pes, 2), round(opt, 2)
            ])

        print(f"   ✅ Tendencia Histórica")
    except Exception as e:
        print(f"   ❌ Tendencia Histórica: {str(e)[:30]}")

    # ======================
    # ENSEMBLE PONDERADO
    # ======================
    for i, fecha in enumerate(fechas_futuras):
        preds = predicciones_por_periodo[i]

        if len(preds) == 0:
            continue

        pesos = calcular_pesos_ensemble(preds, stats)

        if len(pesos) > 0:
            base_ensemble = sum(preds[m] * pesos[m] for m in preds.keys())
            base_ensemble = validar_proyeccion_realista(base_ensemble, stats, i+1)
        else:
            base_ensemble = np.mean(list(preds.values()))

        pesimista, optimista = calcular_intervalos_adaptativos(base_ensemble, stats, i+1)

        resultados.append([
            id_ind, indicador, periodicidad, tipo_indicador, fecha, "Ensemble_Ponderado",
            round(base_ensemble, 2), round(pesimista, 2), round(optimista, 2)
        ])

    print(f"   📈 Proyección final (año 1): {base_ensemble:.2f}")

    return resultados

# ============================
# EJECUCIÓN
# ============================
print("="*70)
print("🚀 SISTEMA DE PROYECCIONES OPTIMIZADO V6 - CON VALIDACIÓN POR TIPO")
print("="*70)
print(f"📊 Configuración:")
print(f"   - Detección automática de tipo de indicador (%, ENT, DEC, $)")
print(f"   - Límites específicos por tipo:")
print(f"     • Porcentuales: Rango 0-100%, max crecimiento ±5%, intervalos 1-3%")
print(f"     • Enteros/Decimales: max ±15%, intervalos 3-8%")
print(f"     • Moneda: max ±15%, intervalos 3-8%")
print(f"   - Validaciones robustas contra extrapolaciones")
print(f"   - Detección de outliers activa")
print("="*70)

all_results = []
stats_summary = []
total = df["Id"].nunique()

for idx, (id_ind, subdf) in enumerate(df.groupby("Id"), 1):
    print(f"\n[{idx}/{total}]")

    res = proyectar_optimizado(subdf, horizonte=10)
    all_results.extend(res)

    if len(res) > 0:
        ts = subdf.set_index("Fecha")["Ejecución"].astype(float).replace(0, np.nan).dropna()
        if len(ts) >= MIN_DATA_POINTS:
            meta = subdf["Meta"].iloc[0]
            tipo_indicador = detectar_tipo_indicador(meta)
            _, outliers = detectar_outliers(ts.values)
            stats = calcular_estadisticas_robustas(ts, outliers, tipo_indicador)

            stats_summary.append({
                'Id': id_ind,
                'Indicador': subdf["Indicador"].iloc[0],
                'Tipo': tipo_indicador,
                'Meta': meta,
                'Tendencia': stats['tendencia'],
                'Ultimo_Valor': stats['ultimo_valor'],
                'Crecimiento_Mediano_%': stats['crecimiento_mediano'] * 100,
                'Crecimiento_Ponderado_%': stats['crecimiento_promedio_ponderado'] * 100,
                'Volatilidad_%': stats['volatilidad'] * 100,
                'Limite_Superior_%': stats['limite_superior'] * 100,
                'Limite_Inferior_%': stats['limite_inferior'] * 100,
                'Outliers_Detectados': len(outliers),
                'Puntos_Datos': len(ts)
            })

# ============================
# GUARDAR RESULTADOS
# ============================
print("\n" + "="*70)
print("💾 GUARDANDO RESULTADOS")
print("="*70)

if all_results:
    df_proy = pd.DataFrame(
        all_results,
        columns=["Id", "Indicador", "Periodicidad", "Tipo_Indicador", "Fecha_Proyeccion",
                 "Modelo", "Escenario_Base", "Escenario_Pesimista", "Escenario_Optimista"]
    )

    df_stats = pd.DataFrame(stats_summary)

    print(f"\n✅ RESUMEN:")
    print(f"   📈 Proyecciones totales: {len(df_proy):,}")
    print(f"   🏢 Indicadores: {df_proy['Indicador'].nunique()}")
    print(f"   🤖 Modelos: {df_proy['Modelo'].nunique()}")

    print(f"\n📊 DISTRIBUCIÓN POR MODELO:")
    for modelo, count in df_proy['Modelo'].value_counts().items():
        print(f"   {modelo:.<35} {count:>5,}")

    print(f"\n📋 DISTRIBUCIÓN POR TIPO DE INDICADOR:")
    for tipo, count in df_proy['Tipo_Indicador'].value_counts().items():
        limites = obtener_limites(tipo)
        print(f"   {tipo:.<15} {count:>5,} proyecciones | Límites: ±{limites['limite_superior']*100:.0f}%")

    # Validación de indicadores porcentuales
    df_porcentuales = df_proy[df_proy['Tipo_Indicador'] == '%'].copy()
    if len(df_porcentuales) > 0:
        valores_invalidos = df_porcentuales[
            (df_porcentuales['Escenario_Base'] > 100) |
            (df_porcentuales['Escenario_Base'] < 0) |
            (df_porcentuales['Escenario_Optimista'] > 100) |
            (df_porcentuales['Escenario_Pesimista'] < 0)
        ]

        if len(valores_invalidos) > 0:
            print(f"\n⚠️  ADVERTENCIA: {len(valores_invalidos)} proyecciones porcentuales fuera de rango 0-100%")
            print(f"   Se recomienda revisar: {valores_invalidos['Indicador'].unique()}")
        else:
            print(f"\n✅ Todos los indicadores porcentuales dentro del rango 0-100%")

    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        # Hoja 1: Todas las proyecciones
        df_proy.to_excel(writer, sheet_name='Proyecciones', index=False)

        # Hoja 2: Estadísticas robustas
        df_stats.to_excel(writer, sheet_name='Estadisticas_Robustas', index=False)

        # Hoja 3: Solo ensemble (recomendado)
        df_ensemble = df_proy[df_proy['Modelo'] == 'Ensemble_Ponderado'].copy()
        df_ensemble.to_excel(writer, sheet_name='Ensemble_Recomendado', index=False)

        # Hoja 4: Comparación modelos
        pivot = df_proy.pivot_table(
            index=['Indicador', 'Fecha_Proyeccion'],
            columns='Modelo',
            values='Escenario_Base'
        ).round(2)
        pivot.to_excel(writer, sheet_name='Comparacion_Modelos')

        # Hoja 5: Análisis por tipo de indicador
        resumen_tipo = df_proy.groupby(['Tipo_Indicador', 'Modelo']).agg({
            'Escenario_Base': ['mean', 'min', 'max', 'std'],
            'Id': 'count'
        }).round(2)
        resumen_tipo.to_excel(writer, sheet_name='Analisis_Por_Tipo')

        # Hoja 6: Indicadores porcentuales (validación)
        if len(df_porcentuales) > 0:
            df_porcentuales_ensemble = df_porcentuales[
                df_porcentuales['Modelo'] == 'Ensemble_Ponderado'
            ].copy()
            df_porcentuales_ensemble.to_excel(writer, sheet_name='Validacion_Porcentuales', index=False)

    print(f"\n✅ Archivo guardado: {OUTPUT_FILE}")
    print(f"   📋 Hojas creadas:")
    print(f"      1. Proyecciones (todas)")
    print(f"      2. Estadisticas_Robustas")
    print(f"      3. Ensemble_Recomendado")
    print(f"      4. Comparacion_Modelos")
    print(f"      5. Analisis_Por_Tipo")
    print(f"      6. Validacion_Porcentuales")

    # Mostrar algunos ejemplos de proyecciones
    print(f"\n📊 EJEMPLOS DE PROYECCIONES (Año 1 - Ensemble):")
    ejemplos = df_ensemble[df_ensemble['Fecha_Proyeccion'] == df_ensemble['Fecha_Proyeccion'].min()]
    for _, row in ejemplos.head(5).iterrows():
        tipo_sym = "📍" if row['Tipo_Indicador'] == '%' else "📊"
        print(f"   {tipo_sym} {row['Indicador'][:40]:.<45} {row['Escenario_Base']:>8.2f} [{row['Escenario_Pesimista']:.2f} - {row['Escenario_Optimista']:.2f}]")

else:
    print("\n❌ No se generaron proyecciones")

print("\n" + "="*70)
print("✅ PROCESO COMPLETADO")
print("="*70)
print("\n💡 NOTAS IMPORTANTES:")
print("   • Los indicadores porcentuales están limitados al rango 0-100%")
print("   • Los crecimientos máximos son del 5% anual para porcentuales")
print("   • Los intervalos de confianza son más estrechos para porcentuales (1-3%)")
print("   • Revise la hoja 'Validacion_Porcentuales' para verificar coherencia")
print("   • Use 'Ensemble_Recomendado' como proyección oficial")
print("="*70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 SISTEMA DE PROYECCIONES OPTIMIZADO V6 - CON VALIDACIÓN POR TIPO
📊 Configuración:
   - Detección automática de tipo de indicador (%, ENT, DEC, $)
   - Límites específicos por tipo:
     • Porcentuales: Rango 0-100%, max crecimiento ±5%, intervalos 1-3%
     • Enteros/Decimales: max ±15%, intervalos 3-8%
     • Moneda: max ±15%, intervalos 3-8%
   - Validaciones robustas contra extrapolaciones
   - Detección de outliers activa

[1/47]

📊 Total Población
   📌 Tipo: DEFAULT | Meta: nan
   Último: 56935.00 | Tendencia: estable
   Crec.Ponderado: 1.50% | Vol: 3.0%
   Límites: [-8.0%, 12.0%]
   ✅ ARIMA
   ❌ ETS: 'ETSResults' object has no att
   ✅ Holt-Winters
   ✅ Random Forest
   ❌ Prophet: 'Prophet' object has no attrib
   ✅ Tendencia Histórica
   📈 Proyección final (año 1): 58281.18

[2/47]

📊 Estudiantes Presencial
   📌 Tipo: DEFAULT | Meta: nan
   Último: 17

#Multimodelo V6

In [ ]:
# ============================
# IMPORTS
# ============================
import pandas as pd
import numpy as np
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')


# ============================
# CONFIGURACIÓN OPTIMIZADA
# ============================
MIN_DATA_POINTS = 6
OUTLIER_THRESHOLD = 3
RECENT_WEIGHT = 0.7

# LÍMITES DIFERENCIADOS POR TIPO DE INDICADOR
LIMITES_POR_TIPO = {
    '%': {  # Indicadores porcentuales
        'min_confidence': 0.01,      # 1%
        'max_confidence': 0.03,      # 3%
        'limite_superior': 0.05,     # 5% máximo crecimiento anual
        'limite_inferior': -0.05,    # -5% mínimo
        'volatilidad_max': 0.03,     # 3% volatilidad máxima
        'valor_min': 0,              # No puede ser negativo
        'valor_max': 100             # No puede superar 100%
    },
    'ENT': {  # Enteros (conteos, cantidades)
        'min_confidence': 0.03,
        'max_confidence': 0.08,
        'limite_superior': 0.15,
        'limite_inferior': -0.10,
        'volatilidad_max': 0.08,
        'valor_min': 0,
        'valor_max': None
    },
    'DEC': {  # Decimales
        'min_confidence': 0.03,
        'max_confidence': 0.08,
        'limite_superior': 0.12,
        'limite_inferior': -0.08,
        'volatilidad_max': 0.08,
        'valor_min': 0,
        'valor_max': None
    },
    '$': {  # Moneda
        'min_confidence': 0.03,
        'max_confidence': 0.08,
        'limite_superior': 0.15,
        'limite_inferior': -0.10,
        'volatilidad_max': 0.08,
        'valor_min': 0,
        'valor_max': None
    },
    'DEFAULT': {  # Por defecto
        'min_confidence': 0.03,
        'max_confidence': 0.08,
        'limite_superior': 0.12,
        'limite_inferior': -0.08,
        'volatilidad_max': 0.08,
        'valor_min': 0,
        'valor_max': None
    }
}


# ============================
# RUTAS
# ============================
INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Dataset_Unificado.xlsx"
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Proyecciones_Multimodelo_v6.xlsx"

# ============================
# LECTURA Y PREPARACIÓN
# ============================
df = pd.read_excel(INPUT_FILE, sheet_name="Unificado")
df = df[["Id", "Indicador", "Periodicidad", "Fecha", "Ejecución", "Meta"]].copy()
df["Fecha"] = pd.to_datetime(df["Fecha"])
df = df.sort_values(["Id", "Fecha"])

# ============================
# FUNCIÓN PARA DETECTAR TIPO DE INDICADOR
# ============================
def detectar_tipo_indicador(meta_value):
    """
    Detecta el tipo de indicador basado en la columna Meta
    """
    if pd.isna(meta_value):
        return 'DEFAULT'

    meta_str = str(meta_value).upper().strip()

    if '%' in meta_str:
        return '%'
    elif 'ENT' in meta_str:
        return 'ENT'
    elif 'DEC' in meta_str:
        return 'DEC'
    elif '$' in meta_str:
        return '$'
    else:
        return 'DEFAULT'

def obtener_limites(tipo_indicador):
    """
    Obtiene los límites específicos para el tipo de indicador
    """
    return LIMITES_POR_TIPO.get(tipo_indicador, LIMITES_POR_TIPO['DEFAULT'])

# ============================
# FUNCIONES AUXILIARES OPTIMIZADAS
# ============================

def detectar_outliers(ts, threshold=OUTLIER_THRESHOLD):
    """Detecta y marca outliers usando Z-score"""
    if len(ts) < 4:
        return ts, []

    z_scores = np.abs(scipy_stats.zscore(ts))
    outliers = np.where(z_scores > threshold)[0]

    return ts, outliers.tolist()

def calcular_estadisticas_robustas(ts, outliers=[], tipo_indicador='DEFAULT'):
    """
    Calcula estadísticas robustas considerando el tipo de indicador
    """
    limites = obtener_limites(tipo_indicador)

    # Crear serie sin outliers
    ts_clean = ts.copy()
    if len(outliers) > 0:
        ts_clean = ts_clean.drop(ts_clean.index[outliers])

    if len(ts_clean) < 3:
        ts_clean = ts

    # Crecimientos (usando mediana para robustez)
    crecimientos = ts_clean.pct_change().dropna()

    if len(crecimientos) == 0:
        return {
            'crecimiento_mediano': 0,
            'crecimiento_promedio_ponderado': 0,
            'volatilidad': limites['min_confidence'],
            'tendencia': 'estable',
            'limite_superior': limites['limite_superior'],
            'limite_inferior': limites['limite_inferior'],
            'ultimo_valor': ts.iloc[-1],
            'n_datos': len(ts),
            'tipo_indicador': tipo_indicador
        }

    # Mediana (más robusta que promedio)
    crecimiento_mediano = crecimientos.median()

    # Promedio ponderado (más peso a datos recientes)
    n = len(crecimientos)
    pesos = np.array([RECENT_WEIGHT ** (n - i - 1) for i in range(n)])
    pesos = pesos / pesos.sum()
    crecimiento_ponderado = np.average(crecimientos, weights=pesos)

    # Volatilidad (usando MAD - Median Absolute Deviation)
    mad = np.median(np.abs(crecimientos - crecimiento_mediano))
    volatilidad = mad * 1.4826

    # LIMITAR VOLATILIDAD SEGÚN TIPO
    volatilidad = np.clip(volatilidad, limites['min_confidence'], limites['volatilidad_max'])

    # Detectar tendencia CON LÍMITES ESPECÍFICOS
    if crecimiento_ponderado > 0.02:
        tendencia = 'creciente'
        limite_superior = min(crecimiento_ponderado * 1.5, limites['limite_superior'])
        limite_inferior = max(crecimiento_ponderado * 0.3, limites['limite_inferior'])
    elif crecimiento_ponderado < -0.02:
        tendencia = 'decreciente'
        limite_superior = max(crecimiento_ponderado * 0.3, limites['limite_superior'] * 0.5)
        limite_inferior = max(crecimiento_ponderado * 1.5, limites['limite_inferior'])
    else:
        tendencia = 'estable'
        limite_superior = limites['limite_superior']
        limite_inferior = limites['limite_inferior']

    return {
        'crecimiento_mediano': crecimiento_mediano,
        'crecimiento_promedio_ponderado': crecimiento_ponderado,
        'volatilidad': volatilidad,
        'tendencia': tendencia,
        'limite_superior': limite_superior,
        'limite_inferior': limite_inferior,
        'ultimo_valor': ts.iloc[-1],
        'n_datos': len(ts_clean),
        'tipo_indicador': tipo_indicador
    }

def validar_proyeccion_realista(valor_proyectado, stats, periodo):
    """
    Valida que la proyección sea realista según el tipo de indicador
    """
    ultimo_valor = stats['ultimo_valor']
    tipo_indicador = stats['tipo_indicador']
    limites = obtener_limites(tipo_indicador)

    # VALIDACIÓN CRÍTICA PARA PORCENTUALES
    if tipo_indicador == '%':
        # Forzar rango 0-100%
        valor_proyectado = np.clip(valor_proyectado, limites['valor_min'], limites['valor_max'])

        # Si ya está cerca del límite, limitar crecimiento
        if ultimo_valor >= 95:
            valor_proyectado = min(valor_proyectado, 100)
        elif ultimo_valor <= 5:
            valor_proyectado = max(valor_proyectado, 0)

    # VALIDACIÓN DE CRECIMIENTO
    if ultimo_valor <= 0 or valor_proyectado <= 0:
        return max(limites['valor_min'], valor_proyectado) if limites['valor_min'] is not None else max(0, valor_proyectado)

    try:
        crecimiento_total = (valor_proyectado / ultimo_valor) - 1
        crecimiento_anual = crecimiento_total / max(periodo, 1)
    except:
        return ultimo_valor

    # Aplicar límites adaptativos más estrictos para porcentuales
    factor_horizonte = 1 + (periodo * 0.03) if tipo_indicador == '%' else 1 + (periodo * 0.05)
    limite_sup_ajustado = stats['limite_superior'] / factor_horizonte
    limite_inf_ajustado = stats['limite_inferior'] / factor_horizonte

    if crecimiento_anual > limite_sup_ajustado:
        valor_corregido = ultimo_valor * (1 + limite_sup_ajustado) ** periodo
        # Para porcentuales, asegurar que no supere 100
        if tipo_indicador == '%':
            valor_corregido = min(valor_corregido, 100)
        return valor_corregido
    elif crecimiento_anual < limite_inf_ajustado:
        valor_corregido = ultimo_valor * (1 + limite_inf_ajustado) ** periodo
        return max(limites['valor_min'] if limites['valor_min'] is not None else 0, valor_corregido)

    # Validar límite máximo si existe
    if limites['valor_max'] is not None:
        valor_proyectado = min(valor_proyectado, limites['valor_max'])

    return valor_proyectado

def calcular_intervalos_adaptativos(base, stats, periodo):
    """
    Calcula intervalos de confianza adaptativos según tipo de indicador
    """
    tipo_indicador = stats['tipo_indicador']
    limites = obtener_limites(tipo_indicador)

    # Usar volatilidad histórica pero limitada según tipo
    vol_base = min(stats['volatilidad'], limites['volatilidad_max'])

    # Ampliar LIGERAMENTE con el horizonte (más conservador para porcentuales)
    factor_crecimiento = 0.003 if tipo_indicador == '%' else 0.005
    factor_horizonte = 1 + (periodo * factor_crecimiento)
    volatilidad_ajustada = vol_base * factor_horizonte

    # Límite según tipo
    volatilidad_ajustada = np.clip(volatilidad_ajustada,
                                   limites['min_confidence'],
                                   limites['max_confidence'])

    # Calcular intervalos
    optimista = base * (1 + volatilidad_ajustada)
    pesimista = base * (1 - volatilidad_ajustada)

    # VALIDACIONES ESPECÍFICAS POR TIPO
    if tipo_indicador == '%':
        # Para porcentuales, forzar rango 0-100
        optimista = min(optimista, 100)
        pesimista = max(pesimista, 0)

        # Si la base está cerca de los límites, ajustar intervalos asimétricamente
        if base >= 95:
            optimista = min(optimista, 100)
            pesimista = max(base * 0.97, 0)
        elif base <= 5:
            pesimista = max(pesimista, 0)
            optimista = min(base * 1.03, 100)

    # Asegurar que pesimista no sea negativo
    pesimista = max(limites['valor_min'] if limites['valor_min'] is not None else 0, pesimista)

    return pesimista, optimista

def calcular_pesos_ensemble(predicciones, stats):
    """
    Calcula pesos para el ensemble basados en cercanía a la tendencia histórica
    """
    if len(predicciones) == 0:
        return {}

    ultimo_valor = stats['ultimo_valor']
    crec_esperado = stats['crecimiento_promedio_ponderado']

    pesos = {}
    for modelo, valor in predicciones.items():
        if ultimo_valor > 0:
            crec_implicito = (valor / ultimo_valor) - 1
            diferencia = abs(crec_implicito - crec_esperado)
            # Menor diferencia = mayor peso
            pesos[modelo] = 1 / (1 + diferencia * 10)
        else:
            pesos[modelo] = 1

    # Normalizar
    suma_pesos = sum(pesos.values())
    if suma_pesos > 0:
        pesos = {k: v/suma_pesos for k, v in pesos.items()}

    return pesos

def crear_features_ml_optimizado(ts, n_lags=3):
    """
    Crear features con mejor ingeniería para ML
    """
    if len(ts) < n_lags + 3:
        return None, None

    features = []
    target = []

    for i in range(n_lags, len(ts)):
        try:
            # Lags
            lags = [ts.iloc[i-j] for j in range(1, n_lags+1)]

            # Medias móviles
            ma_3 = ts.iloc[max(0, i-3):i].mean()
            ma_6 = ts.iloc[max(0, i-6):i].mean() if i >= 6 else ma_3

            # Volatilidad reciente
            vol = ts.iloc[max(0, i-3):i].std() if i >= 3 else 0

            # Momentum
            pct_change = (ts.iloc[i-1] - ts.iloc[i-2]) / ts.iloc[i-2] if ts.iloc[i-2] != 0 else 0

            # Aceleración
            if i >= 3:
                accel = ((ts.iloc[i-1] - ts.iloc[i-2]) - (ts.iloc[i-2] - ts.iloc[i-3])) / ts.iloc[i-2] if ts.iloc[i-2] != 0 else 0
            else:
                accel = 0

            # Tendencia lineal
            if i >= 3:
                x_trend = np.arange(3)
                y_trend = ts.iloc[i-3:i].values
                if len(y_trend) == 3:
                    slope = np.polyfit(x_trend, y_trend, 1)[0]
                else:
                    slope = 0
            else:
                slope = 0

            feature_row = lags + [ma_3, ma_6, vol, pct_change, accel, slope]
            features.append(feature_row)
            target.append(ts.iloc[i])
        except:
            continue

    if len(features) == 0:
        return None, None

    return np.array(features), np.array(target)

# ============================
# FUNCIÓN PRINCIPAL OPTIMIZADA
# ============================
def proyectar_optimizado(subdf, horizonte=10):
    """
    Sistema de proyecciones optimizado con validaciones por tipo de indicador
    """
    resultados = []
    id_ind = subdf["Id"].iloc[0]
    indicador = subdf["Indicador"].iloc[0]
    periodicidad = subdf["Periodicidad"].iloc[0]
    meta = subdf["Meta"].iloc[0]

    # DETECTAR TIPO DE INDICADOR
    tipo_indicador = detectar_tipo_indicador(meta)
    limites = obtener_limites(tipo_indicador)

    # Validar datos mínimos
    if len(subdf) < MIN_DATA_POINTS:
        print(f"⚠️  {indicador}: Datos insuficientes ({len(subdf)} < {MIN_DATA_POINTS})")
        return resultados

    # Preparar serie temporal
    ts = subdf.set_index("Fecha")["Ejecución"].astype(float)
    ts = ts.replace(0, np.nan).dropna()

    if len(ts) < MIN_DATA_POINTS:
        print(f"⚠️  {indicador}: Datos válidos insuficientes")
        return resultados

    # Detectar outliers
    ts_values, outliers = detectar_outliers(ts.values)

    # Calcular estadísticas robustas CON TIPO DE INDICADOR
    stats = calcular_estadisticas_robustas(ts, outliers, tipo_indicador)

    # Generar fechas futuras
    if periodicidad == "Semestral":
        freq = "6MS"
    else:
        freq = "AS"

    try:
        fechas_futuras = pd.date_range(
            start=ts.index[-1] + pd.DateOffset(months=6 if periodicidad=="Semestral" else 12),
            periods=horizonte,
            freq=freq
        )
    except:
        print(f"⚠️  {indicador}: Error generando fechas")
        return resultados

    print(f"\n📊 {indicador}")
    print(f"   📌 Tipo: {tipo_indicador} | Meta: {meta}")
    print(f"   Último: {stats['ultimo_valor']:.2f} | Tendencia: {stats['tendencia']}")
    print(f"   Crec.Ponderado: {stats['crecimiento_promedio_ponderado']*100:.2f}% | Vol: {stats['volatilidad']*100:.1f}%")
    print(f"   Límites: [{stats['limite_inferior']*100:.1f}%, {stats['limite_superior']*100:.1f}%]")
    if tipo_indicador == '%':
        print(f"   ⚠️  INDICADOR PORCENTUAL - Validación 0-100% activa")
    if outliers:
        print(f"   ⚠️  Outliers detectados: {len(outliers)}")

    predicciones_por_periodo = {i: {} for i in range(horizonte)}

    # ======================
    # MODELOS DE PROYECCIÓN
    # ======================

    # 1. ARIMA
    try:
        modelo = ARIMA(ts, order=(1,1,1))
        fit = modelo.fit()
        pred = fit.get_forecast(steps=horizonte)
        forecast = pred.predicted_mean
        conf_int = pred.conf_int(alpha=0.32)

        for i, fecha in enumerate(fechas_futuras):
            valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
            predicciones_por_periodo[i]['ARIMA'] = valor

            # CALCULAR INTERVALOS ADAPTATIVOS (no usar los de ARIMA directamente)
            pes_arima, opt_arima = calcular_intervalos_adaptativos(valor, stats, i+1)

            # Validar que estén dentro de límites
            pes_arima = validar_proyeccion_realista(pes_arima, stats, i+1)
            opt_arima = validar_proyeccion_realista(opt_arima, stats, i+1)

            resultados.append([
                id_ind, indicador, periodicidad, tipo_indicador, fecha, "ARIMA",
                round(valor, 2), round(pes_arima, 2), round(opt_arima, 2)
            ])

        print(f"   ✅ ARIMA")
    except Exception as e:
        print(f"   ❌ ARIMA: {str(e)[:30]}")

    # 2. ETS
    try:
        modelo = ETSModel(ts, error='add', trend='add', seasonal=None, damped_trend=True)
        fit = modelo.fit(maxiter=1000, disp=False)
        pred = fit.get_forecast(horizonte)
        forecast = pred.predicted_mean

        for i, fecha in enumerate(fechas_futuras):
            valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
            predicciones_por_periodo[i]['ETS'] = valor

            # Calcular intervalos y validar
            pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
            pes = validar_proyeccion_realista(pes, stats, i+1)
            opt = validar_proyeccion_realista(opt, stats, i+1)

            resultados.append([
                id_ind, indicador, periodicidad, tipo_indicador, fecha, "ETS",
                round(valor, 2), round(pes, 2), round(opt, 2)
            ])

        print(f"   ✅ ETS")
    except Exception as e:
        print(f"   ❌ ETS: {str(e)[:30]}")

    # 3. Holt-Winters
    try:
        if len(ts) >= 8:
            modelo = ExponentialSmoothing(ts, trend='add', seasonal=None, damped_trend=True)
            fit = modelo.fit()
            forecast = fit.forecast(horizonte)

            for i, fecha in enumerate(fechas_futuras):
                valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
                predicciones_por_periodo[i]['Holt_Winters'] = valor

                # Calcular intervalos y validar
                pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
                pes = validar_proyeccion_realista(pes, stats, i+1)
                opt = validar_proyeccion_realista(opt, stats, i+1)

                resultados.append([
                    id_ind, indicador, periodicidad, tipo_indicador, fecha, "Holt_Winters",
                    round(valor, 2), round(pes, 2), round(opt, 2)
                ])

            print(f"   ✅ Holt-Winters")
    except Exception as e:
        print(f"   ❌ Holt-Winters: {str(e)[:30]}")

    # 4. Random Forest
    try:
        X, y = crear_features_ml_optimizado(ts, n_lags=3)

        if X is not None and len(X) >= 5:
            modelo = RandomForestRegressor(
                n_estimators=100,
                max_depth=5,
                min_samples_split=3,
                min_samples_leaf=2,
                random_state=42
            )
            modelo.fit(X, y)

            ts_extended = ts.copy()

            for periodo in range(horizonte):
                if len(ts_extended) >= 6:
                    lags = [ts_extended.iloc[-1], ts_extended.iloc[-2], ts_extended.iloc[-3]]
                    ma_3 = ts_extended.iloc[-3:].mean()
                    ma_6 = ts_extended.iloc[-6:].mean() if len(ts_extended) >= 6 else ma_3
                    vol = ts_extended.iloc[-3:].std()
                    pct = (ts_extended.iloc[-1] - ts_extended.iloc[-2])/ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0

                    if len(ts_extended) >= 3:
                        accel = ((ts_extended.iloc[-1] - ts_extended.iloc[-2]) - (ts_extended.iloc[-2] - ts_extended.iloc[-3])) / ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0
                        x_trend = np.arange(3)
                        y_trend = ts_extended.iloc[-3:].values
                        slope = np.polyfit(x_trend, y_trend, 1)[0]
                    else:
                        accel = 0
                        slope = 0

                    features = [lags + [ma_3, ma_6, vol, pct, accel, slope]]
                    pred = modelo.predict(features)[0]
                    pred = validar_proyeccion_realista(pred, stats, periodo+1)

                    predicciones_por_periodo[periodo]['Random_Forest'] = pred
                    ts_extended = pd.concat([ts_extended, pd.Series([pred])])

            for i, fecha in enumerate(fechas_futuras):
                if i in predicciones_por_periodo and 'Random_Forest' in predicciones_por_periodo[i]:
                    valor = predicciones_por_periodo[i]['Random_Forest']

                    # Calcular intervalos y validar
                    pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
                    pes = validar_proyeccion_realista(pes, stats, i+1)
                    opt = validar_proyeccion_realista(opt, stats, i+1)

                    resultados.append([
                        id_ind, indicador, periodicidad, tipo_indicador, fecha, "Random_Forest",
                        round(valor, 2), round(pes, 2), round(opt, 2)
                    ])

            print(f"   ✅ Random Forest")
    except Exception as e:
        print(f"   ❌ Random Forest: {str(e)[:30]}")

    # 5. Prophet
    try:
        prophet_df = subdf.rename(columns={"Fecha":"ds","Ejecución":"y"})[["ds","y"]].dropna()

        if len(prophet_df) >= MIN_DATA_POINTS:
            m = Prophet(
                growth='linear',
                changepoint_prior_scale=0.01,
                yearly_seasonality=False,
                weekly_seasonality=False,
                daily_seasonality=False,
                interval_width=0.8
            )
            m.fit(prophet_df)

            future = pd.DataFrame({'ds': fechas_futuras})
            forecast = m.predict(future)

            for i, row in forecast.iterrows():
                valor = validar_proyeccion_realista(row["yhat"], stats, i+1)
                predicciones_por_periodo[i]['Prophet'] = valor

                # Calcular intervalos adaptativos (ignorar los de Prophet)
                pes_prophet, opt_prophet = calcular_intervalos_adaptativos(valor, stats, i+1)

                # Validar que estén dentro de límites
                pes_prophet = validar_proyeccion_realista(pes_prophet, stats, i+1)
                opt_prophet = validar_proyeccion_realista(opt_prophet, stats, i+1)

                resultados.append([
                    id_ind, indicador, periodicidad, tipo_indicador, fechas_futuras[i], "Prophet",
                    round(valor, 2), round(pes_prophet, 2), round(opt_prophet, 2)
                ])

            print(f"   ✅ Prophet")
    except Exception as e:
        print(f"   ❌ Prophet: {str(e)[:30]}")

    # 6. Tendencia Histórica
    try:
        for i in range(horizonte):
            valor = stats['ultimo_valor'] * ((1 + stats['crecimiento_promedio_ponderado']) ** (i+1))
            valor = validar_proyeccion_realista(valor, stats, i+1)
            predicciones_por_periodo[i]['Tendencia_Historica'] = valor

            # Calcular intervalos y validar
            pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
            pes = validar_proyeccion_realista(pes, stats, i+1)
            opt = validar_proyeccion_realista(opt, stats, i+1)

            resultados.append([
                id_ind, indicador, periodicidad, tipo_indicador, fechas_futuras[i], "Tendencia_Historica",
                round(valor, 2), round(pes, 2), round(opt, 2)
            ])

        print(f"   ✅ Tendencia Histórica")
    except Exception as e:
        print(f"   ❌ Tendencia Histórica: {str(e)[:30]}")

    # ======================
    # ENSEMBLE PONDERADO
    # ======================
    for i, fecha in enumerate(fechas_futuras):
        preds = predicciones_por_periodo[i]

        if len(preds) == 0:
            continue

        pesos = calcular_pesos_ensemble(preds, stats)

        if len(pesos) > 0:
            base_ensemble = sum(preds[m] * pesos[m] for m in preds.keys())
            base_ensemble = validar_proyeccion_realista(base_ensemble, stats, i+1)
        else:
            base_ensemble = np.mean(list(preds.values()))

        # Calcular intervalos y validar
        pesimista, optimista = calcular_intervalos_adaptativos(base_ensemble, stats, i+1)
        pesimista = validar_proyeccion_realista(pesimista, stats, i+1)
        optimista = validar_proyeccion_realista(optimista, stats, i+1)

        resultados.append([
            id_ind, indicador, periodicidad, tipo_indicador, fecha, "Ensemble_Ponderado",
            round(base_ensemble, 2), round(pesimista, 2), round(optimista, 2)
        ])

    print(f"   📈 Proyección final (año 1): {base_ensemble:.2f}")

    return resultados

# ============================
# EJECUCIÓN
# ============================
print("="*70)
print("🚀 SISTEMA DE PROYECCIONES OPTIMIZADO V6 - CON VALIDACIÓN POR TIPO")
print("="*70)
print(f"📊 Configuración:")
print(f"   - Detección automática de tipo de indicador (%, ENT, DEC, $)")
print(f"   - Límites específicos por tipo:")
print(f"     • Porcentuales: Rango 0-100%, max crecimiento ±5%, intervalos 1-3%")
print(f"     • Enteros/Decimales: max ±15%, intervalos 3-8%")
print(f"     • Moneda: max ±15%, intervalos 3-8%")
print(f"   - Validaciones robustas contra extrapolaciones")
print(f"   - Detección de outliers activa")
print("="*70)

all_results = []
stats_summary = []
total = df["Id"].nunique()

for idx, (id_ind, subdf) in enumerate(df.groupby("Id"), 1):
    print(f"\n[{idx}/{total}]")

    res = proyectar_optimizado(subdf, horizonte=10)
    all_results.extend(res)

    if len(res) > 0:
        ts = subdf.set_index("Fecha")["Ejecución"].astype(float).replace(0, np.nan).dropna()
        if len(ts) >= MIN_DATA_POINTS:
            meta = subdf["Meta"].iloc[0]
            tipo_indicador = detectar_tipo_indicador(meta)
            _, outliers = detectar_outliers(ts.values)
            stats = calcular_estadisticas_robustas(ts, outliers, tipo_indicador)

            stats_summary.append({
                'Id': id_ind,
                'Indicador': subdf["Indicador"].iloc[0],
                'Tipo': tipo_indicador,
                'Meta': meta,
                'Tendencia': stats['tendencia'],
                'Ultimo_Valor': stats['ultimo_valor'],
                'Crecimiento_Mediano_%': stats['crecimiento_mediano'] * 100,
                'Crecimiento_Ponderado_%': stats['crecimiento_promedio_ponderado'] * 100,
                'Volatilidad_%': stats['volatilidad'] * 100,
                'Limite_Superior_%': stats['limite_superior'] * 100,
                'Limite_Inferior_%': stats['limite_inferior'] * 100,
                'Outliers_Detectados': len(outliers),
                'Puntos_Datos': len(ts)
            })

# ============================
# GUARDAR RESULTADOS
# ============================
print("\n" + "="*70)
print("💾 GUARDANDO RESULTADOS")
print("="*70)

if all_results:
    df_proy = pd.DataFrame(
        all_results,
        columns=["Id", "Indicador", "Periodicidad", "Tipo_Indicador", "Fecha_Proyeccion",
                 "Modelo", "Escenario_Base", "Escenario_Pesimista", "Escenario_Optimista"]
    )

    df_stats = pd.DataFrame(stats_summary)

    print(f"\n✅ RESUMEN:")
    print(f"   📈 Proyecciones totales: {len(df_proy):,}")
    print(f"   🏢 Indicadores: {df_proy['Indicador'].nunique()}")
    print(f"   🤖 Modelos: {df_proy['Modelo'].nunique()}")

    print(f"\n📊 DISTRIBUCIÓN POR MODELO:")
    for modelo, count in df_proy['Modelo'].value_counts().items():
        print(f"   {modelo:.<35} {count:>5,}")

    print(f"\n📋 DISTRIBUCIÓN POR TIPO DE INDICADOR:")
    for tipo, count in df_proy['Tipo_Indicador'].value_counts().items():
        limites = obtener_limites(tipo)
        print(f"   {tipo:.<15} {count:>5,} proyecciones | Límites: ±{limites['limite_superior']*100:.0f}%")

    # Validación de indicadores porcentuales
    df_porcentuales = df_proy[df_proy['Tipo_Indicador'] == '%'].copy()
    if len(df_porcentuales) > 0:
        valores_invalidos = df_porcentuales[
            (df_porcentuales['Escenario_Base'] > 100) |
            (df_porcentuales['Escenario_Base'] < 0) |
            (df_porcentuales['Escenario_Optimista'] > 100) |
            (df_porcentuales['Escenario_Pesimista'] < 0)
        ]

        if len(valores_invalidos) > 0:
            print(f"\n⚠️  ADVERTENCIA: {len(valores_invalidos)} proyecciones porcentuales fuera de rango 0-100%")
            print(f"   Se recomienda revisar: {valores_invalidos['Indicador'].unique()}")
        else:
            print(f"\n✅ Todos los indicadores porcentuales dentro del rango 0-100%")

    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        # Hoja 1: Todas las proyecciones
        df_proy.to_excel(writer, sheet_name='Proyecciones', index=False)

        # Hoja 2: Estadísticas robustas
        df_stats.to_excel(writer, sheet_name='Estadisticas_Robustas', index=False)

        # Hoja 3: Solo ensemble (recomendado)
        df_ensemble = df_proy[df_proy['Modelo'] == 'Ensemble_Ponderado'].copy()
        df_ensemble.to_excel(writer, sheet_name='Ensemble_Recomendado', index=False)

        # Hoja 4: Comparación modelos
        pivot = df_proy.pivot_table(
            index=['Indicador', 'Fecha_Proyeccion'],
            columns='Modelo',
            values='Escenario_Base'
        ).round(2)
        pivot.to_excel(writer, sheet_name='Comparacion_Modelos')

        # Hoja 5: Análisis por tipo de indicador
        resumen_tipo = df_proy.groupby(['Tipo_Indicador', 'Modelo']).agg({
            'Escenario_Base': ['mean', 'min', 'max', 'std'],
            'Id': 'count'
        }).round(2)
        resumen_tipo.to_excel(writer, sheet_name='Analisis_Por_Tipo')

        # Hoja 6: Indicadores porcentuales (validación)
        if len(df_porcentuales) > 0:
            df_porcentuales_ensemble = df_porcentuales[
                df_porcentuales['Modelo'] == 'Ensemble_Ponderado'
            ].copy()
            df_porcentuales_ensemble.to_excel(writer, sheet_name='Validacion_Porcentuales', index=False)

    print(f"\n✅ Archivo guardado: {OUTPUT_FILE}")
    print(f"   📋 Hojas creadas:")
    print(f"      1. Proyecciones (todas)")
    print(f"      2. Estadisticas_Robustas")
    print(f"      3. Ensemble_Recomendado")
    print(f"      4. Comparacion_Modelos")
    print(f"      5. Analisis_Por_Tipo")
    print(f"      6. Validacion_Porcentuales")

    # Mostrar algunos ejemplos de proyecciones
    print(f"\n📊 EJEMPLOS DE PROYECCIONES (Año 1 - Ensemble):")
    ejemplos = df_ensemble[df_ensemble['Fecha_Proyeccion'] == df_ensemble['Fecha_Proyeccion'].min()]
    for _, row in ejemplos.head(5).iterrows():
        tipo_sym = "📍" if row['Tipo_Indicador'] == '%' else "📊"
        print(f"   {tipo_sym} {row['Indicador'][:40]:.<45} {row['Escenario_Base']:>8.2f} [{row['Escenario_Pesimista']:.2f} - {row['Escenario_Optimista']:.2f}]")

else:
    print("\n❌ No se generaron proyecciones")

print("\n" + "="*70)
print("✅ PROCESO COMPLETADO")
print("="*70)
print("\n💡 NOTAS IMPORTANTES:")
print("   • Los indicadores porcentuales están limitados al rango 0-100%")
print("   • Los crecimientos máximos son del 5% anual para porcentuales")
print("   • Los intervalos de confianza son más estrechos para porcentuales (1-3%)")
print("   • Revise la hoja 'Validacion_Porcentuales' para verificar coherencia")
print("   • Use 'Ensemble_Recomendado' como proyección oficial")
print("="*70)

🚀 SISTEMA DE PROYECCIONES OPTIMIZADO V6 - CON VALIDACIÓN POR TIPO
📊 Configuración:
   - Detección automática de tipo de indicador (%, ENT, DEC, $)
   - Límites específicos por tipo:
     • Porcentuales: Rango 0-100%, max crecimiento ±5%, intervalos 1-3%
     • Enteros/Decimales: max ±15%, intervalos 3-8%
     • Moneda: max ±15%, intervalos 3-8%
   - Validaciones robustas contra extrapolaciones
   - Detección de outliers activa

[1/47]

📊 Total Población
   📌 Tipo: DEFAULT | Meta: nan
   Último: 56935.00 | Tendencia: estable
   Crec.Ponderado: 1.50% | Vol: 3.0%
   Límites: [-8.0%, 12.0%]
   ✅ ARIMA
   ❌ ETS: 'ETSResults' object has no att
   ✅ Holt-Winters
   ✅ Random Forest
   ❌ Prophet: 'Prophet' object has no attrib
   ✅ Tendencia Histórica
   📈 Proyección final (año 1): 58281.18

[2/47]

📊 Estudiantes Presencial
   📌 Tipo: DEFAULT | Meta: nan
   Último: 17485.00 | Tendencia: creciente
   Crec.Ponderado: 35.16% | Vol: 8.0%
   Límites: [10.5%, 12.0%]
   ✅ ARIMA
   ❌ ETS: 'ETSResults' 

#Multimodelo V7

In [ ]:
# ============================
# IMPORTS
# ============================
import pandas as pd
import numpy as np
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import RandomForestRegressor
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

# ============================
# CONFIGURACIÓN OPTIMIZADA
# ============================
MIN_DATA_POINTS = 6
OUTLIER_THRESHOLD = 3
RECENT_WEIGHT = 0.7

# LÍMITES DIFERENCIADOS POR TIPO DE INDICADOR
LIMITES_POR_TIPO = {
    '%': {
        'min_confidence': 0.01,
        'max_confidence': 0.03,
        'limite_superior': 0.05,
        'limite_inferior': -0.05,
        'volatilidad_max': 0.03,
        'valor_min': 0,
        'valor_max': 100
    },
    'ENT': {
        'min_confidence': 0.03,
        'max_confidence': 0.08,
        'limite_superior': 0.15,
        'limite_inferior': -0.10,
        'volatilidad_max': 0.08,
        'valor_min': 0,
        'valor_max': None
    },
    'DEC': {
        'min_confidence': 0.03,
        'max_confidence': 0.08,
        'limite_superior': 0.12,
        'limite_inferior': -0.08,
        'volatilidad_max': 0.08,
        'valor_min': 0,
        'valor_max': None
    },
    '$': {
        'min_confidence': 0.03,
        'max_confidence': 0.08,
        'limite_superior': 0.15,
        'limite_inferior': -0.10,
        'volatilidad_max': 0.08,
        'valor_min': 0,
        'valor_max': None
    },
    'DEFAULT': {
        'min_confidence': 0.03,
        'max_confidence': 0.08,
        'limite_superior': 0.12,
        'limite_inferior': -0.08,
        'volatilidad_max': 0.08,
        'valor_min': 0,
        'valor_max': None
    }
}


# ============================
# RUTAS
# ============================
INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Dataset_Unificado.xlsx"
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/Indicadores/Proyecciones_Multimodelo_v7.xlsx"

# ============================
# LECTURA Y PREPARACIÓN
# ============================
df = pd.read_excel(INPUT_FILE, sheet_name="Unificado")

columnas_requeridas = ["Id", "Indicador", "Periodicidad", "Fecha", "Ejecución", "Meta"]
if "Meta" not in df.columns:
    print("⚠️ ADVERTENCIA: No se encontró la columna 'Meta'")
    df["Meta"] = "DEFAULT"

df = df[columnas_requeridas].copy()
df["Fecha"] = pd.to_datetime(df["Fecha"])
df = df.sort_values(["Id", "Fecha"])

print("\n📋 MUESTRA DE VALORES EN COLUMNA META:")
print(df[["Indicador", "Meta"]].drop_duplicates().head(10))
print(f"\n📊 Tipos de Meta detectados: {df['Meta'].value_counts().to_dict()}")

# ============================
# FUNCIONES AUXILIARES
# ============================

def detectar_tipo_indicador(meta_value, valores_historicos=None):
    """Detecta el tipo de indicador basado en Meta y valores históricos"""
    if pd.isna(meta_value):
        tipo_por_meta = 'DEFAULT'
    else:
        meta_str = str(meta_value).upper().strip()
        if '%' in meta_str or 'PORCENTAJE' in meta_str:
            tipo_por_meta = '%'
        elif 'ENT' in meta_str:
            tipo_por_meta = 'ENT'
        elif 'DEC' in meta_str:
            tipo_por_meta = 'DEC'
        elif '$' in meta_str or 'PESO' in meta_str:
            tipo_por_meta = '$'
        else:
            tipo_por_meta = 'DEFAULT'

    if valores_historicos is not None and len(valores_historicos) > 0:
        valores_clean = valores_historicos.dropna()
        if len(valores_clean) > 0:
            min_val = valores_clean.min()
            max_val = valores_clean.max()
            if min_val >= 0 and max_val <= 100 and max_val > 10:
                return '%'

    return tipo_por_meta

def obtener_limites(tipo_indicador):
    """Obtiene los límites específicos para el tipo de indicador"""
    return LIMITES_POR_TIPO.get(tipo_indicador, LIMITES_POR_TIPO['DEFAULT'])

def detectar_outliers(ts, threshold=OUTLIER_THRESHOLD):
    """Detecta y marca outliers usando Z-score"""
    if len(ts) < 4:
        return ts, []
    z_scores = np.abs(scipy_stats.zscore(ts))
    outliers = np.where(z_scores > threshold)[0]
    return ts, outliers.tolist()

def calcular_estadisticas_robustas(ts, outliers=[], tipo_indicador='DEFAULT'):
    """Calcula estadísticas robustas considerando el tipo de indicador"""
    limites = obtener_limites(tipo_indicador)

    ts_clean = ts.copy()
    if len(outliers) > 0:
        ts_clean = ts_clean.drop(ts_clean.index[outliers])

    if len(ts_clean) < 3:
        ts_clean = ts

    crecimientos = ts_clean.pct_change().dropna()

    if len(crecimientos) == 0:
        return {
            'crecimiento_mediano': 0,
            'crecimiento_promedio_ponderado': 0,
            'volatilidad': limites['min_confidence'],
            'tendencia': 'estable',
            'limite_superior': limites['limite_superior'],
            'limite_inferior': limites['limite_inferior'],
            'ultimo_valor': ts.iloc[-1],
            'n_datos': len(ts),
            'tipo_indicador': tipo_indicador
        }

    crecimiento_mediano = crecimientos.median()

    n = len(crecimientos)
    pesos = np.array([RECENT_WEIGHT ** (n - i - 1) for i in range(n)])
    pesos = pesos / pesos.sum()
    crecimiento_ponderado = np.average(crecimientos, weights=pesos)

    mad = np.median(np.abs(crecimientos - crecimiento_mediano))
    volatilidad = mad * 1.4826
    volatilidad = np.clip(volatilidad, limites['min_confidence'], limites['volatilidad_max'])

    if crecimiento_ponderado > 0.02:
        tendencia = 'creciente'
        limite_superior = min(crecimiento_ponderado * 1.5, limites['limite_superior'])
        limite_inferior = max(crecimiento_ponderado * 0.3, limites['limite_inferior'])
    elif crecimiento_ponderado < -0.02:
        tendencia = 'decreciente'
        limite_superior = max(crecimiento_ponderado * 0.3, limites['limite_superior'] * 0.5)
        limite_inferior = max(crecimiento_ponderado * 1.5, limites['limite_inferior'])
    else:
        tendencia = 'estable'
        limite_superior = limites['limite_superior']
        limite_inferior = limites['limite_inferior']

    return {
        'crecimiento_mediano': crecimiento_mediano,
        'crecimiento_promedio_ponderado': crecimiento_ponderado,
        'volatilidad': volatilidad,
        'tendencia': tendencia,
        'limite_superior': limite_superior,
        'limite_inferior': limite_inferior,
        'ultimo_valor': ts.iloc[-1],
        'n_datos': len(ts_clean),
        'tipo_indicador': tipo_indicador
    }

def validar_proyeccion_realista(valor_proyectado, stats, periodo):
    """Valida que la proyección sea realista según el tipo de indicador"""
    ultimo_valor = stats['ultimo_valor']
    tipo_indicador = stats['tipo_indicador']
    limites = obtener_limites(tipo_indicador)

    # VALIDACIÓN CRÍTICA: Si valores entre 0-100, forzar límites porcentuales
    if ultimo_valor >= 0 and ultimo_valor <= 100:
        if valor_proyectado > 100:
            valor_proyectado = 100
        elif valor_proyectado < 0:
            valor_proyectado = 0
        if ultimo_valor >= 95 and valor_proyectado > 100:
            valor_proyectado = 100

    if tipo_indicador == '%':
        valor_proyectado = np.clip(valor_proyectado, limites['valor_min'], limites['valor_max'])
        if ultimo_valor >= 95:
            valor_proyectado = min(valor_proyectado, 100)
        elif ultimo_valor <= 5:
            valor_proyectado = max(valor_proyectado, 0)

    if ultimo_valor <= 0 or valor_proyectado <= 0:
        return max(limites['valor_min'], valor_proyectado) if limites['valor_min'] is not None else max(0, valor_proyectado)

    try:
        crecimiento_total = (valor_proyectado / ultimo_valor) - 1
        crecimiento_anual = crecimiento_total / max(periodo, 1)
    except:
        return ultimo_valor

    factor_horizonte = 1 + (periodo * 0.03) if tipo_indicador == '%' else 1 + (periodo * 0.05)
    limite_sup_ajustado = stats['limite_superior'] / factor_horizonte
    limite_inf_ajustado = stats['limite_inferior'] / factor_horizonte

    if crecimiento_anual > limite_sup_ajustado:
        valor_corregido = ultimo_valor * (1 + limite_sup_ajustado) ** periodo
        if ultimo_valor <= 100:
            valor_corregido = min(valor_corregido, 100)
        return valor_corregido
    elif crecimiento_anual < limite_inf_ajustado:
        valor_corregido = ultimo_valor * (1 + limite_inf_ajustado) ** periodo
        return max(limites['valor_min'] if limites['valor_min'] is not None else 0, valor_corregido)

    if limites['valor_max'] is not None:
        valor_proyectado = min(valor_proyectado, limites['valor_max'])

    if ultimo_valor <= 100 and valor_proyectado > 100:
        valor_proyectado = 100

    return valor_proyectado

def calcular_intervalos_adaptativos(base, stats, periodo):
    """Calcula intervalos de confianza adaptativos según tipo de indicador"""
    tipo_indicador = stats['tipo_indicador']
    ultimo_valor = stats['ultimo_valor']
    limites = obtener_limites(tipo_indicador)

    es_porcentual = (tipo_indicador == '%') or (ultimo_valor >= 0 and ultimo_valor <= 100 and base <= 110)

    if es_porcentual:
        limites = obtener_limites('%')

    vol_base = min(stats['volatilidad'], limites['volatilidad_max'])
    factor_crecimiento = 0.003 if es_porcentual else 0.005
    factor_horizonte = 1 + (periodo * factor_crecimiento)
    volatilidad_ajustada = vol_base * factor_horizonte
    volatilidad_ajustada = np.clip(volatilidad_ajustada, limites['min_confidence'], limites['max_confidence'])

    optimista = base * (1 + volatilidad_ajustada)
    pesimista = base * (1 - volatilidad_ajustada)

    if es_porcentual:
        optimista = min(optimista, 100)
        pesimista = max(pesimista, 0)
        if base >= 95:
            optimista = min(optimista, 100)
            pesimista = max(base * 0.97, 0)
        elif base <= 5:
            pesimista = max(pesimista, 0)
            optimista = min(base * 1.03, 100)

    pesimista = max(limites['valor_min'] if limites['valor_min'] is not None else 0, pesimista)

    if ultimo_valor <= 100 and optimista > 100:
        optimista = 100

    return pesimista, optimista

def calcular_pesos_ensemble(predicciones, stats):
    """Calcula pesos para el ensemble"""
    if len(predicciones) == 0:
        return {}

    ultimo_valor = stats['ultimo_valor']
    crec_esperado = stats['crecimiento_promedio_ponderado']

    pesos = {}
    for modelo, valor in predicciones.items():
        if ultimo_valor > 0:
            crec_implicito = (valor / ultimo_valor) - 1
            diferencia = abs(crec_implicito - crec_esperado)
            pesos[modelo] = 1 / (1 + diferencia * 10)
        else:
            pesos[modelo] = 1

    suma_pesos = sum(pesos.values())
    if suma_pesos > 0:
        pesos = {k: v/suma_pesos for k, v in pesos.items()}

    return pesos

def crear_features_ml_optimizado(ts, n_lags=3):
    """Crear features para ML"""
    if len(ts) < n_lags + 3:
        return None, None

    features = []
    target = []

    for i in range(n_lags, len(ts)):
        try:
            lags = [ts.iloc[i-j] for j in range(1, n_lags+1)]
            ma_3 = ts.iloc[max(0, i-3):i].mean()
            ma_6 = ts.iloc[max(0, i-6):i].mean() if i >= 6 else ma_3
            vol = ts.iloc[max(0, i-3):i].std() if i >= 3 else 0
            pct_change = (ts.iloc[i-1] - ts.iloc[i-2]) / ts.iloc[i-2] if ts.iloc[i-2] != 0 else 0

            if i >= 3:
                accel = ((ts.iloc[i-1] - ts.iloc[i-2]) - (ts.iloc[i-2] - ts.iloc[i-3])) / ts.iloc[i-2] if ts.iloc[i-2] != 0 else 0
            else:
                accel = 0

            if i >= 3:
                x_trend = np.arange(3)
                y_trend = ts.iloc[i-3:i].values
                if len(y_trend) == 3:
                    slope = np.polyfit(x_trend, y_trend, 1)[0]
                else:
                    slope = 0
            else:
                slope = 0

            feature_row = lags + [ma_3, ma_6, vol, pct_change, accel, slope]
            features.append(feature_row)
            target.append(ts.iloc[i])
        except:
            continue

    if len(features) == 0:
        return None, None

    return np.array(features), np.array(target)

# ============================
# FUNCIÓN PRINCIPAL
# ============================
def proyectar_optimizado(subdf, horizonte=10):
    """Sistema de proyecciones optimizado"""
    resultados = []
    id_ind = subdf["Id"].iloc[0]
    indicador = subdf["Indicador"].iloc[0]
    periodicidad = subdf["Periodicidad"].iloc[0]
    meta = subdf["Meta"].iloc[0] if "Meta" in subdf.columns else "DEFAULT"

    if len(subdf) < MIN_DATA_POINTS:
        print(f"⚠️  {indicador}: Datos insuficientes ({len(subdf)} < {MIN_DATA_POINTS})")
        return resultados

    ts = subdf.set_index("Fecha")["Ejecución"].astype(float)
    ts = ts.replace(0, np.nan).dropna()

    if len(ts) < MIN_DATA_POINTS:
        print(f"⚠️  {indicador}: Datos válidos insuficientes")
        return resultados

    tipo_indicador = detectar_tipo_indicador(meta, ts)
    limites = obtener_limites(tipo_indicador)

    ts_values, outliers = detectar_outliers(ts.values)
    stats = calcular_estadisticas_robustas(ts, outliers, tipo_indicador)

    if periodicidad == "Semestral":
        freq = "6MS"
    else:
        freq = "AS"

    try:
        fechas_futuras = pd.date_range(
            start=ts.index[-1] + pd.DateOffset(months=6 if periodicidad=="Semestral" else 12),
            periods=horizonte,
            freq=freq
        )
    except:
        print(f"⚠️  {indicador}: Error generando fechas")
        return resultados

    print(f"\n📊 {indicador}")
    print(f"   📌 Tipo: {tipo_indicador} | Meta: {meta}")
    print(f"   Último: {stats['ultimo_valor']:.2f} | Tendencia: {stats['tendencia']}")
    print(f"   Crec.Ponderado: {stats['crecimiento_promedio_ponderado']*100:.2f}% | Vol: {stats['volatilidad']*100:.1f}%")
    print(f"   Límites: [{stats['limite_inferior']*100:.1f}%, {stats['limite_superior']*100:.1f}%]")
    if tipo_indicador == '%':
        print(f"   ⚠️  INDICADOR PORCENTUAL - Validación 0-100% activa")
    if outliers:
        print(f"   ⚠️  Outliers detectados: {len(outliers)}")

    predicciones_por_periodo = {i: {} for i in range(horizonte)}

    # ARIMA
    try:
        modelo = ARIMA(ts, order=(1,1,1))
        fit = modelo.fit()
        pred = fit.get_forecast(steps=horizonte)
        forecast = pred.predicted_mean

        for i, fecha in enumerate(fechas_futuras):
            valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
            predicciones_por_periodo[i]['ARIMA'] = valor

            pes_arima, opt_arima = calcular_intervalos_adaptativos(valor, stats, i+1)
            pes_arima = validar_proyeccion_realista(pes_arima, stats, i+1)
            opt_arima = validar_proyeccion_realista(opt_arima, stats, i+1)

            resultados.append([
                id_ind, indicador, periodicidad, tipo_indicador, fecha, "ARIMA",
                round(valor, 2), round(pes_arima, 2), round(opt_arima, 2)
            ])

        print(f"   ✅ ARIMA")
    except Exception as e:
        print(f"   ❌ ARIMA: {str(e)[:30]}")

    # ETS
    try:
        modelo = ETSModel(ts, error='add', trend='add', seasonal=None, damped_trend=True)
        fit = modelo.fit(maxiter=1000, disp=False)
        pred = fit.get_forecast(horizonte)
        forecast = pred.predicted_mean

        for i, fecha in enumerate(fechas_futuras):
            valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
            predicciones_por_periodo[i]['ETS'] = valor

            pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
            pes = validar_proyeccion_realista(pes, stats, i+1)
            opt = validar_proyeccion_realista(opt, stats, i+1)

            resultados.append([
                id_ind, indicador, periodicidad, tipo_indicador, fecha, "ETS",
                round(valor, 2), round(pes, 2), round(opt, 2)
            ])

        print(f"   ✅ ETS")
    except Exception as e:
        print(f"   ❌ ETS: {str(e)[:30]}")

    # Holt-Winters
    try:
        if len(ts) >= 8:
            modelo = ExponentialSmoothing(ts, trend='add', seasonal=None, damped_trend=True)
            fit = modelo.fit()
            forecast = fit.forecast(horizonte)

            for i, fecha in enumerate(fechas_futuras):
                valor = validar_proyeccion_realista(forecast.iloc[i], stats, i+1)
                predicciones_por_periodo[i]['Holt_Winters'] = valor

                pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
                pes = validar_proyeccion_realista(pes, stats, i+1)
                opt = validar_proyeccion_realista(opt, stats, i+1)

                resultados.append([
                    id_ind, indicador, periodicidad, tipo_indicador, fecha, "Holt_Winters",
                    round(valor, 2), round(pes, 2), round(opt, 2)
                ])

            print(f"   ✅ Holt-Winters")
    except Exception as e:
        print(f"   ❌ Holt-Winters: {str(e)[:30]}")

    # Random Forest
    try:
        X, y = crear_features_ml_optimizado(ts, n_lags=3)

        if X is not None and len(X) >= 5:
            modelo = RandomForestRegressor(
                n_estimators=100,
                max_depth=5,
                min_samples_split=3,
                min_samples_leaf=2,
                random_state=42
            )
            modelo.fit(X, y)

            ts_extended = ts.copy()

            for periodo in range(horizonte):
                if len(ts_extended) >= 6:
                    lags = [ts_extended.iloc[-1], ts_extended.iloc[-2], ts_extended.iloc[-3]]
                    ma_3 = ts_extended.iloc[-3:].mean()
                    ma_6 = ts_extended.iloc[-6:].mean() if len(ts_extended) >= 6 else ma_3
                    vol = ts_extended.iloc[-3:].std()
                    pct = (ts_extended.iloc[-1] - ts_extended.iloc[-2])/ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0

                    if len(ts_extended) >= 3:
                        accel = ((ts_extended.iloc[-1] - ts_extended.iloc[-2]) - (ts_extended.iloc[-2] - ts_extended.iloc[-3])) / ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0
                        x_trend = np.arange(3)
                        y_trend = ts_extended.iloc[-3:].values
                        slope = np.polyfit(x_trend, y_trend, 1)[0]
                    else:
                        accel = 0
                        slope = 0

                    features = [lags + [ma_3, ma_6, vol, pct, accel, slope]]
                    pred = modelo.predict(features)[0]
                    pred = validar_proyeccion_realista(pred, stats, periodo+1)

                    predicciones_por_periodo[periodo]['Random_Forest'] = pred
                    ts_extended = pd.concat([ts_extended, pd.Series([pred])])

            for i, fecha in enumerate(fechas_futuras):
                if i in predicciones_por_periodo and 'Random_Forest' in predicciones_por_periodo[i]:
                    valor = predicciones_por_periodo[i]['Random_Forest']

                    pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
                    pes = validar_proyeccion_realista(pes, stats, i+1)
                    opt = validar_proyeccion_realista(opt, stats, i+1)

                    resultados.append([
                        id_ind, indicador, periodicidad, tipo_indicador, fecha, "Random_Forest",
                        round(valor, 2), round(pes, 2), round(opt, 2)
                    ])

            print(f"   ✅ Random Forest")
    except Exception as e:
        print(f"   ❌ Random Forest: {str(e)[:30]}")

    # Prophet
    try:
        prophet_df = subdf.rename(columns={"Fecha":"ds","Ejecución":"y"})[["ds","y"]].dropna()

        if len(prophet_df) >= MIN_DATA_POINTS:
            m = Prophet(
                growth='linear',
                changepoint_prior_scale=0.01,
                yearly_seasonality=False,
                weekly_seasonality=False,
                daily_seasonality=False,
                interval_width=0.8
            )
            m.fit(prophet_df)

            future = pd.DataFrame({'ds': fechas_futuras})
            forecast = m.predict(future)

            for i, row in forecast.iterrows():
                valor = validar_proyeccion_realista(row["yhat"], stats, i+1)
                predicciones_por_periodo[i]['Prophet'] = valor

                pes_prophet, opt_prophet = calcular_intervalos_adaptativos(valor, stats, i+1)
                pes_prophet = validar_proyeccion_realista(pes_prophet, stats, i+1)
                opt_prophet = validar_proyeccion_realista(opt_prophet, stats, i+1)

                resultados.append([
                    id_ind, indicador, periodicidad, tipo_indicador, fechas_futuras[i], "Prophet",
                    round(valor, 2), round(pes_prophet, 2), round(opt_prophet, 2)
                ])

            print(f"   ✅ Prophet")
    except Exception as e:
        print(f"   ❌ Prophet: {str(e)[:30]}")

    # Tendencia Histórica
    try:
        for i in range(horizonte):
            valor = stats['ultimo_valor'] * ((1 + stats['crecimiento_promedio_ponderado']) ** (i+1))
            valor = validar_proyeccion_realista(valor, stats, i+1)
            predicciones_por_periodo[i]['Tendencia_Historica'] = valor

            pes, opt = calcular_intervalos_adaptativos(valor, stats, i+1)
            pes = validar_proyeccion_realista(pes, stats, i+1)
            opt = validar_proyeccion_realista(opt, stats, i+1)

            resultados.append([
                id_ind, indicador, periodicidad, tipo_indicador, fechas_futuras[i], "Tendencia_Historica",
                round(valor, 2), round(pes, 2), round(opt, 2)
            ])

        print(f"   ✅ Tendencia Histórica")
    except Exception as e:
        print(f"   ❌ Tendencia Histórica: {str(e)[:30]}")

    # Ensemble
    for i, fecha in enumerate(fechas_futuras):
        preds = predicciones_por_periodo[i]

        if len(preds) == 0:
            continue

        pesos = calcular_pesos_ensemble(preds, stats)

        if len(pesos) > 0:
            base_ensemble = sum(preds[m] * pesos[m] for m in preds.keys())
            base_ensemble = validar_proyeccion_realista(base_ensemble, stats, i+1)
        else:
            base_ensemble = np.mean(list(preds.values()))

        pesimista, optimista = calcular_intervalos_adaptativos(base_ensemble, stats, i+1)
        pesimista = validar_proyeccion_realista(pesimista, stats, i+1)
        optimista = validar_proyeccion_realista(optimista, stats, i+1)

        resultados.append([
            id_ind, indicador, periodicidad, tipo_indicador, fecha, "Ensemble_Ponderado",
            round(base_ensemble, 2), round(pesimista, 2), round(optimista, 2)
        ])

    print(f"   📈 Proyección final (año 1): {base_ensemble:.2f}")

    return resultados

# ============================
# EJECUCIÓN
# ============================
print("="*70)
print("🚀 SISTEMA DE PROYECCIONES OPTIMIZADO V6")
print("="*70)

all_results = []
stats_summary = []
total = df["Id"].nunique()

for idx, (id_ind, subdf) in enumerate(df.groupby("Id"), 1):
    print(f"\n[{idx}/{total}]")
    res = proyectar_optimizado(subdf, horizonte=10)
    all_results.extend(res)

    if len(res) > 0:
        ts = subdf.set_index("Fecha")["Ejecución"].astype(float).replace(0, np.nan).dropna()
        if len(ts) >= MIN_DATA_POINTS:
            meta = subdf["Meta"].iloc[0]
            tipo_indicador = detectar_tipo_indicador(meta, ts)
            _, outliers = detectar_outliers(ts.values)
            stats = calcular_estadisticas_robustas(ts, outliers, tipo_indicador)

            stats_summary.append({
                'Id': id_ind,
                'Indicador': subdf["Indicador"].iloc[0],
                'Tipo': tipo_indicador,
                'Meta': meta,
                'Tendencia': stats['tendencia'],
                'Ultimo_Valor': stats['ultimo_valor'],
                'Crecimiento_Mediano_%': stats['crecimiento_mediano'] * 100,
                'Crecimiento_Ponderado_%': stats['crecimiento_promedio_ponderado'] * 100,
                'Volatilidad_%': stats['volatilidad'] * 100,
                'Limite_Superior_%': stats['limite_superior'] * 100,
                'Limite_Inferior_%': stats['limite_inferior'] * 100,
                'Outliers_Detectados': len(outliers),
                'Puntos_Datos': len(ts)
            })

# ============================
# GUARDAR RESULTADOS
# ============================
print("\n" + "="*70)
print("💾 GUARDANDO RESULTADOS")
print("="*70)

if all_results:
    df_proy = pd.DataFrame(
        all_results,
        columns=["Id", "Indicador", "Periodicidad", "Tipo_Indicador", "Fecha_Proyeccion",
                 "Modelo", "Escenario_Base", "Escenario_Pesimista", "Escenario_Optimista"]
    )

    df_stats = pd.DataFrame(stats_summary)

    print(f"\n✅ RESUMEN:")
    print(f"   📈 Proyecciones totales: {len(df_proy):,}")
    print(f"   🏢 Indicadores: {df_proy['Indicador'].nunique()}")
    print(f"   🤖 Modelos: {df_proy['Modelo'].nunique()}")

    print(f"\n📊 DISTRIBUCIÓN POR MODELO:")
    for modelo, count in df_proy['Modelo'].value_counts().items():
        print(f"   {modelo:.<35} {count:>5,}")

    print(f"\n📋 DISTRIBUCIÓN POR TIPO DE INDICADOR:")
    for tipo, count in df_proy['Tipo_Indicador'].value_counts().items():
        limites = obtener_limites(tipo)
        print(f"   {tipo:.<15} {count:>5,} proyecciones | Límites: ±{limites['limite_superior']*100:.0f}%")

    # Validación de indicadores porcentuales
    df_porcentuales = df_proy[df_proy['Tipo_Indicador'] == '%'].copy()
    if len(df_porcentuales) > 0:
        valores_invalidos = df_porcentuales[
            (df_porcentuales['Escenario_Base'] > 100) |
            (df_porcentuales['Escenario_Base'] < 0) |
            (df_porcentuales['Escenario_Optimista'] > 100) |
            (df_porcentuales['Escenario_Pesimista'] < 0)
        ]

        if len(valores_invalidos) > 0:
            print(f"\n⚠️  ADVERTENCIA: {len(valores_invalidos)} proyecciones porcentuales fuera de rango 0-100%")
            print(f"   Se recomienda revisar: {valores_invalidos['Indicador'].unique()}")
        else:
            print(f"\n✅ Todos los indicadores porcentuales dentro del rango 0-100%")

    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        df_proy.to_excel(writer, sheet_name='Proyecciones', index=False)
        df_stats.to_excel(writer, sheet_name='Estadisticas_Robustas', index=False)

        df_ensemble = df_proy[df_proy['Modelo'] == 'Ensemble_Ponderado'].copy()
        df_ensemble.to_excel(writer, sheet_name='Ensemble_Recomendado', index=False)

        pivot = df_proy.pivot_table(
            index=['Indicador', 'Fecha_Proyeccion'],
            columns='Modelo',
            values='Escenario_Base'
        ).round(2)
        pivot.to_excel(writer, sheet_name='Comparacion_Modelos')

        resumen_tipo = df_proy.groupby(['Tipo_Indicador', 'Modelo']).agg({
            'Escenario_Base': ['mean', 'min', 'max', 'std'],
            'Id': 'count'
        }).round(2)
        resumen_tipo.to_excel(writer, sheet_name='Analisis_Por_Tipo')

        if len(df_porcentuales) > 0:
            df_porcentuales_ensemble = df_porcentuales[
                df_porcentuales['Modelo'] == 'Ensemble_Ponderado'
            ].copy()
            df_porcentuales_ensemble.to_excel(writer, sheet_name='Validacion_Porcentuales', index=False)

    print(f"\n✅ Archivo guardado: {OUTPUT_FILE}")
    print(f"   📋 Hojas creadas:")
    print(f"      1. Proyecciones (todas)")
    print(f"      2. Estadisticas_Robustas")
    print(f"      3. Ensemble_Recomendado")
    print(f"      4. Comparacion_Modelos")
    print(f"      5. Analisis_Por_Tipo")
    print(f"      6. Validacion_Porcentuales")

    print(f"\n📊 EJEMPLOS DE PROYECCIONES (Año 1 - Ensemble):")
    ejemplos = df_ensemble[df_ensemble['Fecha_Proyeccion'] == df_ensemble['Fecha_Proyeccion'].min()]
    for _, row in ejemplos.head(5).iterrows():
        tipo_sym = "📍" if row['Tipo_Indicador'] == '%' else "📊"
        print(f"   {tipo_sym} {row['Indicador'][:40]:.<45} {row['Escenario_Base']:>8.2f} [{row['Escenario_Pesimista']:.2f} - {row['Escenario_Optimista']:.2f}]")

else:
    print("\n❌ No se generaron proyecciones")

print("\n" + "="*70)
print("✅ PROCESO COMPLETADO")
print("="*70)
print("\n💡 NOTAS IMPORTANTES:")
print("   • Los indicadores porcentuales están limitados al rango 0-100%")
print("   • Los crecimientos máximos son del 5% anual para porcentuales")
print("   • Los intervalos de confianza son más estrechos para porcentuales (1-3%)")
print("   • Revise la hoja 'Validacion_Porcentuales' para verificar coherencia")
print("   • Use 'Ensemble_Recomendado' como proyección oficial")
print("="*70)


📋 MUESTRA DE VALORES EN COLUMNA META:
                 Indicador          Meta
0          Total Población           NaN
12         Total Población  49004.000000
13         Total Población  50430.000000
15         Total Población  52177.000000
16         Total Población  51687.000000
18         Total Población  52403.186119
19         Total Población  51116.334115
21         Total Población  53634.000000
24  Estudiantes Presencial           NaN
36  Estudiantes Presencial   7541.000000

📊 Tipos de Meta detectados: {0.0: 21, 80.0: 17, 5.0: 14, 97.0: 11, 85.0: 9, 81.0: 7, 88.0: 7, 3.0: 7, 1.0: 6, 30.0: 6, 73.0: 6, 25.0: 6, 2.0: 5, 6.0: 4, 8.0: 4, 1.8: 4, 50.0: 4, 75.0: 3, 8.3: 3, 88.5: 3, 87.0: 3, 93.0: 3, 90.0: 3, 92.0: 3, 9557.195010681125: 3, 76.0: 3, 1.6: 3, 1.5: 3, 82.0: 3, 7.0: 3, 4.5: 3, 10.0: 2, 4.0: 2, 50430.0: 2, 6.5: 2, 8.4: 2, 8.8: 2, 2797.663886: 2, 66.0: 2, 68.7: 2, 23623.074007: 2, 16064.0: 2, 238770.966572458: 2, 14097.721: 2, 26073.47270271854: 2, 27010.10828083451: 2, 18

#Modelo 8

In [ ]:
# ============================
# SISTEMA DE PROYECCIONES V13 COMPLETO
# CON MODELOS PARA DATOS ESCASOS Y EVALUACIÓN INTEGRADA
# ============================

import pandas as pd
import numpy as np
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from scipy import stats as scipy_stats
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ============================
# CONFIGURACIÓN GLOBAL
# ============================

CONFIG = {
    'min_data_points': 2,
    'min_data_full_models': 6,
    'outlier_threshold': 3.5,
    'recent_weight': 0.8,
    'confidence': {
        'optimista_pct': 0.05,
        'pesimista_pct': 0.05,
        'ajuste_horizonte': 0.002
    },
    'limites': {
        '%': {
            'crecimiento_anual_max': 0.15,
            'crecimiento_anual_min': -0.15,
            'crecimiento_periodo_max': 0.20,
            'crecimiento_periodo_min': -0.20,
            'valor_min': 0,
            'valor_max': 100,
            'volatilidad_max': 0.05,
            'min_confidence': 0.05,
            'max_confidence': 0.05
        },
        'ENT': {
            'crecimiento_anual_max': 0.25,
            'crecimiento_anual_min': -0.20,
            'crecimiento_periodo_max': 0.30,
            'crecimiento_periodo_min': -0.25,
            'valor_min': 0,
            'valor_max': None,
            'volatilidad_max': 0.15,
            'min_confidence': 0.05,
            'max_confidence': 0.15
        },
        'DEC': {
            'crecimiento_anual_max': 0.20,
            'crecimiento_anual_min': -0.20,
            'crecimiento_periodo_max': 0.25,
            'crecimiento_periodo_min': -0.25,
            'valor_min': 0,
            'valor_max': None,
            'volatilidad_max': 0.12,
            'min_confidence': 0.05,
            'max_confidence': 0.12
        },
        '$': {
            'crecimiento_anual_max': 0.30,
            'crecimiento_anual_min': -0.25,
            'crecimiento_periodo_max': 0.35,
            'crecimiento_periodo_min': -0.30,
            'valor_min': 0,
            'valor_max': None,
            'volatilidad_max': 0.15,
            'min_confidence': 0.05,
            'max_confidence': 0.15
        },
        'DEFAULT': {
            'crecimiento_anual_max': 0.20,
            'crecimiento_anual_min': -0.20,
            'crecimiento_periodo_max': 0.25,
            'crecimiento_periodo_min': -0.25,
            'valor_min': 0,
            'valor_max': None,
            'volatilidad_max': 0.12,
            'min_confidence': 0.05,
            'max_confidence': 0.12
        }
    },
    'jerarquias': [
        {
            'padre': 'Total Población',
            'hijos': ['Estudiantes Presencial', 'Estudiantes Virtual'],
            'nombre': 'Jerarquia_Modalidad'
        },
        {
            'padre': 'Total Población',
            'hijos': ['Estudiantes Pregrado', 'Estudiantes Posgrado'],
            'nombre': 'Jerarquia_Nivel'
        }
    ]
}

# ============================
# CLASE PRINCIPAL DEL SISTEMA
# ============================

class SistemaProyecciones:
    """Sistema completo de proyecciones con todos los modelos y evaluación"""

    def __init__(self, config=CONFIG):
        self.config = config
        self.resultados = []
        self.estadisticas = []
        self.metricas_confiabilidad = []
        self.alertas = []

    # ============================
    # UTILIDADES
    # ============================

    def detectar_tipo_indicador(self, meta_value, valores_historicos=None):
        """Detecta el tipo de indicador"""
        if pd.isna(meta_value):
            tipo = 'DEFAULT'
        else:
            meta_str = str(meta_value).upper().strip()
            if '%' in meta_str or 'PORCENTAJE' in meta_str:
                tipo = '%'
            elif 'ENT' in meta_str:
                tipo = 'ENT'
            elif 'DEC' in meta_str:
                tipo = 'DEC'
            elif '$' in meta_str or 'PESO' in meta_str:
                tipo = '$'
            else:
                tipo = 'DEFAULT'

        if valores_historicos is not None and len(valores_historicos) > 0:
            valores_clean = valores_historicos.dropna()
            if len(valores_clean) > 0:
                if valores_clean.min() >= 0 and valores_clean.max() <= 100 and tipo == '%':
                    return '%'

        return tipo

    def detectar_outliers(self, ts):
        """Detecta outliers usando Z-score"""
        if len(ts) < 4:
            return ts, []

        pct_changes = pd.Series(ts).pct_change().dropna()
        if len(pct_changes) < 3:
            return ts, []

        z_scores = np.abs(scipy_stats.zscore(pct_changes))
        outliers = np.where(z_scores > self.config['outlier_threshold'])[0]
        return ts, outliers.tolist()

    def calcular_estadisticas(self, ts, outliers, tipo_indicador):
        """Calcula estadísticas robustas de la serie"""
        limites = self.config['limites'].get(tipo_indicador, self.config['limites']['DEFAULT'])

        # Limpiar outliers si son muchos
        if len(outliers) > len(ts) * 0.3:
            ts_clean = ts.drop(ts.index[outliers])
        else:
            ts_clean = ts

        if len(ts_clean) < 2:
            ts_clean = ts

        crecimientos = ts.pct_change().dropna()

        if len(crecimientos) == 0:
            return {
                'crecimiento_mediano': 0,
                'crecimiento_promedio_ponderado': 0,
                'volatilidad': limites['min_confidence'],
                'tendencia': 'estable',
                'ultimo_valor': ts.iloc[-1],
                'n_datos': len(ts),
                'tipo_indicador': tipo_indicador,
                'limites': limites
            }

        # Crecimiento con pesos exponenciales
        n = len(crecimientos)
        pesos = np.array([self.config['recent_weight'] ** (n - i - 1) for i in range(n)])
        pesos = pesos / pesos.sum()
        crecimiento_ponderado = np.average(crecimientos, weights=pesos)

        # Aplicar límites
        crecimiento_ponderado = np.clip(
            crecimiento_ponderado,
            limites['crecimiento_anual_min'],
            limites['crecimiento_anual_max']
        )

        # Ajustar por últimos crecimientos
        ultimos = crecimientos.tail(min(3, len(crecimientos)))
        if len(ultimos) >= 2:
            if all(ultimos > 0):
                crecimiento_ponderado = max(crecimiento_ponderado, ultimos.mean())
            elif all(ultimos < 0):
                crecimiento_ponderado = min(crecimiento_ponderado, ultimos.mean())

        # Volatilidad usando MAD
        mad = np.median(np.abs(crecimientos - crecimientos.median()))
        volatilidad = np.clip(mad * 1.4826, limites['min_confidence'], limites['volatilidad_max'])

        # Tendencia
        if crecimiento_ponderado > 0.03:
            tendencia = 'creciente'
        elif crecimiento_ponderado < -0.03:
            tendencia = 'decreciente'
        else:
            tendencia = 'estable'

        return {
            'crecimiento_mediano': crecimientos.median(),
            'crecimiento_promedio_ponderado': crecimiento_ponderado,
            'volatilidad': volatilidad,
            'tendencia': tendencia,
            'ultimo_valor': ts.iloc[-1],
            'n_datos': len(ts_clean),
            'tipo_indicador': tipo_indicador,
            'limites': limites
        }

    def validar_proyeccion(self, valor_proyectado, valor_anterior, stats, periodo, periodicidad):
        """Validación anti-extrapolación estricta"""
        limites = stats['limites']
        tipo = stats['tipo_indicador']

        # Validar rango para porcentajes
        if tipo == '%':
            valor_proyectado = np.clip(valor_proyectado, 0, 100)

        # Evitar negativos
        if valor_proyectado < 0:
            valor_proyectado = max(0, valor_anterior * 0.9)

        if valor_anterior <= 0:
            return max(0, valor_proyectado)

        # Validar crecimiento por periodo
        crec_periodo = (valor_proyectado / valor_anterior) - 1

        if crec_periodo > limites['crecimiento_periodo_max']:
            valor_proyectado = valor_anterior * (1 + limites['crecimiento_periodo_max'])
        elif crec_periodo < limites['crecimiento_periodo_min']:
            valor_proyectado = valor_anterior * (1 + limites['crecimiento_periodo_min'])

        # Validar crecimiento anualizado
        ultimo_hist = stats['ultimo_valor']
        periodos_anuales = periodo * 0.5 if periodicidad == "Semestral" else periodo

        try:
            crec_anual = ((valor_proyectado / ultimo_hist) ** (1/periodos_anuales)) - 1
        except:
            return valor_anterior

        if crec_anual > limites['crecimiento_anual_max']:
            valor_proyectado = ultimo_hist * ((1 + limites['crecimiento_anual_max']) ** periodos_anuales)
        elif crec_anual < limites['crecimiento_anual_min']:
            valor_proyectado = max(0, ultimo_hist * ((1 + limites['crecimiento_anual_min']) ** periodos_anuales))

        # Validación final
        if tipo == '%':
            valor_proyectado = np.clip(valor_proyectado, 0, 100)
        else:
            valor_proyectado = max(0, valor_proyectado)

        return valor_proyectado

    def calcular_intervalos(self, base, stats, periodo):
        """Calcula intervalos de confianza fijos ±5%"""
        conf = self.config['confidence']
        limites = stats['limites']
        tipo = stats['tipo_indicador']

        # Porcentajes fijos + ajuste por horizonte
        pct_opt = conf['optimista_pct'] + (conf['ajuste_horizonte'] * periodo)
        pct_pes = conf['pesimista_pct'] + (conf['ajuste_horizonte'] * periodo)

        # Respetar límites máximos
        pct_opt = min(pct_opt, limites['max_confidence'])
        pct_pes = min(pct_pes, limites['max_confidence'])

        optimista = base * (1 + pct_opt)
        pesimista = base * (1 - pct_pes)

        # Validar por tipo
        if tipo == '%':
            optimista = min(optimista, 100)
            pesimista = max(pesimista, 0)
        else:
            pesimista = max(0, pesimista)

        return pesimista, optimista

    def calcular_pesos_ensemble(self, predicciones, stats):
        """Calcula pesos para ensemble basado en tendencia"""
        if len(predicciones) == 0:
            return {}

        ultimo_valor = stats['ultimo_valor']
        crec_esperado = stats['crecimiento_promedio_ponderado']
        tendencia = stats['tendencia']

        pesos = {}
        for modelo, valor in predicciones.items():
            if ultimo_valor > 0:
                crec_implicito = (valor / ultimo_valor) - 1
                diferencia = abs(crec_implicito - crec_esperado)

                # Penalizar modelos que van contra la tendencia
                if tendencia == 'creciente' and crec_implicito < 0:
                    diferencia += 0.5
                elif tendencia == 'decreciente' and crec_implicito > 0:
                    diferencia += 0.5

                pesos[modelo] = 1 / (1 + diferencia * 10)
            else:
                pesos[modelo] = 1

        # Normalizar
        suma = sum(pesos.values())
        if suma > 0:
            pesos = {k: v/suma for k, v in pesos.items()}

        return pesos

    # ============================
    # MODELOS PARA DATOS ESCASOS (2-5)
    # ============================

    def modelo_regresion_lineal(self, ts, stats, horizonte, periodicidad):
        """Regresión lineal simple para datos escasos"""
        try:
            X = np.arange(len(ts)).reshape(-1, 1)
            y = ts.values

            modelo = LinearRegression()
            modelo.fit(X, y)

            X_futuro = np.arange(len(ts), len(ts) + horizonte).reshape(-1, 1)
            predicciones = modelo.predict(X_futuro)

            # Validar cada proyección
            proyecciones = []
            valor_actual = stats['ultimo_valor']

            for i, pred in enumerate(predicciones):
                valor = self.validar_proyeccion(pred, valor_actual, stats, i+1, periodicidad)
                proyecciones.append(valor)
                valor_actual = valor

            return proyecciones
        except:
            return None

    def modelo_promedio_movil(self, ts, stats, horizonte, periodicidad):
        """Promedio móvil adaptativo para datos escasos"""
        try:
            # Calcular crecimiento promedio ponderado
            if len(ts) >= 2:
                crecimientos = ts.pct_change().dropna()
                if len(crecimientos) > 0:
                    n = len(crecimientos)
                    pesos = np.array([0.7 ** (n - i - 1) for i in range(n)])
                    pesos = pesos / pesos.sum()
                    crecimiento = np.average(crecimientos, weights=pesos)
                else:
                    crecimiento = 0
            else:
                crecimiento = 0

            # Limitar crecimiento
            limites = stats['limites']
            crecimiento = np.clip(crecimiento, limites['crecimiento_anual_min'], limites['crecimiento_anual_max'])

            # Proyectar
            proyecciones = []
            valor_actual = stats['ultimo_valor']

            for i in range(horizonte):
                valor = valor_actual * (1 + crecimiento)
                valor = self.validar_proyeccion(valor, valor_actual, stats, i+1, periodicidad)
                proyecciones.append(valor)
                valor_actual = valor

            return proyecciones
        except:
            return None

    # ============================
    # MODELOS COMPLETOS (6+)
    # ============================

    def modelo_arima(self, ts, stats, horizonte, periodicidad):
        """ARIMA con selección automática de orden"""
        try:
            # Seleccionar orden óptimo
            try:
                adf_result = adfuller(ts)
                d = 1 if adf_result[1] > 0.05 else 0
            except:
                d = 1

            mejor_aic = float('inf')
            mejor_orden = (1, 1, 1)

            for p in range(4):
                for q in range(4):
                    if p == 0 and q == 0:
                        continue
                    try:
                        modelo = ARIMA(ts, order=(p, d, q))
                        fit = modelo.fit()
                        if fit.aic < mejor_aic:
                            mejor_aic = fit.aic
                            mejor_orden = (p, d, q)
                    except:
                        continue

            # Ajustar mejor modelo
            modelo = ARIMA(ts, order=mejor_orden)
            fit = modelo.fit()
            forecast = fit.get_forecast(steps=horizonte).predicted_mean.values

            # Validar
            return self.validar_proyecciones_secuencial(forecast, stats, horizonte, periodicidad)
        except:
            return None

    def modelo_ets(self, ts, stats, horizonte, periodicidad):
        """ETS (Error-Trend-Seasonal)"""
        try:
            modelo = ETSModel(ts, error='add', trend='add', seasonal=None, damped_trend=False)
            fit = modelo.fit(maxiter=1000, disp=False)
            forecast = fit.get_forecast(horizonte).predicted_mean.values
            return self.validar_proyecciones_secuencial(forecast, stats, horizonte, periodicidad)
        except:
            return None

    def modelo_holt_winters(self, ts, stats, horizonte, periodicidad):
        """Holt-Winters"""
        try:
            if len(ts) < 8:
                return None
            modelo = ExponentialSmoothing(ts, trend='add', seasonal=None, damped_trend=False)
            fit = modelo.fit()
            forecast = fit.forecast(horizonte).values
            return self.validar_proyecciones_secuencial(forecast, stats, horizonte, periodicidad)
        except:
            return None

    def modelo_random_forest(self, ts, stats, horizonte, periodicidad):
        """Random Forest con features de series temporales"""
        try:
            # Crear features
            n_lags = 3
            if len(ts) < n_lags + 3:
                return None

            features = []
            target = []

            for i in range(n_lags, len(ts)):
                lags = [ts.iloc[i-j] for j in range(1, n_lags+1)]
                ma_3 = ts.iloc[max(0, i-3):i].mean()
                vol = ts.iloc[max(0, i-3):i].std() if i >= 3 else 0
                pct = (ts.iloc[i-1] - ts.iloc[i-2])/ts.iloc[i-2] if ts.iloc[i-2] != 0 else 0

                features.append(lags + [ma_3, vol, pct])
                target.append(ts.iloc[i])

            if len(features) < 5:
                return None

            X = np.array(features)
            y = np.array(target)

            # Entrenar
            modelo = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
            modelo.fit(X, y)

            # Proyectar iterativamente
            ts_extended = ts.copy()
            predicciones = []

            for _ in range(horizonte):
                lags = [ts_extended.iloc[-1], ts_extended.iloc[-2], ts_extended.iloc[-3]]
                ma_3 = ts_extended.iloc[-3:].mean()
                vol = ts_extended.iloc[-3:].std()
                pct = (ts_extended.iloc[-1] - ts_extended.iloc[-2])/ts_extended.iloc[-2] if ts_extended.iloc[-2] != 0 else 0

                X_pred = np.array([lags + [ma_3, vol, pct]])
                pred = modelo.predict(X_pred)[0]

                predicciones.append(pred)
                ts_extended = pd.concat([ts_extended, pd.Series([pred])])

            return self.validar_proyecciones_secuencial(predicciones, stats, horizonte, periodicidad)
        except:
            return None

    def modelo_prophet(self, ts, fechas, horizonte, periodicidad):
        """Prophet de Facebook"""
        try:
            df = pd.DataFrame({
                'ds': ts.index,
                'y': ts.values
            })

            modelo = Prophet(
                growth='linear',
                changepoint_prior_scale=0.05,
                yearly_seasonality=False,
                weekly_seasonality=False,
                daily_seasonality=False,
                interval_width=0.8
            )
            modelo.fit(df)

            future = pd.DataFrame({'ds': fechas})
            forecast = modelo.predict(future)

            return forecast['yhat'].values
        except:
            return None

    def modelo_tendencia_historica(self, stats, horizonte, periodicidad):
        """Tendencia histórica simple - siempre disponible"""
        proyecciones = []
        valor_actual = stats['ultimo_valor']

        for i in range(horizonte):
            periodos = (i+1) * 0.5 if periodicidad == "Semestral" else (i+1)
            valor = stats['ultimo_valor'] * ((1 + stats['crecimiento_promedio_ponderado']) ** periodos)
            valor = self.validar_proyeccion(valor, valor_actual, stats, i+1, periodicidad)
            proyecciones.append(valor)
            valor_actual = valor

        return proyecciones

    def validar_proyecciones_secuencial(self, valores_base, stats, horizonte, periodicidad):
        """Valida proyecciones secuencialmente"""
        proyecciones = []
        valor_actual = stats['ultimo_valor']

        for i in range(horizonte):
            if i < len(valores_base):
                valor = valores_base[i]
            else:
                periodos = (i+1) * 0.5 if periodicidad == "Semestral" else (i+1)
                valor = stats['ultimo_valor'] * ((1 + stats['crecimiento_promedio_ponderado']) ** periodos)

            valor = self.validar_proyeccion(valor, valor_actual, stats, i+1, periodicidad)
            proyecciones.append(valor)
            valor_actual = valor

        return proyecciones

    # ============================
    # EVALUACIÓN
    # ============================

    def calcular_metricas_confiabilidad(self, ts, predicciones_periodo, stats):
        """Calcula score de confiabilidad 0-100"""
        n_datos = len(ts)

        # Score de datos
        if n_datos <= 2:
            score_datos = 20
        elif n_datos <= 3:
            score_datos = 40
        elif n_datos <= 5:
            score_datos = 60
        elif n_datos <= 8:
            score_datos = 80
        else:
            score_datos = 100

        # Score de estabilidad
        vol = stats['volatilidad']
        if vol < 0.05:
            score_estab = 100
        elif vol < 0.10:
            score_estab = 80
        elif vol < 0.15:
            score_estab = 60
        elif vol < 0.20:
            score_estab = 40
        else:
            score_estab = 20

        # Score de consenso entre modelos
        if len(predicciones_periodo) > 1:
            valores = list(predicciones_periodo.values())
            cv = np.std(valores) / np.mean(valores) if np.mean(valores) > 0 else 1

            if cv < 0.05:
                score_cons = 100
            elif cv < 0.10:
                score_cons = 80
            elif cv < 0.15:
                score_cons = 60
            else:
                score_cons = 40
        else:
            score_cons = 50

        # Score de tendencia
        score_tend = 80 if stats['tendencia'] in ['creciente', 'decreciente'] else 60

        # Score total ponderado
        score_total = (
            score_datos * 0.35 +
            score_estab * 0.30 +
            score_cons * 0.25 +
            score_tend * 0.10
        )

        # Clasificación
        if score_total >= 80:
            clasificacion = 'ALTA'
            emoji = '🟢'
        elif score_total >= 60:
            clasificacion = 'MEDIA'
            emoji = '🟡'
        else:
            clasificacion = 'BAJA'
            emoji = '🔴'

        return {
            'Score_Total': round(score_total, 1),
            'Clasificacion': clasificacion,
            'Emoji': emoji,
            'Score_Datos': score_datos,
            'Score_Estabilidad': score_estab,
            'Score_Consenso': score_cons,
            'Score_Tendencia': score_tend,
            'N_Modelos': len(predicciones_periodo)
        }

    def validacion_cruzada(self, ts, stats, tipo_indicador, periodicidad):
        """Validación cruzada leave-one-out"""
        if len(ts) < 6:
            return None

        errores = []
        n = len(ts)
        n_val = min(3, n - 3)

        for i in range(n_val):
            train_size = n - (i + 1)
            ts_train = ts.iloc[:train_size]
            valor_real = ts.iloc[train_size]

            # Proyección simple
            crecimientos = ts_train.pct_change().dropna()
            if len(crecimientos) > 0:
                crec = crecimientos.mean()
                valor_pred = ts_train.iloc[-1] * (1 + crec)

                error_pct = abs(valor_real - valor_pred) / valor_real * 100 if valor_real != 0 else 0
                errores.append(error_pct)

        if len(errores) > 0:
            mae = np.mean(errores)

            if mae < 5:
                precision = 'Muy Alta'
            elif mae < 10:
                precision = 'Alta'
            elif mae < 15:
                precision = 'Media'
            else:
                precision = 'Baja'

            return {
                'MAE_%': round(mae, 2),
                'Precision': precision,
                'N_Validaciones': len(errores)
            }

        return None

    def generar_alertas(self, df_stats, df_proyecciones):
        """Genera alertas de calidad"""
        alertas = []

        for _, row in df_stats.iterrows():
            indicador = row['Indicador']
            n_datos = row['N_Datos']
            volatilidad = row['Volatilidad_%']
            outliers = row['Outliers']

            # Alerta datos escasos
            if n_datos <= 3:
                alertas.append({
                    'Indicador': indicador,
                    'Tipo': 'CRÍTICO',
                    'Severidad': '🔴',
                    'Mensaje': f'Solo {n_datos} datos - Proyección muy incierta',
                    'Recomendacion': 'Recolectar más datos o validar con expertos'
                })
            elif n_datos <= 5:
                alertas.append({
                    'Indicador': indicador,
                    'Tipo': 'ADVERTENCIA',
                    'Severidad': '🟡',
                    'Mensaje': f'Datos escasos ({n_datos} puntos)',
                    'Recomendacion': 'Considerar intervalos más amplios'
                })

            # Alerta volatilidad
            if volatilidad > 20:
                alertas.append({
                    'Indicador': indicador,
                    'Tipo': 'ADVERTENCIA',
                    'Severidad': '🟡',
                    'Mensaje': f'Volatilidad alta ({volatilidad:.1f}%)',
                    'Recomendacion': 'Revisar causas de variabilidad'
                })

            # Alerta outliers
            if outliers > 0 and n_datos > 0:
                pct_out = (outliers / n_datos) * 100
                if pct_out > 30:
                    alertas.append({
                        'Indicador': indicador,
                        'Tipo': 'ADVERTENCIA',
                        'Severidad': '🟡',
                        'Mensaje': f'{pct_out:.1f}% outliers',
                        'Recomendacion': 'Investigar eventos atípicos'
                    })

        return pd.DataFrame(alertas) if alertas else pd.DataFrame()

    # ============================
    # MOTOR DE PROYECCIONES
    # ============================

    def proyectar_indicador(self, subdf, horizonte=10):
        """Proyecta un indicador individual"""
        id_ind = subdf["Id"].iloc[0]
        indicador = subdf["Indicador"].iloc[0]
        periodicidad = subdf["Periodicidad"].iloc[0]
        meta = subdf["Meta"].iloc[0] if "Meta" in subdf.columns else "DEFAULT"

        # Validar datos mínimos
        if len(subdf) < self.config['min_data_points']:print(f"⚠️  {indicador}: Datos insuficientes ({len(subdf)} < {self.config['min_data_points']})")
            return []

        # Preparar serie temporal
        ts = subdf.set_index("Fecha")["Ejecución"].astype(float)
        ts = ts.replace(0, np.nan).dropna()

        if len(ts) < self.config['min_data_points']:
            print(f"⚠️  {indicador}: Datos válidos insuficientes")
            return []

        # Detectar tipo y outliers
        tipo_indicador = self.detectar_tipo_indicador(meta, ts)
        ts_values, outliers = self.detectar_outliers(ts.values)
        stats = self.calcular_estadisticas(ts, outliers, tipo_indicador)

        # Generar fechas futuras
        freq = "6ME" if periodicidad == "Semestral" else "YE"
        try:
            fechas_futuras = pd.date_range(
                start=ts.index[-1] + pd.DateOffset(months=6 if periodicidad=="Semestral" else 12),
                periods=horizonte,
                freq=freq
            )
        except:
            print(f"⚠️  {indicador}: Error generando fechas")
            return []

        n_datos = len(ts)
        es_datos_escasos = (n_datos <= 5)
        limites = stats['limites']

        # Imprimir información
        print(f"\n📊 {indicador}")
        print(f"   📌 Tipo: {tipo_indicador} | Datos: {n_datos} {'⚠️ ESCASOS' if es_datos_escasos else '✅'}")
        print(f"   Último: {stats['ultimo_valor']:.2f} | Tendencia: {stats['tendencia'].upper()}")
        print(f"   Crec: {stats['crecimiento_promedio_ponderado']*100:.2f}%/año | Vol: {stats['volatilidad']*100:.1f}%")

        resultados = []
        predicciones_por_periodo = {i: {} for i in range(horizonte)}

        # Seleccionar y ejecutar modelos
        modelos_ejecutados = []

        if es_datos_escasos:
            print(f"   🔬 MODO DATOS ESCASOS")

            # Modelo 1: Regresión Lineal
            proy_rl = self.modelo_regresion_lineal(ts, stats, horizonte, periodicidad)
            if proy_rl:
                modelos_ejecutados.append(('Regresion_Lineal', proy_rl))
                print(f"   ✅ Regresión Lineal: P1={proy_rl[0]:.2f}")

            # Modelo 2: Promedio Móvil
            proy_pma = self.modelo_promedio_movil(ts, stats, horizonte, periodicidad)
            if proy_pma:
                modelos_ejecutados.append(('Promedio_Movil', proy_pma))
                print(f"   ✅ Promedio Móvil: P1={proy_pma[0]:.2f}")

        if n_datos >= self.config['min_data_full_models']:
            # Modelos completos

            # ARIMA
            proy_arima = self.modelo_arima(ts, stats, horizonte, periodicidad)
            if proy_arima:
                modelos_ejecutados.append(('ARIMA', proy_arima))
                print(f"   ✅ ARIMA: P1={proy_arima[0]:.2f}")

            # ETS
            proy_ets = self.modelo_ets(ts, stats, horizonte, periodicidad)
            if proy_ets:
                modelos_ejecutados.append(('ETS', proy_ets))
                print(f"   ✅ ETS: P1={proy_ets[0]:.2f}")

            # Holt-Winters
            proy_hw = self.modelo_holt_winters(ts, stats, horizonte, periodicidad)
            if proy_hw:
                modelos_ejecutados.append(('Holt_Winters', proy_hw))
                print(f"   ✅ Holt-Winters: P1={proy_hw[0]:.2f}")

            # Random Forest
            proy_rf = self.modelo_random_forest(ts, stats, horizonte, periodicidad)
            if proy_rf:
                modelos_ejecutados.append(('Random_Forest', proy_rf))
                print(f"   ✅ Random Forest: P1={proy_rf[0]:.2f}")

            # Prophet
            proy_prophet = self.modelo_prophet(ts, fechas_futuras, horizonte, periodicidad)
            if proy_prophet is not None:
                proy_prophet = self.validar_proyecciones_secuencial(proy_prophet, stats, horizonte, periodicidad)
                modelos_ejecutados.append(('Prophet', proy_prophet))
                print(f"   ✅ Prophet: P1={proy_prophet[0]:.2f}")

        # Tendencia Histórica (siempre)
        proy_th = self.modelo_tendencia_historica(stats, horizonte, periodicidad)
        modelos_ejecutados.append(('Tendencia_Historica', proy_th))
        print(f"   ✅ Tendencia: P1={proy_th[0]:.2f}")

        # Registrar resultados individuales
        for modelo, proyecciones in modelos_ejecutados:
            for i, fecha in enumerate(fechas_futuras):
                valor = proyecciones[i]
                predicciones_por_periodo[i][modelo] = valor

                pes, opt = self.calcular_intervalos(valor, stats, i+1)

                resultados.append([
                    id_ind, indicador, periodicidad, tipo_indicador, fecha, modelo,
                    round(valor, 2), round(pes, 2), round(opt, 2)
                ])

        # Ensemble Ponderado
        print(f"   🎯 Ensemble Ponderado:")
        for i, fecha in enumerate(fechas_futuras):
            preds = predicciones_por_periodo[i]

            if len(preds) > 0:
                pesos = self.calcular_pesos_ensemble(preds, stats)

                if len(pesos) > 0:
                    valor_ensemble = sum(preds[m] * pesos[m] for m in preds.keys())
                else:
                    valor_ensemble = np.mean(list(preds.values()))

                # Validar ensemble
                if i == 0:
                    valor_anterior = stats['ultimo_valor']
                else:
                    ensemble_anterior = [r for r in resultados if r[0] == id_ind and r[5] == "Ensemble_Ponderado"]
                    valor_anterior = ensemble_anterior[-1][6] if ensemble_anterior else valor_ensemble

                valor_ensemble = self.validar_proyeccion(valor_ensemble, valor_anterior, stats, i+1, periodicidad)

                pes, opt = self.calcular_intervalos(valor_ensemble, stats, i+1)

                resultados.append([
                    id_ind, indicador, periodicidad, tipo_indicador, fecha, "Ensemble_Ponderado",
                    round(valor_ensemble, 2), round(pes, 2), round(opt, 2)
                ])

        print(f"      ✅ Ensemble P1={predicciones_por_periodo[0].get('Ensemble_Ponderado', valor_ensemble):.2f}")
        print(f"      Modelos: {len(predicciones_por_periodo[0])}")

        # Calcular métricas de confiabilidad
        metricas = self.calcular_metricas_confiabilidad(ts, predicciones_por_periodo[0], stats)

        # Validación cruzada
        validacion = self.validacion_cruzada(ts, stats, tipo_indicador, periodicidad)

        # Guardar estadísticas
        self.estadisticas.append({
            'Id': id_ind,
            'Indicador': indicador,
            'Tipo': tipo_indicador,
            'Meta': meta,
            'N_Datos': n_datos,
            'Categoria_Datos': 'Escasos (2-5)' if es_datos_escasos else 'Normales (6+)',
            'Tendencia': stats['tendencia'],
            'Ultimo_Valor': stats['ultimo_valor'],
            'Crecimiento_Historico_%': stats['crecimiento_promedio_ponderado'] * 100,
            'Limite_Crec_Anual_%': limites['crecimiento_anual_max'] * 100,
            'Volatilidad_%': stats['volatilidad'] * 100,
            'Outliers': len(outliers),
            'Score_Confiabilidad': metricas['Score_Total'],
            'Clasificacion': f"{metricas['Emoji']} {metricas['Clasificacion']}",
            'N_Modelos': metricas['N_Modelos'],
            'MAE_%': validacion['MAE_%'] if validacion else None,
            'Precision': validacion['Precision'] if validacion else 'N/A'
        })

        self.metricas_confiabilidad.append({
            'Indicador': indicador,
            **metricas
        })

           return resultados

    # ============================
    # RECONCILIACIÓN JERÁRQUICA
    # ============================

    def reconciliar_jerarquias(self, df_proyecciones, df_original):
        """Reconcilia jerarquías usando bottom-up"""
        print("\n" + "="*80)
        print("🔄 RECONCILIACIÓN JERÁRQUICA")
        print("="*80)

        df_reconciliado = df_proyecciones.copy()
        cambios = []

        mapeo_id = df_original.groupby('Indicador')['Id'].first().to_dict()

        for config in self.config['jerarquias']:
            print(f"\n📊 {config['nombre']}: {config['padre']} = {' + '.join(config['hijos'])}")

            padre_id = mapeo_id.get(config['padre'])
            hijos_ids = {h: mapeo_id.get(h) for h in config['hijos']}

            if padre_id is None or any(id_h is None for id_h in hijos_ids.values()):
                print(f"   ❌ No se encontraron todos los indicadores")
                continue

            # Obtener combinaciones fecha-modelo
            combinaciones = df_reconciliado[df_reconciliado['Id'] == padre_id][
                ['Fecha_Proyeccion', 'Modelo']
            ].drop_duplicates()

            ajustes = 0

            for _, row in combinaciones.iterrows():
                fecha = row['Fecha_Proyeccion']
                modelo = row['Modelo']

                # Sumar hijos
                suma_hijos = 0
                todos_presentes = True

                for hijo_id in hijos_ids.values():
                    hijo_data = df_reconciliado[
                        (df_reconciliado['Id'] == hijo_id) &
                        (df_reconciliado['Fecha_Proyeccion'] == fecha) &
                        (df_reconciliado['Modelo'] == modelo)
                    ]

                    if len(hijo_data) > 0:
                        suma_hijos += hijo_data.iloc[0]['Escenario_Base']
                    else:
                        todos_presentes = False
                        break

                if not todos_presentes:
                    continue

                # Ajustar padre
                padre_data = df_reconciliado[
                    (df_reconciliado['Id'] == padre_id) &
                    (df_reconciliado['Fecha_Proyeccion'] == fecha) &
                    (df_reconciliado['Modelo'] == modelo)
                ]

                if len(padre_data) == 0:
                    continue

                idx = padre_data.index[0]
                valor_original = padre_data.iloc[0]['Escenario_Base']

                diferencia = abs(valor_original - suma_hijos)
                diferencia_pct = (diferencia / valor_original * 100) if valor_original > 0 else 0

                # Actualizar
                df_reconciliado.loc[idx, 'Escenario_Base'] = round(suma_hijos, 2)

                # Ajustar intervalos proporcionalmente
                if valor_original > 0:
                    ratio_opt = padre_data.iloc[0]['Escenario_Optimista'] / valor_original
                    ratio_pes = padre_data.iloc[0]['Escenario_Pesimista'] / valor_original
                else:
                    ratio_opt = 1.05
                    ratio_pes = 0.95

                df_reconciliado.loc[idx, 'Escenario_Optimista'] = round(suma_hijos * ratio_opt, 2)
                df_reconciliado.loc[idx, 'Escenario_Pesimista'] = round(suma_hijos * ratio_pes, 2)

                if diferencia_pct > 0.01:
                    cambios.append({
                        'Jerarquia': config['nombre'],
                        'Padre': config['padre'],
                        'Fecha': fecha,
                        'Modelo': modelo,
                        'Valor_Original': round(valor_original, 2),
                        'Valor_Reconciliado': round(suma_hijos, 2),
                        'Diferencia': round(diferencia, 2),
                        'Diferencia_%': round(diferencia_pct, 2)
                    })

                ajustes += 1

            print(f"   ✅ {ajustes} ajustes realizados")

        df_cambios = pd.DataFrame(cambios) if cambios else pd.DataFrame()

        if len(df_cambios) > 0:
            print(f"\n✅ Total ajustes: {len(df_cambios)}")
        else:
            print(f"\n✅ No se requirieron ajustes")

        return df_reconciliado, df_cambios

IndentationError: unexpected indent (ipython-input-1019933760.py, line 758)